# **Data Project: What Determines a Song's Success on the Billboard Hot 100? (Group C)**

[![Green-Black-and-White-Modern-Music-Charts-Presentation.png](https://i.postimg.cc/Qdt6grb1/Green-Black-and-White-Modern-Music-Charts-Presentation.png)](https://github.com/Isabelle333/billboard-musicbrainz-extract)



**Members:** Thien Anh NGUYEN, Norelle CHY--GLASSCOCK, Lio SUCHET--WANG, Georg RIEKHAKAINEN, Isabelle LIEW YU CHEN

### **About this study**

**Research question.** What determines whether a song reaches the Billboard Hot 100?

**What we mean by success.** A song is successful if it appears on the Billboard Hot 100 at least once, i.e. it *reaches the chart*. This is a yes/no outcome. How high a song climbs, how long it stays, or whether it reaches No. 1 are **not** our measure of success; chart positions are used only to describe the data and to check its quality.
*Why this definition:* it is objective (one clear threshold), it matches our question, and it means the same thing in every year, whereas "how successful" a charting song is depends on Billboard's changing chart rules.

**What this implies for our data.** Billboard chart data contain only songs that reached the chart, so on their own they cannot show what distinguishes these songs from songs that never charted. We therefore built a comparison group from MusicBrainz: songs by the same lead artists, released in the same years, that never charted (Section 10). The role of the Billboard data is to identify *which* songs reached the chart and *when* they first entered it.

**Analysis plan: factors examined.** We fixed our main factors in the proposal, before analysing the data, so that results are not found by searching through many possible patterns. Factors added later are listed separately below, with when and why they were added, and the exploratory one is reported apart from the main results.
- *Planned in our proposal (16 September):*
  - **Time to chart:** the time between a song's release and its first appearance on the chart (Section 9; 11.5).
  - **Genre** (9.5; 11.4).
  - **Artist popularity:** whether the lead artist had already reached the chart before (7.5; 9.7).
- *Added during the work:* **release type**, i.e. whether a song was first released as a single or on an album (9.6; 11.3). MusicBrainz records it for every song, and the descriptive analysis (7.4–7.7) showed that album tracks by established artists make up a large share of recent entries, so it could not be left out.
- *Robustness:* the main comparisons are repeated under alternative choices that we made ourselves (11.6), to check that the conclusions do not depend on them.
- *Exploratory:* **release month** (11.7), added after the main analysis because the January dip found in 7.3 (fewer charting songs enter the chart in January) raised the question whether the month of release is associated with reaching the chart. It is reported separately, as a secondary part of the answer that needs confirming on other data.

**Hypothesis.** Most songs that reach the Hot 100 do so soon after release; beyond a certain time after release, a song becomes much less likely to ever enter the chart. We test this first among charting songs (9.2–9.4), then including songs that never charted (11.5).

**Scope.**
- *Chart:* the Billboard Hot 100 only, Billboard's main all-genre singles chart. Other Billboard charts are specific to a genre or format and would change what "reaching the chart" means.
- *Period:* charts dated 2000-01-01 to 2025-12-27 (26 complete years, 1,357 weekly charts).
- *Unit of analysis:* a song, identified by its title and performer credit (refined during cleaning).

**Workflow.** Collection → raw-data quality assessment → cleaning → storage → additional source and matching → analysis → communication (report and presentation). Each step is justified in this notebook.

**Data sources.**
1. **Billboard Hot 100 weekly charts**, from the public archive `utdata/rwd-billboard-data` (Sections 1 and 3).
2. **MusicBrainz**, the open music encyclopedia: release dates, release types, genres and lead artists of the charting songs (Section 8), and the comparison group of songs that did not chart (Section 10).

The Kaggle dataset named in our proposal is not used (1.10).

**Caveats known from the start.**
- Songs already on the chart on 2000-01-01 first entered before our period, so their entry is not observed.
- Songs released in late 2025 may enter the chart only in 2026, after our period ends.
- Billboard's chart rules changed over 2000–2025 (e.g. the inclusion of streaming), so years are not perfectly comparable.

## **Contents**

- How to reproduce this notebook
- Decision log
- Changes since our earlier submissions
1. Choice of Billboard data source: legal and ethical considerations
2. Setup
3. Collection: acquiring the Billboard Hot 100 archive
4. Raw-data quality assessment
5. Cleaning
6. Storage
7. Descriptive analysis of the Billboard data
8. Additional source: MusicBrainz
9. When do songs reach the chart?
10. Comparison group: songs that did not chart
11. What distinguishes the songs that charted?
12. Generated documentation: README and file inventory
13. Conclusion

## **How to reproduce this notebook**

1. Open this notebook in Google Colab.
2. Click **Runtime → Run all**. (To re-run from scratch, first use **Runtime → Disconnect and delete runtime**, so that no files from an earlier run remain.)
3. Nothing needs to be uploaded and no Google Drive or login is needed: the notebook downloads its data directly from the original public sources, at fixed versions, and checks that they are exactly the files we used.
4. The records the notebook produces (provenance, quality assessments, cleaning logs, storage documentation, analysis results and figures) are downloaded to your computer automatically. If your browser asks whether to allow multiple downloads, click **Allow**. The chart data itself is not downloaded (Section 1.8), with one exception: `billboard_song_names.csv`, the list of song titles and performers needed for the MusicBrainz step (8.1), which is kept within the group.
5. An internet connection is required. Section 3 takes about a minute.
6. **Order of the tasks, and files not in our submission.** The MusicBrainz step runs outside the notebook, in PostgreSQL, and does not need to be repeated: the notebook downloads its published results. To repeat it, the order is: (1) this notebook up to Section 8.1, which produces `billboard_song_names.csv`; (2) Job A, `run_all.sh`; (3) Job B, `run_comparison.sh`; (4) Sections 8.3–13 of this notebook, which download the published results of Jobs A and B at fixed commits (a rerun on the same MusicBrainz export gives files with the same SHA-256). Intermediate files that are not in our submission, such as `billboard_song_names.csv` and the database, are rebuilt by these steps. Details: README (12.2) and the README of the extract repository.

If an integrity check fails (Sections 3.2, 8.3, 10.2), the notebook stops on purpose rather than continuing with different data (see Section 3.6).

## **Decision log**

Every major decision in this project, the alternatives considered, and where it is justified.

| # | Decision | Alternatives considered | Main reason | Section |
|---|---|---|---|---|
| 1 | Billboard data from the UT Data archive, pinned to one commit and verified by SHA-256 | Collecting charts ourselves with billboard.py; a paid Billboard subscription; other public datasets | Billboard.com's Terms of Use (§9) prohibit scraping, also for subscribers; pinning makes the data reproducible | 1, 3 |
| 2 | Problems in the chart data flagged, not deleted (e.g. the 2018-01-06 chart) | Deleting or correcting affected rows | The correct values cannot be known; flags let each analysis decide | 4.7, 5 |
| 3 | Renamed chart entries linked into one song using Billboard's own last-week ranks | Exact names only; manual matching | Uses the source's own evidence; checked so that no song appears twice in a week | 5.10 |
| 4 | Storage in SQLite | CSV files, Parquet | Enforces keys and rules on our data; tested in 6.6–6.7 | 6 |
| 5 | MusicBrainz processed from the full official dump in a local PostgreSQL database | Processing in Colab | About 25 GB of tables, too large for Colab; only one row per song is imported | 8.2 |
| 6 | Dump of 17 October 2026 | Dump of 30 September 2026 | The earlier dump had already been removed from the server | 8.2 |
| 7 | Release dates on 1 January treated as known to the year only (M1) | Keeping them as exact dates | 236 such dates against about 29 expected: mostly placeholders | 8.7 |
| 8 | 218 genres grouped into 10 families by ordered keyword rules; unknown genres kept as "Other" | Keeping all genres; a hand-made list | Too many genres to compare; rules are transparent and the full mapping is saved | 8.7 |
| 9 | Every join declares its expected relationship and checks the row count | Unchecked joins | A duplicated key can silently multiply rows (Lecture 6) | 4.7, 5.10, 6.3, 8.7, 9, 10 |
| 10 | Time to chart counted in charts waited after the first possible chart | Days from release to first chart | Billboard's chart-dating rules changed in 2015 and 2018 | 9.1 |
| 11 | Every song followed for the same number of charts (52, or 260 for long-term results); releases up to December 2024 | Including all 2025 releases | Recent songs cannot yet show slow entries; including them would make recent years look faster | 9.2 |
| 12 | Comparison group: songs by charting lead artists, released 2000–2024, that never charted | Random songs released in the US market | Release countries exist in MusicBrainz, but many digital releases are only labelled "worldwide" and a random draw would be dominated by obscure releases; same-artist songs are realistic candidates and need only tables already loaded | 10, Job B |
| 13 | A comparison song = lead artist + title without brackets; songs sharing a title with any Billboard song, and intros/interludes/skits, excluded | Counting every recording; excluding only exact matches | Remixes and re-releases count once, as on the chart; no hidden hits or album fillers in the comparison group | 10.1, Job B (`mb_07`) |
| 14 | Same rules as Job A for release dates, release types and genres of comparison songs | Simpler rules for Job B | Both groups must be measured the same way to be comparable | 10.1, Job B (`mb_07`) |
| 15 | Genre, release-type and artist-history comparisons made within release periods, shares shown only for groups of 30+ songs | Overall comparisons only | The genre mix and release practices changed over time; small groups are too noisy | 9.5–9.7 |
| 16 | Problems in the comparison group cleaned in the notebook (K1–K5); the published extract is unchanged | Re-running `mb_07` with more exclusions | The raw extract stays exactly as produced and verified; every exclusion is visible, counted and justified | 10.5 |
| 17 | Artists that reached the chart only with songs released 5+ years earlier excluded from the comparison (K2) | Keeping all lead artists | Their 2000–2024 releases are mostly reissues of old catalogues, not new songs competing for the chart (same idea as M3) | 10.5 |
| 18 | Comparisons between charting and non-charting songs give each artist equal weight (artists with at least 10 songs) | Pooling all songs | The 20 largest catalogues hold 10.5% of comparison songs; a few prolific artists would otherwise dominate | 11 |
| 19 | Comparison group placed in its own section after the analysis of charting songs | Inside Section 8 (same source) | It is a separate population with its own collection, quality, cleaning and storage, and it answers the limitation found in 9.4 | 10 |
| 20 | MusicBrainz scripts and extracts published in a separate public repository | Keeping them only on a laptop; storing them in the group's project repository | The notebook must download the extracts from a public, fixed and verifiable location; the extract carries MusicBrainz's licence, unlike our own code | 8.3, 10.2 |
| 21 | Charting vs non-charting comparison limited to songs first released 2000–2023 | 2000–2024 as in Section 9 | 14% of comparison songs only have a release year, so the cut-off must be a whole year; 2023 gives every song at least two years to chart | 11.1 |
| 22 | Each category's chart rate compared with the same artist's own average | Comparing pooled shares; statistical models | Removes differences between artists (e.g. prolific vs occasional) in a simple, transparent way; no modelling, which the brief requires discussing with the professor first | 11.3–11.4 |
| 23 | README and file inventory generated by the notebook from this run's settings and results | Writing the README by hand | Every number, version and fingerprint in it comes from the run that produced the results, so the README cannot drift from the notebook | 12 |
| 24 | Kaggle "Spotify Top Hit Playlist 2010–2022" not used | Using it as planned in the proposal | Contains only hits and covers only 2010–2022; the MusicBrainz comparison group answers what it was meant to | 1.10, 10 |
| 25 | Years of chart entry grouped into four eras whose boundaries follow the Hot 100 formula changes (2005, 2012–13, 2018) | Equal five-year periods | How songs enter and how long they stay are set partly by Billboard's rules, so eras bounded by rule changes make comparisons meaningful; Sections 9 and 11 keep five-year periods because they group songs by release year | 7.1 |
| 26 | Glee Cast kept in the data, with its effect checked | Removing it as an outlier | Its songs genuinely charted; the checks show which results it drives (only 2009–2012 in Section 7; none in Section 11) | 7.5, 11.6 |
| 27 | Main comparisons repeated under alternative choices (success = at least 2 weeks on the chart; 5 or 20 songs per artist; without Glee Cast; genre within one release type) | Reporting one specification only | Several thresholds are our own choices; the checks show whether the conclusions depend on them | 11.6 |
| 28 | Release month analysed after the main results, among singles only, and reported as a secondary, exploratory result | Treating it like the planned factors; comparing all songs by release month | A factor chosen after seeing the data is more likely to show a chance pattern, so it is labelled and checked separately; comparing all songs mostly repeated the single-versus-album result | 11.7, 13 |
| 29 | Genre votes also stored as one row per song and genre; the extraction's tie rule (most votes, then alphabetical) kept and checked on every run; genre results repeated without tied or single-vote genres | One text field only; choosing tied genres by hand | Ties can now be queried and the rule is verified; any other tie rule would be equally arbitrary, so we test whether the results depend on it instead | 8.7, 10.6, 11.6 |
| 30 | Title rule of the comparison group (any artist) kept; its worst-case effect computed | Re-running `mb_07` with the rule limited to the same artist (peer review) | A re-run would change the published extract, its fingerprints and every result of Sections 10–13; the worst case lowers chart rates by at most about a quarter and leaves the shape of 11.5 unchanged | 10.1, 11.6 |
| 31 | Release date = first release of the recording, in both groups; time to chart repeated for songs first released as singles | Using a single's date where one exists (peer review) | Both groups must be measured with the same rule (row 14); the singles-only check shows whether album dates drive the result | 9.2, 11.5, 11.6 |

## **Changes since our earlier submissions**

This section summarises how the project changed since the raw-data submission (29 September) and the intermediate version (3 November), and how we responded to the first peer review (Group D) and the second peer review (Group I). The reasons for each decision are in the decision log above and in the sections referred to.

**Since the raw-data submission (29 September)**

| Change | Why | Where |
|---|---|---|
| Billboard data no longer collected by scraping billboard.com with billboard.py, but taken from the public UT Data archive, pinned to one commit and verified by SHA-256 | Billboard.com's Terms of Use (§9) prohibit scraping; our professor advised us to stop; PMC did not grant permission (1.3) | 1, 3; row 1 |
| A comparison group of songs that never charted added (MusicBrainz, songs by the same lead artists) | Chart data contain only songs that charted, so on their own they cannot show what distinguishes them | 10, 11; rows 12–14 |
| Kaggle "Spotify Top Hit Playlist 2010–2022" no longer used | It contains only hits and covers only 2010–2022; the comparison group answers what it was meant to | 1.10; row 24 |
| Time to chart counted in charts waited, not in days | Billboard changed how charts are dated in 2015 and 2018 | 9.1; row 10 |
| Release type added as a factor | The descriptive analysis showed that album tracks make up a large share of recent entries | About this study; 9.6, 11.3 |

**Since the intermediate version (3 November)**
- Robustness checks of the main comparisons (11.6) and an exploratory analysis of the release month (11.7), each documented with its reasons (rows 27–28).
- An analysis plan (About this study) listing which factors were planned and which were added later.
- A conclusion (Section 13).
- The changes made in response to the second peer review, below.

**Response to the first peer review (Group D)**

| Point raised | What we did | Where |
|---|---|---|
| 1. "Success" not clearly defined (reaching the chart, peak position or weeks on chart?) | Defined as a yes/no outcome: a song is successful if it reaches the Hot 100 at least once. Peak position and weeks on chart are used only to describe the data | About this study |
| 2. The Kaggle Spotify dataset covers only 2010–2022 | No longer used: it covers only 2010–2022 and contains only hits; the MusicBrainz comparison group answers what it was meant to | 1.10; row 24 |
| 3. Matching Billboard songs to MusicBrainz (typos, "featuring", live versions) and what to do with songs that do not match | Same normalisation on both sides; title and artist must both match; live, karaoke, instrumental, demo and a cappella versions excluded; fallback without bracketed parts. 95.7% matched, 30/30 correct in a manual check; the 504 unmatched songs (4.3%, mostly collaborations and recent songs) are described and stated as a limitation | 5, 8.2, 8.4a, 8.4d, 8.4e |
| 4. Time-to-chart hypothesis | Kept as our main hypothesis; tested among charting songs and then including songs that never charted | 9.2–9.4, 11.5 |
| 5. README explaining which script to run and in which order | The scraping repository was removed when we stopped scraping (1.3). The MusicBrainz extract repository has a README with the order of execution, and the notebook has "How to reproduce" and a generated README | How to reproduce; 12.2 |

**Response to the second peer review (Group I)**

| Point raised | What we did | Where |
|---|---|---|
| Manual check of 30 MusicBrainz matches still empty | Completed: 30/30 correct matches, 28/30 plausible release dates, 26/30 sensible genres | 8.4d |
| No short "what changed" note | Added (this section) | – |
| 1. The release date is the first release of the recording, often the album, so some late entries may only reflect an early album date | Time to chart repeated for songs first released as singles, whose date is the single's own. Late entries are indeed more common among album-first songs, but the early drop is even sharper among singles, so the threshold does not come from album dates. One date rule is kept for both groups, because both must be measured the same way | 9.2, 11.5, 11.6; row 31 |
| 2. The title rule of the comparison group (any artist) is too wide | Kept, because re-running the extraction would change every result of Sections 10–13; we state its direction and compute its worst case: chart rates at most about a quarter lower, same shape of the 11.5 curve. A same-artist rule is listed as further work | 10.1, 11.1, 11.6, 13.3–13.4; row 30 |
| 3. The notebook stopped in 6.10 | The accidentally deleted comparison cell of 6.7 is restored; the notebook runs from a fresh runtime without errors | 6.7 |
| 3. README: run the notebook first; where the MusicBrainz files go | Added to the generated README and to the extract repository's README | 12.2 |
| 4. Genre votes kept in one text field; ties without a clear rule | Votes also stored as one row per song and genre; the tie rule (most votes, then alphabetical) is documented and checked on every run; 12.7% of charting songs have a tie between families and 27.7% a top genre from a single vote; the pop result holds without these songs | 8.7, 10.6, 11.6; row 29 |
| 5. The days from release to first chart are stored as a fixed value | Kept, with the reason; a final check now recomputes the value and stops the notebook if it differs | 8.7, 12.3 |
| 6. The 9.4 figure uses only charting songs; show 11.5 first | The 9.4 title now says "Charting songs only"; the report and presentation show the 11.5 figure first | 9.4 |
| 7. Strongest part: the time-to-chart measure | Kept unchanged | 9.1 |
| 8. Is the Spotify dataset dropped? | Yes, for the reasons above; audio features are not studied and are listed as further work | 1.10, 13.4 |
| 9. Check non-charting songs by hand too | 30 randomly selected comparison songs checked by hand: 25/30 real songs, 28/30 plausible dates, 29/30 never charted. Intro, live and medley titles counted (3.0% of comparison songs, 0.6% of charting songs) and the main results repeated without them: unchanged | 10.5, 11.6, 13.3 |

## **1. Choice of Billboard data source: legal and ethical considerations**

This section documents why we obtain Billboard chart data from a public archive rather than collecting it ourselves: where the data originally comes from, who compiled the archive and how, the outcome of our permission request, why we consider our use acceptable, what remains debatable, and what limitations the archive introduces.

### **1.1 Our original plan: collecting charts with billboard.py**

We originally planned to collect every weekly Hot 100 chart ourselves with the open-source Python library **billboard.py** (github.com/guoguo12/billboard-charts).

Although the library describes itself as a "Python API", it is not an API in the sense of a data service offered by Billboard: **Billboard provides no public API for chart data.** The library downloads the chart web pages from billboard.com and extracts the values from their HTML code. It is therefore a **web scraper** that offers a convenient Python interface.

In 2026 Billboard changed the layout of its chart pages (adding a "WEEKS AT NO. 1" column), which broke the library's parser (GitHub issue #101). A group member wrote a local patch to make it work again.

### **1.2 Why we stopped: Billboard.com's Terms of Use**

Billboard.com's Terms of Use are published by **Penske Media Corporation (PMC)** (version last updated 21 August 2026). The clauses relevant to us:

| Clause | What it says (summary) | Why it matters to us |
|---|---|---|
| §9 | Users shall not engage in "spidering, 'screen scraping,' 'database scraping'", or use robots or other automated data-gathering tools | billboard.py does exactly this |
| §1 | PMC claims ownership of the site's content, data, compilation and database rights | the chart data itself is claimed by PMC |
| §3 | Users get a limited, personal, non-commercial licence to view the site | scraping is outside this licence |
| §4 | No reproduction, redistribution or derivative works | constrains what we may publish |
| §9 | No use of content to train or ground AI/ML models "or other algorithms" | relevant to how we analyse the data (see 1.7) |

A non-commercial or educational purpose is **not** an exception to §9.

After a group member had patched billboard.py and tested it, we reviewed these terms, and confirmed that billboard.py, being a parser of billboard.com pages, is the kind of automatic extraction §9 prohibits. We therefore:
- **stopped** using billboard.py and rebuilt our chart data from a public archive (Section 3), so that our pipeline is reproducible from an accessible source;
- **deleted** the group's public GitHub repository, which contained our patched version of billboard.py and test files with chart data downloaded while testing it, so that neither the scraper nor the data is published by us.

### **1.3 Permission request to PMC / Billboard**

On 29 October 2026 we wrote to Billboard's research contact and PMC's licensing contact (research@billboard.com, licensing@pmc.com), with our professor in copy. We explained that this is a non-commercial university course project, stated the data we need (weekly Hot 100 chart positions, 2000–2025: chart date, rank, song title, artist, last-week position, peak position, weeks on chart), how we will use it (statistical analysis for coursework), and that we will not redistribute it.

**Reply.** The same day, a member of Billboard's chart data team called us. He explained that Billboard's chart data is available to subscribers, through a Billboard subscription billed at $191.88 per year. No permission to use the data for our project was given.

**Our decision: we continue with the public archive (1.4–1.5) and do not buy a subscription.**
- A subscription gives access to the charts on billboard.com, but subscribers are bound by the same Terms of Use. §9 forbids extracting content "whether automated or manual", so a subscription would not allow us to collect 26 years of weekly charts, by program or by hand.
- It would therefore not solve the legal question, only the access question, which the public archive already answers in a reproducible way.

### **1.4 Alternatives we considered**

| Option | What it is | Decision and reason |
|---|---|---|
| Scrape with patched billboard.py | We run the scraper ourselves | **Rejected.** Directly prohibited by §9; also fragile, since it breaks whenever Billboard changes its page layout |
| Scrape with `billboard-charts` (PyPI) | A different scraper library | **Rejected.** Same §9 problem |
| Billboard subscription | Paid access to billboard.com ($191.88 per year), offered by Billboard in reply to our request (1.3) | **Rejected.** Subscribers are bound by the same Terms of Use, so extracting 26 years of charts would still breach §9; it gives access, not permission |
| **`utdata/rwd-billboard-data`** | Public archive of every Hot 100 chart since 1958, one CSV file, updated weekly | **Chosen.** Reasons below |
| `mhollingshead/billboard-hot-100` | Public archive, one JSON file per week, updated daily | **Not chosen as main source.** Same legal situation (also built by scraping); documentation and licence not checked in the same depth; possible independent cross-check later |
| ListenBrainz | Open listening-history data | **Not chosen.** It measures what ListenBrainz users listen to, not Billboard chart success, so it would change our research question; its users are also not representative of the general public |

Why `utdata/rwd-billboard-data`:
- Recommended by our professor as an acceptable public archive.
- **Transparent:** its scraping and combining scripts are public, its sources are named, and its known errors are listed on its documentation site.
- **Maintained** by a university journalism programme (University of Texas at Austin) for teaching, and updated automatically every week.
- **Simple and verifiable:** the whole history is one CSV file, so we can pin and verify a single file (Section 3).
- **Licensed:** the repository carries an MIT licence (this covers the repository's files as distributed by UT; it cannot grant rights over Billboard's data, see 1.7).

### **1.5 Where the data comes from: the provenance chain**

1. **Billboard** compiles the weekly Hot 100 (based on sales, streaming and radio airplay) and publishes it on billboard.com.
2. **The archive's maintainer**, Christian McDonald (GitHub: `critmcdonald`) of UT Data, University of Texas at Austin, collects it:
   - **2022 onwards:** an R script (`action_scrape_charts.R`) scrapes the current chart from billboard.com. It runs automatically on GitHub Actions (Tuesday to Friday) and saves one CSV per chart week in `data-scraped/hot-100/`.
   - **Before 2022:** combined from three third-party sources: a Kaggle dataset (charts to November 2021, with at least 13 missing records), a data.world dataset (to June 2021) used to fill gaps, and gap data collected with the Data Miner Chrome plugin and stored in a Google Sheet. These were merged in `notebooks/02-hot100-archive` and saved as `previous_archive.csv`.
   - The script `action_combine_charts.R` stacks all of these into `data-out/hot-100-current.csv`.
3. **We** download that file at a fixed version (commit `f5801b0`, 13 October 2026) and verify it with a SHA-256 fingerprint (Section 3).

The repository also contains `hot100_assignment.csv`. It holds the same rows with deliberately messy headers and date formats, created for UT's own data-cleaning exercise. We do not use it: it adds nothing and would only create artificial cleaning work, which the project brief tells us to avoid.

### **1.6 Why we consider our use acceptable**

1. **We do not access or scrape billboard.com ourselves.** We download a file published openly on GitHub.
2. **The data is factual.** Chart positions record which song ranked where in a given week. Under US law, which governs the Terms of Use, facts as such are not protected by copyright (*Feist v. Rural*, 1991), although a selection or arrangement of facts can be (see 1.7).
3. **Non-commercial, educational purpose.** The data is used only for a university course. Our outputs are aggregate analyses, not a substitute for Billboard's charts or services.
4. **Minimal use.** We use only the fields we need (positions, titles, artists) for the period we need (2000–2025). We use no images, editorial content, audio or lyrics.
5. **Transparency.** We disclose the full provenance (1.5), asked PMC for permission and report its reply (1.3), and kept our professor informed.
6. **No redistribution.** We do not publish the chart data. Our code downloads it from the original public source at run time, and we link to that source instead of sharing copies.
7. **The data is already freely republished.** Although Billboard sells access to its charts (1.3), the same chart history is republished in several open archives. This does not by itself make our use lawful, but our use does not make the data any more available than it already is.

### **1.7 What remains debatable**

- **Shifting responsibility.** The archive is itself built by scraping, against §9. Using it means the breach is not ours, but one can fairly object that this only moves the problem to someone else while we still benefit from it. Billboard's reply (1.3) makes this objection stronger: Billboard sells access to its charts, and the archive offers the same history for free. We acknowledge this objection.
- **Database rights.** §1 claims database rights, and EU law (Database Directive 96/9/EC) protects databases against extraction of a substantial part of their contents; 26 years of weekly charts is plausibly substantial. EU law also has exceptions for teaching and scientific research and for text-and-data mining (Directive 2019/790), but whether these apply to a student group project using a third-party archive is uncertain.
- **Facts or compilation?** Individual positions are facts, but the chart as a whole is Billboard's ranking, produced with its own methodology. Whether that compilation is protected is debatable.
- **"Other algorithms" (§9).** §9 forbids using content to train or ground AI/ML models "or other algorithms". This wording is very broad. If we fit statistical models to chart data, this could arguably apply; we will keep methods as simple as the question allows and discuss any modelling with our professor first (which the project brief also requires).
- **Derivative works (§4).** Our cleaned dataset could count as a derivative work. This is a further reason not to publish it.

We are not lawyers, and our professor has described the legal position as grey. We therefore document the issue rather than claim a definitive answer.

### **1.8 Measures we take**

- **Billboard.com is never accessed by our code**, and billboard.py is not used anywhere in this notebook. Chart data comes only from the public archive, downloaded from its original source at a **fixed, verified version** (Section 3), so others can reproduce our work without us sharing the data.
- **No chart data in any public repository.** The group repository that held our patched billboard.py and its test files with chart data has been deleted (1.2). Our only public repository is the extract repository (below), which also holds this notebook, published without its outputs, because some outputs show sample rows of chart data.
- **Chart data is not redistributed.** The database, the raw Billboard file and the cleaned tables stay in the Colab session and are not downloaded by default (`DOWNLOAD_RAW_BACKUP`, `DOWNLOAD_CLEAN_DATA` and `DOWNLOAD_DATABASE` are set to `False`). Section 12.1 lists every file the notebook writes and whether it was downloaded.
- **One file with chart data is downloaded:** `billboard_song_names.csv` (8.1), the list of song titles and performers needed to match our songs to MusicBrainz in PostgreSQL. It is shared only within the group and never published.
- **The MusicBrainz extract and this notebook are published** in [github.com/Isabelle333/billboard-musicbrainz-extract](https://github.com/Isabelle333/billboard-musicbrainz-extract). The extract contains MusicBrainz data and our own `song_id` only, no chart data (Section 8.3); the notebook is published without outputs, so it contains code and text only, and running it downloads the chart data from its original source.
- Notebook outputs display only small samples (a few rows), never full tables.

### **1.9 Limitations the archive introduces**

Expected limitations, to be checked with data in Section 4:

- **Mixed origins.** Pre-2022 rows come from three third-party datasets; 2022 onwards from UT's scraper. Formatting (e.g. how featured artists are written) may change at the boundary.
- **Known errors.** The maintainer documents missing records in the Kaggle source and duplicate entries (in 1961 and 1970, outside our period). We cannot assume our period is error-free.
- **Second-hand data.** We cannot verify the values against billboard.com ourselves, since we do not access it.
- **Corrections.** Billboard occasionally corrects charts. Our fixed snapshot reflects what the maintainer captured as of 13 October 2026.
- **Name inconsistencies.** The same artist or song may be written differently across weeks or sources. This matters when matching to MusicBrainz.
- **Unusable column.** `wks_at_no1` was only added recently and is almost entirely empty.
- **Missing `last_week` by design.** It is empty for new entries and re-entries.
- **Chart date ≠ listening week.** `chart_week` is Billboard's official chart date, which falls after the week in which sales and streams were counted. This matters for our "time from release to chart" hypothesis.
- **Methodology changes.** Billboard changed how the Hot 100 is calculated between 2000 and 2025 (e.g. adding streaming), so years are not perfectly comparable.
- **Selection.** The archive contains only songs that charted. It tells us nothing about songs that never charted, which matters for a question about what makes a song reach the chart (addressed in Sections 10–11).
- **Single maintainer.** The archive could change or disappear. This is addressed by pinning and verification (Section 3).

### **1.10 Licences of our other sources**

We apply the same check to our other sources:
- **MusicBrainz:** obtained from the official full export `20261017-002244` published by the MetaBrainz Foundation. Core data (recordings, artists, releases) is **CC0** (public domain); genres are derived from user tags, published under **CC BY-NC-SA 3.0** (attribution, non-commercial, share-alike). Our extract is therefore published with attribution, for non-commercial educational use, under the same licence.
- **Kaggle "Spotify Top Hit Playlist 2010–2022": not used.** Planned in our proposal, but it contains only hit songs (the same limitation as the chart data itself), covers only 2010–2022, and would add a third name-matching step without helping to answer our question. The MusicBrainz comparison group (Section 10) replaced it. Because it was our only source of audio features (e.g. tempo, energy, danceability), we do not study them; that would need a source that also covers songs that never charted (13.4).

## **2. Setup**

This section loads the libraries, defines every fixed setting in one place, and creates the folder for the files the notebook produces. Putting all settings in one cell means anyone can see, and if needed change, every assumption the notebook depends on without searching through the code.

### **2.1 Libraries**

We use only Python’s standard library plus pandas, requests, numpy and matplotlib, all pre-installed on Colab, so no installation step is needed. Library versions are printed and later saved in the provenance record, because results can depend on them.

In [ ]:
# --- Standard library: built into Python, nothing to install ---------------
import hashlib                              # computes SHA-256 fingerprints of files
import io                                   # lets pandas read data held in memory as if it were a file
import json                                 # saves the provenance record in JSON format
import platform                             # records the operating system the notebook ran on
import sys                                  # gives the Python version
import time                                 # used to pause between download retries
from datetime import datetime, timezone     # timestamps in UTC
from pathlib import Path                    # builds file paths that work on any operating system

# --- Third-party libraries: pre-installed on Google Colab ------------------
import pandas as pd                         # tables (DataFrames)
import requests                             # downloads files from the web over HTTPS
import matplotlib                           # version recording
import matplotlib.pyplot as plt             # charts
import numpy as np                          # numerical helpers

# --- Colab helper for downloading files to your own computer --------------
# google.colab only exists inside Colab. try/except lets the notebook also run
# in a normal Jupyter environment without crashing.
try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Print versions so they are visible in the notebook output.
print(f"Python   : {sys.version.split()[0]}")
print(f"pandas   : {pd.__version__}")
print(f"requests : {requests.__version__}")
print(f"In Colab : {IN_COLAB}")
print(f"numpy    : {np.__version__}")
print(f"matplotlib: {matplotlib.__version__}")

### **2.2 Configuration**

All fixed settings are defined here, in capital letters (the Python convention for values that should not change while the notebook runs).

- **Pinned source.** We download the archive file as it was in one specific commit (snapshot) of the repository, not the latest version. The file is updated every week, so downloading "the latest" would give different data on different days. A commit's content can never change, so this link always returns the same file.
- **Expected fingerprint.** `EXPECTED_SHA256` is the SHA-256 fingerprint of that exact file, calculated when we chose the snapshot on 2026-10-14. Section 3.2 compares every download against it.
- **Study window.** Charts dated 2000-01-01 (the first Saturday chart of 2000) to 2025-12-27 (the last chart dated in 2025). We define years by Billboard's official chart date, because that is how Billboard labels its charts. Note that the chart dated 2000-01-01 reflects listening in late December 1999.

In [ ]:
# --- Notebook identification ------------------------------------------------
NOTEBOOK_NAME = "Data_Project_Billboard_Charts_Group_C.ipynb"

# --- Pinned data source ------------------------------------------------------
SOURCE_NAME        = "UT Data – rwd-billboard-data (Billboard Hot 100 archive)"
SOURCE_REPO        = "utdata/rwd-billboard-data"                    # GitHub owner/repository
SOURCE_COMMIT      = "f5801b03d5e629845e65213ff236df98f1aae3fd"     # the frozen snapshot we use
SOURCE_COMMIT_TIME = "2026-10-13T16:29:00Z"                         # when that snapshot was made (UTC)
SOURCE_PATH        = "data-out/hot-100-current.csv"                 # file inside the repository
SNAPSHOT_CHOSEN_ON = "2026-10-14"                                   # when our group chose this snapshot

# --- Expected properties of that exact file ----------------------------------
EXPECTED_SHA256 = "3f2a274dd7cb26217c3bce8fb2e761ed3e131c5da6c00744a39c070e8b47dc20"
EXPECTED_BYTES  = 20_699_092                                        # file size in bytes
EXPECTED_COLUMNS = ["chart_week", "current_week", "title", "performer",
                    "last_week", "peak_pos", "wks_on_chart", "wks_at_no1"]

# --- Links built from the settings above ------------------------------------
RAW_URL    = f"https://raw.githubusercontent.com/{SOURCE_REPO}/{SOURCE_COMMIT}/{SOURCE_PATH}"
COMMIT_URL = f"https://github.com/{SOURCE_REPO}/commit/{SOURCE_COMMIT}"
REPO_URL   = f"https://github.com/{SOURCE_REPO}"

# --- Study window (chart dates, inclusive) -----------------------------------
STUDY_START = "2000-01-01"
STUDY_END   = "2025-12-27"

# --- Where the notebook writes its files -------------------------------------
OUTPUT_DIR = Path("outputs")
RAW_DIR    = OUTPUT_DIR / "raw"

# --- Chart colours (used by every figure in the notebook) --------------------
# COLOR_MAIN for every chart with one series; COLOR_SECOND for the second series when two are compared,
# and for bars that carry black error bars (11.3-11.4), which the dark main colour would hide;
# COLOR_THIRD for a third series and as the light end of colour scales (heatmaps).
COLOR_MAIN   = "#2C5F2D"   # dark green: the main series
COLOR_SECOND = "#6FAF6B"   # mid green: the comparison series
COLOR_THIRD  = "#99D792"   # light green: third series / light end of colour scales

print("Pinned download URL:")
print(RAW_URL)

### **2.3 Output folder and download helper**

In [ ]:
# Create outputs/ and outputs/raw/ (no error if they already exist).
RAW_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output folder ready: {OUTPUT_DIR.resolve()}")

DOWNLOADED = []   # every file sent to computer, listed in the file inventory (Section 12.1)

def download_file(path):
    """Send a file from Colab to the user's computer and record it.
    Outside Colab, just report where the file was saved."""
    if str(path) not in DOWNLOADED:
        DOWNLOADED.append(str(path))
    if IN_COLAB:
        colab_files.download(str(path))
    else:
        print(f"Not in Colab – file saved at: {Path(path).resolve()}")

## **3. Collection: acquiring the Billboard Hot 100 archive**

**Method.** We download `data-out/hot-100-current.csv` from the `utdata/rwd-billboard-data` repository at the pinned commit `f5801b0` (13 October 2026), verify that it is byte-for-byte the file we chose, save it unchanged as our raw data, check its structure, and record its provenance.

**Why this design:**
- **Automatic.** No manual upload, Drive or login, so anyone can re-run it.
- **Same data every time.** The commit pin always returns the same file; the SHA-256 check proves it did, and stops the notebook if not.
- **Traceable.** The provenance record states exactly what was downloaded, from where, when, and with which software.
- **No redistribution.** We link to the original public source instead of sharing copies of Billboard data (Section 1.8).
- **Raw data stays raw.** The downloaded file is saved without any modification. All changes happen later, in documented cleaning steps.

### **3.1 Download the pinned file**

In [ ]:
def download_bytes(url, attempts=3, wait_seconds=5, timeout_seconds=60):
    """Download a file and return its raw bytes.
    Retries a few times, because temporary network errors are common."""
    last_error = None
    for attempt in range(1, attempts + 1):
        try:
            response = requests.get(url, timeout=timeout_seconds)
            response.raise_for_status()         # turns HTTP errors (e.g. 404 Not Found) into exceptions
            return response.content             # the file's exact bytes
        except requests.RequestException as error:
            last_error = error
            print(f"Attempt {attempt}/{attempts} failed: {error}")
            if attempt < attempts:
                time.sleep(wait_seconds)        # wait before trying again
    raise RuntimeError(f"Download failed after {attempts} attempts: {url}") from last_error

# Record when the data was accessed (UTC, to the second).
ACCESS_TIME_UTC = datetime.now(timezone.utc).replace(microsecond=0)
raw_bytes = download_bytes(RAW_URL)

print(f"Accessed at (UTC): {ACCESS_TIME_UTC.isoformat()}")
print(f"Downloaded       : {len(raw_bytes):,} bytes")

### **3.2 Verify integrity (SHA-256 fingerprint)**

A SHA-256 fingerprint is a 64-character code calculated from a file's exact contents: the same file always gives the same code, and changing even one byte gives a completely different one. We compare the downloaded file's fingerprint with the one recorded when we chose the snapshot. Only if they match is the file saved as our raw data; otherwise the notebook stops, so that no later step can run on different data.

In [ ]:
actual_sha256 = hashlib.sha256(raw_bytes).hexdigest()   # fingerprint of what we received
actual_bytes  = len(raw_bytes)                          # size of what we received

print(f"Expected SHA-256 : {EXPECTED_SHA256}")
print(f"Actual SHA-256   : {actual_sha256}")
print(f"Expected size    : {EXPECTED_BYTES:,} bytes")
print(f"Actual size      : {actual_bytes:,} bytes")

if actual_sha256 != EXPECTED_SHA256 or actual_bytes != EXPECTED_BYTES:
    raise ValueError(
        "The downloaded file is NOT the snapshot this notebook was built on. "
        "Stop here: do not continue with different data. "
        "Check the commit SHA and URL in Section 2.2, and see Section 3.6."
    )

# Only a verified file is saved. The commit ID in the file name shows which snapshot it is.
RAW_FILE = RAW_DIR / f"hot-100-current_{SOURCE_COMMIT[:7]}.csv"
RAW_FILE.write_bytes(raw_bytes)
print(f"Integrity check passed. Raw file saved to: {RAW_FILE}")

### **3.3 First structural check**

The fingerprint proves we have the right file; this check confirms our understanding of it: that it reads as a table with the expected columns, and how many rows and weekly charts it contains, overall and within our study window. The full quality assessment follows in Section 4.

We read every column as text (`dtype=str`) and keep the text `"NA"` as written (`keep_default_na=False`). At this stage we only look at the data, so we do not let pandas guess types or convert values; type decisions are made, and justified, during cleaning.

In [ ]:
raw_hot100 = pd.read_csv(io.BytesIO(raw_bytes), dtype=str, keep_default_na=False)

actual_columns = list(raw_hot100.columns)
if actual_columns != EXPECTED_COLUMNS:
    raise ValueError(f"Unexpected columns: {actual_columns}")

# Whole file
n_rows   = len(raw_hot100)
n_weeks  = raw_hot100["chart_week"].nunique()
first_wk = raw_hot100["chart_week"].min()
last_wk  = raw_hot100["chart_week"].max()

# Study window (text comparison works because dates are YYYY-MM-DD)
in_scope      = raw_hot100["chart_week"].between(STUDY_START, STUDY_END)
n_rows_scope  = int(in_scope.sum())
n_weeks_scope = raw_hot100.loc[in_scope, "chart_week"].nunique()

print(f"Columns                     : {actual_columns}")
print(f"Rows (whole file)           : {n_rows:,}")
print(f"Chart weeks (whole file)    : {n_weeks:,}  ({first_wk} to {last_wk})")
print(f"Rows in study window        : {n_rows_scope:,}")
print(f"Chart weeks in study window : {n_weeks_scope:,}  ({STUDY_START} to {STUDY_END})")

raw_hot100.head()   # show only 5 rows (no redistribution of the full table)

### **3.4 Record provenance**

We save a machine-readable record (JSON) of exactly what was acquired: source, pinned snapshot, access time, fingerprint, size, structure, software versions, and the status of our permission request. This file is the evidence behind our Collection documentation, and it will be used to generate the README in Section 12.

In [ ]:
provenance = {
    "record_type": "data provenance – raw Billboard Hot 100 archive",
    "notebook": NOTEBOOK_NAME,
    "source": {
        "name": SOURCE_NAME,
        "repository_url": REPO_URL,
        "file_path_in_repository": SOURCE_PATH,
        "pinned_commit": SOURCE_COMMIT,
        "pinned_commit_time_utc": SOURCE_COMMIT_TIME,
        "commit_url": COMMIT_URL,
        "download_url": RAW_URL,
        "snapshot_chosen_on": SNAPSHOT_CHOSEN_ON,
        "repository_licence": "MIT (covers the repository's code and files as distributed by UT Data)",
        "underlying_data_rights": "Billboard chart data; Billboard.com Terms of Use published by "
                                  "Penske Media Corporation (PMC). See notebook Section 1.",
        "original_collection_method": "Web scraping of billboard.com by the archive maintainer "
                                      "(2022 onwards, R script run weekly by GitHub Actions); pre-2022 "
                                      "assembled from Kaggle, data.world and a Data Miner Chrome-plugin scrape.",
    },
    "this_run": {
        "accessed_at_utc": ACCESS_TIME_UTC.isoformat(),
        "sha256": actual_sha256,
        "size_bytes": actual_bytes,
        "integrity_check": "passed (matches expected SHA-256 and size)",
        "rows_total": n_rows,
        "chart_weeks_total": n_weeks,
        "first_chart_week": first_wk,
        "last_chart_week": last_wk,
        "columns": actual_columns,
        "study_window": [STUDY_START, STUDY_END],
        "rows_in_study_window": n_rows_scope,
        "chart_weeks_in_study_window": n_weeks_scope,
    },
    "environment": {
        "python": sys.version.split()[0],
        "pandas": pd.__version__,
        "requests": requests.__version__,
        "platform": platform.platform(),
        "in_colab": IN_COLAB,
        "numpy": np.__version__,
        "matplotlib": matplotlib.__version__,
    },
    "permission_request": {
        "sent_to": "research@billboard.com and licensing@pmc.com",
        "sent_on": "2026-10-29",
        "professor_cc": True,
        "reply": "phone call the same day: chart data available to Billboard subscribers (USD 191.88 per year); "
                 "no permission to use it for the project",
        "decision": "continue with the public archive; see notebook Section 1.3",
    },
}

PROVENANCE_FILE = OUTPUT_DIR / "provenance_billboard.json"
PROVENANCE_FILE.write_text(json.dumps(provenance, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Provenance record written to: {PROVENANCE_FILE}")
print(json.dumps(provenance["this_run"], indent=2))

### **3.5 Download the provenance record**

In [ ]:
download_file(PROVENANCE_FILE)

### **3.6 Optional: private backup of the raw file**

The pinned link works as long as the archive repository exists. If it were deleted, made private, or its history rewritten, the download in 3.1 would fail with an error (the notebook can never silently receive different data). As insurance, one group member may keep a private copy of the raw file. If ever needed, it can be loaded instead and verified with the same fingerprint check.

**This copy must stay private:** do not upload it to any repository or share it publicly (Section 1.8).

In [ ]:
DOWNLOAD_RAW_BACKUP = False   # set to True only if you are the group member keeping the backup

if DOWNLOAD_RAW_BACKUP:
    download_file(RAW_FILE)   # about 20 MB
else:
    print("Raw backup download skipped (DOWNLOAD_RAW_BACKUP = False).")

### **3.7 Summary of Section 3**

**Observations** (from the outputs above):
- The file downloaded from the pinned commit `f5801b0` matched the expected SHA-256 fingerprint and size (20,699,092 bytes).
- It has 8 columns and 355,900 rows covering 3,559 weekly charts (1958-08-04 to 2026-10-17).
- Our study window (2000-01-01 to 2025-12-27) contains 135,700 rows across 1,357 weekly charts, i.e. 100 rows per chart on average.
- The rows are not stored in date order: 2022+ charts come first.

**Interpretation and next step:** the acquisition is complete, verified and documented. The row counts are consistent with a complete set of charts for our window, but this has not yet been checked week by week. Section 4 assesses the raw data's quality (missing weeks, duplicates, impossible values, inconsistencies) before any cleaning.

## **4. Raw-data quality assessment**

Before changing anything, we diagnose the raw data: we check it against what valid Billboard Hot 100 data should look like, and record every problem with evidence. Cleaning decisions (Section 5) are then based on these findings rather than on assumptions.

**Rules we follow in this section:**
- **Observe, do not modify.** All checks run on a copy of the data; the raw table is never changed.
- **Study window only.** We assess the charts dated 2000-01-01 to 2025-12-27, because that is the data we will use. (Exception: to check the first week of 2000, we look up the previous week, 1999-12-25, in the full file.)
- **Compare the two collection processes.** The archive is built in two different ways (Section 1.5): charts before 2022 come from combined third-party datasets, charts from 2022 onwards from UT's own scraper. Data quality can differ between them, so results are shown separately for each "era".
- **Every check states what it tests and why.** Results are stored and saved to a file (Section 4.11) as evidence.

**Quality dimensions checked:**

| Dimension | Question | Section |
|---|---|---|
| Completeness | Is every weekly chart there, with all 100 positions? | 4.3 |
| Missing values | How are missing values written, and where are they? | 4.4 |
| Validity | Are values in the right format and in possible ranges? | 4.5 |
| Uniqueness | Are there duplicate rows? | 4.6 |
| Consistency across weeks | Does each chart agree with the previous week's chart? | 4.7 |
| Text consistency | Are titles and performer names written consistently? | 4.8 |

Accuracy, i.e. whether the values match what Billboard actually published, cannot be checked directly, because we do not access billboard.com (Section 1.8). The consistency checks in 4.7 are our indirect test of it.

### **4.1 Assessment copy and collection eras**

In [ ]:
# Every check below stores its result here; all results are saved at the end of Section 4.
QA_RESULTS = {}

# Work on a COPY of the study-window rows: the raw table itself is never changed.
qa = raw_hot100.loc[in_scope].copy()

# Label each row with the collection process ("era") it comes from (Section 1.5).
ERA_SPLIT = "2022-01-01"
qa["era"] = np.where(qa["chart_week"] < ERA_SPLIT, "pre-2022 archive", "2022+ UT scrape")

print(f"Rows in the assessment copy: {len(qa):,}")
print(qa["era"].value_counts().to_string())

### **4.2 Column overview**

| Column | Meaning | Expected values |
|---|---|---|
| `chart_week` | Date of the chart (Billboard's official chart date, a Saturday) | `YYYY-MM-DD`, weekly |
| `current_week` | The song's **rank** on this chart (despite the name) | 1–100, each once per week |
| `title` | Song title as printed on the chart | text |
| `performer` | Artist credit as printed, including featured artists | text |
| `last_week` | Rank on the previous week's chart; empty if the song was not on it | 1–100 or missing |
| `peak_pos` | Best rank reached so far | 1–100, never worse than this week's rank |
| `wks_on_chart` | Total weeks on the chart so far, including earlier runs | 1 or more |
| `wks_at_no1` | Weeks at No. 1 (recently added by Billboard) | mostly missing |

In [ ]:
overview = pd.DataFrame({
    "distinct values": qa.drop(columns="era").nunique(),
    "example (first row)": qa.drop(columns="era").iloc[0],
})
overview

### **4.3 Completeness: every week, every position**

A complete dataset has one chart for every Saturday from 2000-01-01 to 2025-12-27 (one every 7 days), each with exactly 100 rows and every rank from 1 to 100 exactly once. We build the list of expected dates independently and compare it with the dates present, so that a missing week cannot go unnoticed.

In [ ]:
# Expected: one chart every 7 days from STUDY_START to STUDY_END.
expected_weeks = set(pd.date_range(STUDY_START, STUDY_END, freq="7D").strftime("%Y-%m-%d"))
present_weeks  = set(qa["chart_week"])
missing_weeks  = sorted(expected_weeks - present_weeks)
extra_weeks    = sorted(present_weeks - expected_weeks)

# Each week should have exactly 100 rows ...
rows_per_week = qa.groupby("chart_week").size()
weeks_not_100 = rows_per_week[rows_per_week != 100].index.tolist()

# ... and contain every rank from 1 to 100 exactly once.
full_ranks = list(range(1, 101))
ranks_complete = qa.groupby("chart_week")["current_week"].apply(
    lambda ranks: sorted(ranks.astype(int)) == full_ranks)
weeks_bad_ranks = ranks_complete[~ranks_complete].index.tolist()

completeness = {
    "expected weekly charts": len(expected_weeks),
    "weekly charts present": len(present_weeks),
    "missing weeks": len(missing_weeks),
    "unexpected extra weeks": len(extra_weeks),
    "weeks without exactly 100 rows": len(weeks_not_100),
    "weeks without ranks 1 to 100 exactly once": len(weeks_bad_ranks),
}
QA_RESULTS["completeness"] = {**completeness, "missing_week_list": missing_weeks,
                              "extra_week_list": extra_weeks}
pd.Series(completeness, name="count").to_frame()

### **4.4 Missing values and placeholders**

Missing information can be written in different ways ("NA", an empty cell, or a placeholder number such as 0). We count each form, per column and per era, because the same meaning written two ways must be harmonised before analysis.

In [ ]:
value_cols = [c for c in qa.columns if c != "era"]
placeholder_rules = {
    "'NA'":  lambda s: s == "NA",
    "empty": lambda s: s.str.strip() == "",
    "'0'":   lambda s: s == "0",
}

rows = []
for era, group in qa.groupby("era"):
    for col in value_cols:
        row = {"era": era, "column": col}
        for label, rule in placeholder_rules.items():
            row[label] = int(rule(group[col]).sum())
        rows.append(row)

placeholder_table = pd.DataFrame(rows).set_index(["era", "column"])
placeholder_table = placeholder_table[placeholder_table.sum(axis=1) > 0]   # keep only rows with something to report
QA_RESULTS["placeholders"] = {f"{era} | {col}": r.to_dict()
                              for (era, col), r in placeholder_table.iterrows()}
placeholder_table

### **4.5 Validity: formats and possible values**

We check that every numeric column contains only whole numbers, that chart dates are real dates falling on a Saturday, and that values respect the rules of the Hot 100: ranks and peaks lie between 1 and 100, and a song's peak can never be worse (a higher number) than its current or last-week rank. For these rules, both missing codes for `last_week` ("NA" and "0") are treated as "no last-week rank", following 4.4.

In [ ]:
numeric_cols = ["current_week", "last_week", "peak_pos", "wks_on_chart"]
format_summary = {}
for col in numeric_cols:
    values = qa.loc[qa[col] != "NA", col]
    is_digits = values.str.fullmatch(r"\d+")          # True if the text is only digits
    as_int = values[is_digits].astype(int)
    format_summary[col] = {"non-integer values": int((~is_digits).sum()),
                           "min": int(as_int.min()), "max": int(as_int.max())}
QA_RESULTS["formats"] = format_summary
display(pd.DataFrame(format_summary).T)

chart_dates = pd.to_datetime(qa["chart_week"], format="%Y-%m-%d", errors="coerce")
QA_RESULTS["chart_dates"] = {"unparseable": int(chart_dates.isna().sum()),
                             "weekdays": chart_dates.dt.day_name().value_counts().to_dict()}
print(f"Unparseable chart dates : {QA_RESULTS['chart_dates']['unparseable']}")
print(f"Weekday of chart dates  : {QA_RESULTS['chart_dates']['weekdays']}")

Result:
* There are no non-integer values.
* last_week has a minimum of 0 (the placeholder from 4.4).
* wks_on_chart has a maximum of 112 (“Lose Control”, Teddy Swims).
* All 135,700 dates are valid Saturdays.

In [ ]:
rank      = qa["current_week"].astype(int)
peak      = qa["peak_pos"].astype(int)
last_rank = pd.to_numeric(qa["last_week"].replace({"NA": None, "0": None}))   # both codes → missing

rule_checks = {
    "rank outside 1-100":           int((~rank.between(1, 100)).sum()),
    "peak outside 1-100":           int((~peak.between(1, 100)).sum()),
    "last-week rank outside 1-100": int((last_rank.notna() & ~last_rank.between(1, 100)).sum()),
    "peak worse than this week":    int((peak > rank).sum()),
    "peak worse than last week":    int((last_rank.notna() & (peak > last_rank)).sum()),
}
QA_RESULTS["value_rules"] = rule_checks
pd.Series(rule_checks, name="rows breaking the rule").to_frame()

### **4.6 Uniqueness: duplicate rows**

In [ ]:
duplicates = {
    "identical rows":              int(qa.duplicated().sum()),
    "same week and rank twice":    int(qa.duplicated(["chart_week", "current_week"]).sum()),
    "same song twice in one week": int(qa.duplicated(["chart_week", "title", "performer"]).sum()),
}
QA_RESULTS["duplicates"] = duplicates
pd.Series(duplicates, name="rows").to_frame()

### **4.7 Consistency across weeks**

Consecutive charts must agree with each other. If a song claims it was at rank *k* last week, last week's chart must show it at rank *k*; its weeks-on-chart must have increased by exactly 1; its peak must equal the better of its previous peak and its current rank; and a debut (weeks on chart = 1) cannot have a last-week rank. Because we cannot compare with billboard.com, these internal agreements are our best test of accuracy.

A song is identified here by its exact title and performer text. A song whose name is spelled differently from one week to the next will therefore look "not found" in the previous chart; 4.8 investigates this.

Joins in this notebook declare their expected relationship (one-to-one or many-to-one) and check the row count, so a duplicated key stops the notebook instead of silently multiplying rows.

| Check | Rule being tested |
|---|---|
| A | Has a last-week rank, but is not on the previous chart |
| B | Last-week rank differs from its rank on the previous chart |
| C | Is on the previous chart, but has no last-week rank |
| D | Weeks on chart is not the previous value + 1 |
| E | Peak is not the better of previous peak and current rank |
| F | Debut (weeks on chart = 1) but has a last-week rank |

In [ ]:
# Previous-week lookup table: every chart from the week before the study window onwards
# (so 2000-01-01 can be checked against 1999-12-25). Earlier weeks are never needed, and the
# raw file has 13 duplicate (week, title, performer) rows in late 1990 that would break the key.
FIRST_PREV_WEEK = (pd.Timestamp(STUDY_START) - pd.Timedelta(days=7)).strftime("%Y-%m-%d")
previous_chart = (raw_hot100.loc[raw_hot100["chart_week"] >= FIRST_PREV_WEEK,
                                 ["chart_week", "title", "performer", "current_week", "peak_pos", "wks_on_chart"]]
                  .rename(columns={"chart_week": "prev_week", "current_week": "prev_rank",
                                   "peak_pos": "prev_peak", "wks_on_chart": "prev_wks"}))

# Attach to every row the same song's values from the previous week (if it was there).
# Join contract: many-to-one (each row has at most one partner), so the row count must not change.
cont = qa.copy()
cont["prev_week"] = (pd.to_datetime(cont["chart_week"]) - pd.Timedelta(days=7)).dt.strftime("%Y-%m-%d")
cont = cont.merge(previous_chart, on=["prev_week", "title", "performer"], how="left", validate="many_to_one")
if len(cont) != len(qa):
    raise ValueError(f"Join contract broken: {len(qa):,} rows became {len(cont):,}.")

claims_last = ~cont["last_week"].isin(["NA", "0"])     # row gives a real last-week rank
on_prev     = cont["prev_rank"].notna()                 # song found in previous chart
wks         = cont["wks_on_chart"].astype(int)
this_rank   = cont["current_week"].astype(int)
this_peak   = cont["peak_pos"].astype(int)
prev_rank   = pd.to_numeric(cont["prev_rank"])
prev_peak   = pd.to_numeric(cont["prev_peak"])
prev_wks    = pd.to_numeric(cont["prev_wks"])

cont["A_last_week_but_not_on_previous_chart"]  = claims_last & ~on_prev
cont["B_last_week_differs_from_previous_rank"] = claims_last & on_prev & (pd.to_numeric(cont["last_week"], errors="coerce") != prev_rank)
cont["C_on_previous_chart_but_no_last_week"]   = ~claims_last & on_prev
cont["D_weeks_not_previous_plus_1"]            = on_prev & (wks != prev_wks + 1)
cont["E_peak_not_min_of_previous_and_now"]     = on_prev & (this_peak != np.minimum(prev_peak, this_rank))
cont["F_debut_with_last_week"]                 = (wks == 1) & claims_last

check_cols = [c for c in cont.columns if c[:2] in {"A_", "B_", "C_", "D_", "E_", "F_"}]
continuity_table = cont.groupby("era")[check_cols].sum().T
continuity_table["total"] = continuity_table.sum(axis=1)
QA_RESULTS["continuity"] = continuity_table["total"].astype(int).to_dict()
continuity_table

#### **4.7b Where are the inconsistencies concentrated?**

In [ ]:
for col in check_cols:
    flagged = cont.loc[cont[col], "chart_week"]
    if len(flagged):
        print(f"{col}: {len(flagged)} rows; weeks with most cases: {flagged.value_counts().head(3).to_dict()}")

#### **4.7c Investigating the 2018-01-06 chart, and frozen holiday charts**

The previous step showed that all last-week, weeks-on-chart and peak inconsistencies come from a single chart (2018-01-06). We examine it more closely. We also check for "frozen" charts (two consecutive charts that are identical), a known Billboard practice over some past holiday seasons that would make a week's data a copy of the previous one.

In [ ]:
# Frozen charts: identical song order in two consecutive weeks (checked from 1999-12-25 to the end of our window).
week_lists = {week: grp.sort_values("current_week", key=lambda s: s.astype(int))[["title", "performer"]]
              .apply(tuple, axis=1).tolist()
              for week, grp in raw_hot100.groupby("chart_week")}
weeks_sorted = sorted(week_lists)
frozen = [(a, b) for a, b in zip(weeks_sorted, weeks_sorted[1:])
          if a >= "1999-12-25" and b <= STUDY_END and week_lists[a] == week_lists[b]]
QA_RESULTS["frozen_charts_in_window"] = frozen
print(f"Identical consecutive charts in our window: {len(frozen)} {frozen}")

# The 2018-01-06 chart compared with 2017-12-30.
c18 = cont[cont["chart_week"] == "2018-01-06"]
jump2 = int((pd.to_numeric(c18["wks_on_chart"]) - pd.to_numeric(c18["prev_wks"]) == 2).sum())
print(f"2018-01-06: songs whose weeks-on-chart jumps by 2 instead of 1: {jump2} of {int(c18['prev_wks'].notna().sum())} songs also on 2017-12-30")
QA_RESULTS["chart_2018_01_06"] = {"weeks_on_chart_jumps_by_2": jump2}

#### **4.7d Examples: songs "not found" in the previous chart**

In [ ]:
cols_show = ["chart_week", "current_week", "title", "performer", "last_week", "prev_rank"]
cont.loc[cont["A_last_week_but_not_on_previous_chart"], cols_show].head(15)

### **4.8 Text consistency: titles and performer names**

Song titles and performer names are how we identify songs (and later how we will match them with other sources), so inconsistent spelling would split one song into several. We check for stray spaces, special characters, and songs whose title and performer differ only in capitalisation, spacing or apostrophe style. We also count the different ways featured or collaborating artists are written.

In [ ]:
def text_profile(s):
    return {
        "leading/trailing spaces": int((s != s.str.strip()).sum()),
        "double spaces":           int(s.str.contains("  ", regex=False).sum()),
        "curly apostrophe (’)":    int(s.str.contains("’", regex=False).sum()),
        "other non-ASCII":         int(s.str.contains(r"[^\x00-\x7F’]", regex=True).sum()),
    }

text_table = pd.DataFrame({col: text_profile(qa[col]) for col in ["title", "performer"]})
QA_RESULTS["text_profile"] = text_table.to_dict()
text_table

In [ ]:
def loose(s):
    """Lower-case, straight apostrophes, single spaces: used only to DETECT variants."""
    return (s.str.lower()
             .str.replace("’", "'", regex=False)
             .str.replace(r"\s+", " ", regex=True)
             .str.strip())

songs = qa[["title", "performer"]].drop_duplicates().copy()
songs["loose_key"] = loose(songs["title"]) + " | " + loose(songs["performer"])
variants = songs[songs.duplicated("loose_key", keep=False)].sort_values("loose_key")

QA_RESULTS["song_identity"] = {
    "distinct exact title+performer pairs": len(songs),
    "distinct after loose matching": int(songs["loose_key"].nunique()),
    "pairs that are case/space/apostrophe variants": len(variants),
}
display(pd.Series(QA_RESULTS["song_identity"]).to_frame("count"))
variants[["title", "performer"]]

In [ ]:
credit_patterns = {
    "Featuring": r"\bFeaturing\b",
    "Feat./Ft.": r"(?i)\b(?:feat|ft)\.",
    "&":         r"&",
    " x ":       r"\sx\s",
    " X ":       r"\sX\s",
    "With":      r"\bWith\b",
    " And ":     r"\sAnd\s",
    "comma":     r",",
}
performers = qa[["performer", "era"]].drop_duplicates()
credit_table = pd.DataFrame({
    name: performers.groupby("era")["performer"].apply(
        lambda s, p=pattern: int(s.str.contains(p, regex=True).sum()))
    for name, pattern in credit_patterns.items()}).T
QA_RESULTS["credit_separators"] = credit_table.to_dict()
credit_table

### **4.9 Preliminary profile of the raw data**

To understand the data's overall shape, we compute a few descriptive figures. **They are preliminary:** they count songs by their exact raw spelling, so the variants found in 4.8 make them very slightly inaccurate. The descriptive analysis proper is done after cleaning.

In [ ]:
year = qa["chart_week"].str[:4].astype(int)
songs_per_year  = qa.assign(year=year).drop_duplicates(["year", "title", "performer"]).groupby("year").size()
debuts_per_year = qa.assign(year=year).loc[qa["wks_on_chart"] == "1"].groupby("year").size()
longest_run     = qa.groupby(["title", "performer"])["wks_on_chart"].apply(lambda s: s.astype(int).max())

profile = {
    "distinct songs (exact title+performer)": len(longest_run),
    "distinct performer strings": int(qa["performer"].nunique()),
    "debut rows (weeks on chart = 1)": int((qa["wks_on_chart"] == "1").sum()),
    "songs with only 1 week on chart": int((longest_run == 1).sum()),
    "median weeks on chart per song": float(longest_run.median()),
    "max weeks on chart": int(longest_run.max()),
}
QA_RESULTS["preliminary_profile"] = profile
display(pd.Series(profile).to_frame("value"))
pd.DataFrame({"distinct songs": songs_per_year, "debuts": debuts_per_year}).T

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(debuts_per_year.index, debuts_per_year.values, color=COLOR_MAIN, linewidth=2, marker="o", markersize=4)
ax.set_title("New songs entering the Hot 100 each year (raw data, preliminary)")
ax.set_xlabel("Year of chart date")
ax.set_ylabel("Debuts (weeks on chart = 1)")
ax.set_ylim(bottom=0)
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(longest_run, bins=range(1, int(longest_run.max()) + 2), color=COLOR_MAIN, edgecolor="white")
ax.set_title("Weeks each song spent on the Hot 100 (raw data, preliminary)")
ax.set_xlabel("Highest weeks-on-chart value recorded for the song")
ax.set_ylabel("Number of songs")
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

### **4.10 Summary of findings**

**No problems found:**
- All 1,357 weekly charts are present, each with exactly 100 rows and ranks 1–100 once each.
- No duplicates; all values are whole numbers in possible ranges; all chart dates are valid Saturdays; no peak is worse than a current or last-week rank.
- No "frozen" (identical consecutive) charts in our window.

**Problems found** (observations; treatments are proposals, decided and justified in Section 5):

| # | Observation | Rows affected | Interpretation | Proposed treatment |
|---|---|---|---|---|
| 1 | "Not on last week's chart" is coded `NA` before 2022 and `0` from 2022 | 11,154 + 3,203 | Two collection processes, two codes for the same meaning (confirmed: checks C and F = 0) | Convert both to one missing-value code |
| 2 | `wks_at_no1` empty for every row in our window | 135,700 (100%) | Column only recorded since Sept 2026 | Drop the column |
| 3 | The 2018-01-06 chart disagrees with 2017-12-30: 68 last-week ranks, all 83 weeks-on-chart values (+2 instead of +1), 9 peaks | 1 chart | Cause unknown; one of the two charts is likely wrong in the source; cannot be verified without billboard.com | Keep ranks as recorded; do not rely on the supplied `last_week`/`wks_on_chart`/`peak_pos` — recompute what we need from the sequence of charts; report as a limitation |
| 4 | Same song with different capitalisation, spacing or apostrophe | 38 pairs (19 songs), almost all 2022+ | Name styling changes (some genuine, e.g. "Charli xcx") | Build a normalised song key for identifying songs; keep original text for display |
| 5 | Same song with a changed credit (remixes adding an artist, holiday songs credited differently across years) | most of the 61 check-A rows | Credits are not stable identifiers | Decide identity rules in cleaning; document remaining cases as a limitation |
| 6 | Collaborations written in many ways ("Featuring", "Feat.", "Ft.", "&", "x", "With", "And", commas) | thousands of credits | Inconsistent conventions across sources | Standardise before splitting credits into individual artists |
| 7 | Double spaces and curly apostrophes in names | 19 + 21 rows | Typing/formatting noise | Normalise in the song key |
| 8 | Rows not stored in date order | whole file | Result of how UT stacks its files | Sort by chart date and rank |
| 9 | Column `current_week` actually holds the rank | — | Misleading name | Rename (e.g. `rank`) |
| 10 | Songs on the chart at the start of 2000 debuted before our window | at most 100 songs | Their entry is not observed (left-censoring) | Flag them; exclude from "first entry" analyses |
| 11 | Debuts per year double between the early 2000s and the 2020s; spikes at 20 and ~52 weeks on chart | — | Chart-rule effects, not data errors | No cleaning; account for this in analysis |

### **4.11 Save and download the quality assessment**

In [ ]:
QA_FILE = OUTPUT_DIR / "quality_assessment_billboard.json"
qa_record = {
    "record_type": "raw-data quality assessment – Billboard Hot 100 (study window)",
    "source_commit": SOURCE_COMMIT,
    "source_sha256": actual_sha256,
    "study_window": [STUDY_START, STUDY_END],
    "assessed_at_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    "results": QA_RESULTS,
}
QA_FILE.write_text(json.dumps(qa_record, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
print(f"Quality assessment written to: {QA_FILE}")
download_file(QA_FILE)

## **5. Cleaning**

This section turns the raw data into clean, analysis-ready tables. Every step responds to a finding of Section 4, and every decision is justified.

**Principles:**
- **The raw data is never modified.** Cleaning works on a copy; the raw file (Section 3) stays as downloaded.
- **Original values are kept where possible.** We add columns (normalised keys, flags, identifiers) rather than overwrite titles, names or reported values, so nothing is lost and every change can be traced.
- **We flag what we cannot fix.** Where the correct value cannot be known without the original source, the row is flagged, not altered. Changing values without evidence would be inventing data.
- **Every step is logged.** A cleaning log (rows before/after, values changed, justification) is saved and downloaded at the end.
- **The result is validated.** Section 5.13 re-runs the key checks of Section 4 on the clean data.

**Steps:**

| Step | What | Responds to |
|---|---|---|
| C1 | Keep the study window | scope |
| C2 | Rename columns | 4.10 #9 |
| C3 | Drop `wks_at_no1` | 4.10 #2 |
| C4 | Unify the two "not on last week's chart" codes | 4.10 #1 |
| C5 | Convert data types | 4.5 |
| C6 | Sort by date and rank | 4.10 #8 |
| C7 | Add source era; flag the 2018-01-06 chart | 4.10 #3 |
| C8 | Normalised name keys | 4.10 #4, #6, #7 |
| C9 | Song identifier, linking renamed entries | 4.10 #5 |
| C10–C11 | Flag songs whose first entry lies outside the window | 4.10 #10 |
| C12 | One-row-per-song table | our outcome |

### **5.1 Cleaning log**

In [ ]:
CLEANING_LOG = []

def log_step(step, description, rows_before, rows_after, values_changed=0, justification=""):
    """Record one cleaning step so the whole cleaning process can be traced afterwards."""
    CLEANING_LOG.append({
        "step": step, "description": description,
        "rows_before": int(rows_before), "rows_after": int(rows_after),
        "values_changed": int(values_changed), "justification": justification,
    })
    print(f"[{step}] {description}\n      rows {rows_before:,} → {rows_after:,} | values changed: {values_changed:,}")

### **5.2 Study window (C1)**

In [ ]:
clean = raw_hot100.loc[in_scope].copy()
log_step("C1", "Keep only charts dated within the study window",
         len(raw_hot100), len(clean),
         justification=f"Our question covers charts dated {STUDY_START} to {STUDY_END}.")

### **5.3 Clear column names (C2)**

In [ ]:
RENAME = {
    "chart_week":   "chart_date",
    "current_week": "rank",
    "last_week":    "last_week_rank",
    "peak_pos":     "peak_rank_reported",
    "wks_on_chart": "weeks_on_chart_reported",
}
clean = clean.rename(columns=RENAME)
log_step("C2", "Rename columns to describe their content", len(clean), len(clean),
         justification="current_week holds a rank, not a week; 'reported' marks values supplied by the source "
                       "that we do not fully trust (Section 4.7).")

### **5.4 Remove the empty column (C3)**

In [ ]:
if not (clean["wks_at_no1"] == "NA").all():
    raise ValueError("wks_at_no1 is not empty in the window – re-examine before dropping it.")
clean = clean.drop(columns="wks_at_no1")
log_step("C3", "Drop the wks_at_no1 column", len(clean), len(clean),
         justification="Empty for 100% of rows in the window (Section 4.4); it carries no information.")

### **5.5 One code for "not on last week's chart" (C4)**

In [ ]:
is_missing_code = clean["last_week_rank"].isin(["NA", "0"])
n_zero_codes = int((clean["last_week_rank"] == "0").sum())
clean.loc[is_missing_code, "last_week_rank"] = None
log_step("C4", "Treat both codes for 'not on last week's chart' (NA and 0) as missing",
         len(clean), len(clean), values_changed=n_zero_codes,
         justification="0 is not a valid rank; checks C and F in Section 4.7 show 0 is used exactly where "
                       "NA is used before 2022.")

### **5.6 Data types (C5)**

In [ ]:
clean["chart_date"] = pd.to_datetime(clean["chart_date"], format="%Y-%m-%d")
for col in ["rank", "last_week_rank", "peak_rank_reported", "weeks_on_chart_reported"]:
    clean[col] = pd.to_numeric(clean[col]).astype("Int64")
log_step("C5", "Convert chart dates to dates and ranks/counts to whole numbers", len(clean), len(clean),
         justification="Allows date arithmetic and numeric comparisons; Int64 keeps missing values without "
                       "turning whole numbers into decimals.")
clean.dtypes.to_frame("type")

### **5.7 Order (C6)**

In [ ]:
clean = clean.sort_values(["chart_date", "rank"]).reset_index(drop=True)
log_step("C6", "Sort rows by chart date, then rank", len(clean), len(clean),
         justification="The source file is not in date order (Section 3.3); week-to-week steps need it.")

### **5.8 Provenance and flag columns (C7)**

We keep the collection era as a column, so any later result can be checked for differences between the two collection processes. The 2018-01-06 chart is **flagged, not changed and not removed**: its ranks are valid (complete, 1–100, consistent with peaks), but its last-week, peak and weeks-on-chart values disagree with the previous chart, and without access to billboard.com we cannot know which of the two charts is wrong. Removing it would create a gap in an otherwise complete series; correcting it would mean inventing values.

In [ ]:
ERA_SPLIT_DATE = pd.Timestamp("2022-01-01")
INCONSISTENT_CHART = pd.Timestamp("2018-01-06")
clean["source_era"] = np.where(clean["chart_date"] < ERA_SPLIT_DATE, "pre-2022 archive", "2022+ UT scrape")
clean["flag_inconsistent_chart"] = clean["chart_date"] == INCONSISTENT_CHART
log_step("C7", "Add source_era and flag the 2018-01-06 chart", len(clean), len(clean),
         justification="Keeps provenance visible in the clean data; the flagged chart's last-week, peak and "
                       "weeks values are internally inconsistent (Section 4.7c) and cannot be corrected without "
                       "the original source, so they are flagged rather than changed.")
print(f"Rows flagged: {int(clean['flag_inconsistent_chart'].sum())}")

### **5.9 Normalised name keys (C8)**

To decide whether two rows are the same song, we compare simplified versions of their title and performer ("keys"). The original text is kept unchanged for display.

| Rule | Example | Why |
|---|---|---|
| Lower case | `Charli XCX` = `Charli xcx` | Capitalisation styling changes (Section 4.8) |
| All apostrophe styles → `'` | `Victoria’s` = `Victoria's` | Typographic variants |
| `Feat.` / `Ft.` → `featuring` | `Ariana Grande Feat. Doja Cat` | Older rows abbreviate (Section 4.8) |
| `&` → `and` | `A & B` = `A and B` | Same meaning |
| Remove punctuation and symbols | `'Til You Can't` = `Til You Can't`; `Nat "King" Cole` = `Nat King Cole` | Punctuation varies across sources |
| Collapse spaces | `W O R K I N  M E` | Double spaces (Section 4.8) |

Risk: too aggressive a key could merge two *different* songs. We therefore list every merge the keys produce and inspect them.

In [ ]:
def make_key(text):
    """Simplified version of a title or performer, used only to decide whether two names are the same."""
    return (text.str.lower()
                .str.replace("[’‘`´]", "'", regex=True)                  # all apostrophe styles → '
                .str.replace(r"\b(?:feat|ft)\.", "featuring", regex=True) # Feat. / Ft. → featuring
                .str.replace("&", " and ", regex=False)                    # & → and
                .str.replace(r"[^a-z0-9 ]", "", regex=True)                # drop punctuation and symbols
                .str.replace(r"\s+", " ", regex=True)                      # collapse repeated spaces
                .str.strip())

clean["title_key"]     = make_key(clean["title"])
clean["performer_key"] = make_key(clean["performer"])

if (clean["title_key"] == "").any() or (clean["performer_key"] == "").any():
    raise ValueError("A key became empty – make_key removed too much.")

n_exact = len(clean[["title", "performer"]].drop_duplicates())
n_keyed = len(clean[["title_key", "performer_key"]].drop_duplicates())
log_step("C8", "Add normalised title and performer keys (original text kept)", len(clean), len(clean),
         justification=f"Spelling variants split songs (Section 4.8): {n_exact:,} exact title+performer "
                       f"pairs become {n_keyed:,} keyed pairs.")

In [ ]:
# Every group of original spellings that the keys treat as one: inspect for wrong merges.
merged_spellings = (clean[["title", "performer", "title_key", "performer_key"]].drop_duplicates()
                    .groupby(["title_key", "performer_key"]).filter(lambda g: len(g) > 1)
                    .sort_values(["title_key", "performer_key"]))
merged_spellings[["title", "performer"]]

   We inspected every group above: each is one song written in different ways (capitalisation, punctuation, "HUNTRX" vs "HUNTR/X:", "Sing!" vs "Sing"). No different songs were merged.

### **5.10 Song identifier (C9)**

Some songs appear under a changed credit from one week to the next (e.g. "Die For You" by The Weeknd, then by "The Weeknd & Ariana Grande" after the remix). Keys cannot fix this, because the names genuinely differ.

**Our rule uses Billboard's own evidence.** Each row's last-week rank says where the song was the week before. If that position on the previous chart holds an entry with a *different* name, but the **same title or the same performer**, then Billboard itself treats the two as one continuing chart entry, so we link them.

Safeguards:
- the 2018-01-06 chart is excluded, because its last-week ranks are unreliable (5.8);
- links whose title *and* performer both differ are rejected;
- after linking, no song may appear twice in the same week; if it did, two different songs would have been merged, and the notebook stops.

Linked names are collected into groups; each group is one song and receives a `song_id`.

In [ ]:
# Each distinct (title_key, performer_key) pair gets a number; linking puts numbers into shared groups.
pairs = clean[["title_key", "performer_key"]].drop_duplicates()
pair_id = {pair: i for i, pair in enumerate(pairs.itertuples(index=False, name=None))}
parent = list(range(len(pair_id)))      # at the start, every pair is its own group

def find(i):
    """Return the representative ('root') of the group that pair i belongs to."""
    while parent[i] != i:
        parent[i] = parent[parent[i]]   # shortcut: point closer to the root (keeps groups fast to search)
        i = parent[i]
    return i

def union(a, b):
    """Merge the groups of pairs a and b."""
    root_a, root_b = find(a), find(b)
    if root_a != root_b:
        parent[root_b] = root_a

In [ ]:
previous = clean[["chart_date", "rank", "title_key", "performer_key"]].rename(columns={
    "chart_date": "prev_date", "rank": "last_week_rank",
    "title_key": "prev_title_key", "performer_key": "prev_performer_key"})

# For each row with a last-week rank: what was at that rank on the previous chart?
# Join contract: many-to-one – each (week, rank) position holds exactly one entry (Section 4.3).
with_last_week = clean[clean["last_week_rank"].notna()]
linked = (with_last_week.assign(prev_date=with_last_week["chart_date"] - pd.Timedelta(days=7))
                        .merge(previous, on=["prev_date", "last_week_rank"], how="inner", validate="many_to_one"))
if len(linked) > len(with_last_week):
    raise ValueError("Join contract broken: rows multiplied when looking up last week's position.")
print(f"Rows with a last-week rank: {len(with_last_week):,} | found on the previous chart: {len(linked):,} "
      f"(the rest point to 1999-12-25, outside our data, or to a missing entry – Section 4.7)")
linked = linked[~linked["flag_inconsistent_chart"]]

name_changed = ((linked["title_key"] != linked["prev_title_key"]) |
                (linked["performer_key"] != linked["prev_performer_key"]))
candidates = linked[name_changed]
shares_title_or_performer = ((candidates["title_key"] == candidates["prev_title_key"]) |
                             (candidates["performer_key"] == candidates["prev_performer_key"]))
accepted = candidates[shares_title_or_performer]
rejected = candidates[~shares_title_or_performer]

for row in accepted.itertuples():
    union(pair_id[(row.prev_title_key, row.prev_performer_key)], pair_id[(row.title_key, row.performer_key)])

print(f"Rows whose last-week rank points to a differently named entry: {len(candidates)}")
print(f"  accepted as the same song (same title or same performer): {len(accepted)}")
print(f"  rejected (nothing in common): {len(rejected)}")
accepted[["chart_date", "rank", "title", "performer", "prev_title_key", "prev_performer_key"]]

In [ ]:
root_of_pair = {pair: find(i) for pair, i in pair_id.items()}
clean["song_root"] = [root_of_pair[p] for p in zip(clean["title_key"], clean["performer_key"])]

# Readable IDs, numbered in order of first appearance: S00001, S00002, ...
first_seen_order = clean.drop_duplicates("song_root")["song_root"]
song_number = {root: f"S{n:05d}" for n, root in enumerate(first_seen_order, start=1)}
clean["song_id"] = clean["song_root"].map(song_number)
clean = clean.drop(columns="song_root")

collisions = int(clean.duplicated(["chart_date", "song_id"]).sum())
if collisions:
    raise ValueError(f"{collisions} rows: one song_id twice in the same week – linking merged different songs.")
log_step("C9", "Assign a song_id, linking renamed entries via the chart's own last-week ranks",
         len(clean), len(clean),
         justification=f"{len(accepted)} renamed entries linked; {len(rejected)} rejected; "
                       f"{clean['song_id'].nunique():,} songs; no song appears twice in a week.")
print(f"Distinct songs: {clean['song_id'].nunique():,}")

### **5.11 Is each song's first chart entry observed? (C10–C11)**

Our outcome includes *when* a song first reached the chart. That date is only known if the first entry lies inside our window. Songs already on the chart before 2000 (e.g. the whole first chart of 2000) first entered earlier: their entry is **left-censored**. We flag them rather than delete them: they are still songs that reached the chart, and the analysis decides how to treat them.

We detect them in two independent ways and compare:
1. **Direct evidence:** the song's keys appear in the raw file before 2000.
2. **The source's own counter:** at the song's first appearance in our window, its reported weeks-on-chart is above 1.

In [ ]:
pre_window = raw_hot100[raw_hot100["chart_week"] < STUDY_START]
pre_window_pairs = set(zip(make_key(pre_window["title"]), make_key(pre_window["performer"])))
pair_charted_before = pd.Series(
    [p in pre_window_pairs for p in zip(clean["title_key"], clean["performer_key"])], index=clean.index)
songs_charted_before = set(clean.loc[pair_charted_before, "song_id"])
clean["charted_before_window"] = clean["song_id"].isin(songs_charted_before)
log_step("C10", "Flag songs that were already on the chart before 2000", len(clean), len(clean),
         justification="Their first entry is outside our window (left-censoring); found by matching keys "
                       "against the pre-2000 part of the raw file.")
print(f"Songs that charted before 2000: {len(songs_charted_before):,}")

In [ ]:
first_rows = clean.sort_values(["chart_date", "rank"]).drop_duplicates("song_id")
display(pd.crosstab(first_rows["charted_before_window"],
                    first_rows["weeks_on_chart_reported"] > 1,
                    rownames=["found before 2000 (our flag)"],
                    colnames=["source: weeks on chart > 1 at first appearance"]))

reported_earlier_weeks = set(first_rows.loc[first_rows["weeks_on_chart_reported"] > 1, "song_id"])
uncertain = songs_charted_before | reported_earlier_weeks
clean["first_entry_uncertain"] = clean["song_id"].isin(uncertain)
log_step("C11", "Flag songs whose first chart entry is not observed in the window", len(clean), len(clean),
         justification=f"{len(songs_charted_before)} songs found before 2000, plus "
                       f"{len(reported_earlier_weeks - songs_charted_before)} whose own weeks-on-chart counter shows "
                       f"earlier weeks we cannot match (e.g. an earlier run under another name): "
                       f"{len(uncertain)} songs in total.")

unmatched = first_rows[~first_rows["charted_before_window"] & (first_rows["weeks_on_chart_reported"] > 1)]
unmatched[["chart_date", "rank", "title", "performer", "weeks_on_chart_reported"]]

### **5.12 One row per song (C12)**

Our outcome (whether and when a song reached the chart) describes a song, not a chart week. We therefore build a second table with one row per song, alongside the weekly table. Keeping both, rather than one wide table, avoids repeating song information in every week (tidy data) and prepares the storage design of Section 6.

For songs with several spellings, the displayed title and performer are the spelling used in most weeks (ties broken alphabetically, so the result is always the same).

In [ ]:
def most_common_spelling(names):
    return names.mode().iloc[0]

songs = (clean.groupby("song_id")
              .agg(title=("title", most_common_spelling),
                   performer=("performer", most_common_spelling),
                   first_chart_date=("chart_date", "min"),
                   last_chart_date=("chart_date", "max"),
                   weeks_in_window=("chart_date", "size"),
                   best_rank_in_window=("rank", "min"),
                   charted_before_window=("charted_before_window", "first"),
                   first_entry_uncertain=("first_entry_uncertain", "first"))
              .reset_index())

first_appearance = clean.sort_values(["chart_date", "rank"]).drop_duplicates("song_id").set_index("song_id")
songs["entry_rank"] = songs["song_id"].map(first_appearance["rank"]).astype("Int64")

by_song = clean.sort_values(["song_id", "chart_date"])
starts_new_run = by_song.groupby("song_id")["chart_date"].diff() != pd.Timedelta(days=7)
songs["runs_in_window"] = songs["song_id"].map(starts_new_run.groupby(by_song["song_id"]).sum()).astype(int)

name_variants = clean.drop_duplicates(["song_id", "title", "performer"]).groupby("song_id").size()
songs["name_variants"] = songs["song_id"].map(name_variants).astype(int)

songs = songs[["song_id", "title", "performer", "first_chart_date", "entry_rank", "last_chart_date",
               "weeks_in_window", "runs_in_window", "best_rank_in_window", "name_variants",
               "charted_before_window", "first_entry_uncertain"]]
log_step("C12", "Build the one-row-per-song table", len(clean), len(songs),
         justification="Our outcome (whether and when a song reached the chart) describes a song, not a chart week.")
songs.head()

### **5.13 Validation of the clean data**

We re-run the key checks on the clean tables. All counts must match expectations; the continuity checks of Section 4.7, now using `song_id`, must show no remaining inconsistencies outside the flagged 2018-01-06 chart. If any check fails, the notebook stops.

In [ ]:
weekly_cols = ["chart_date", "rank", "song_id", "title", "performer", "last_week_rank",
               "peak_rank_reported", "weeks_on_chart_reported", "title_key", "performer_key",
               "source_era", "flag_inconsistent_chart", "charted_before_window", "first_entry_uncertain"]
weekly = clean[weekly_cols]

# Join contract: many-to-one – a song appears at most once per week, so each row has at most one partner.
prev_ids = weekly[["chart_date", "song_id", "rank"]].rename(columns={"chart_date": "prev_date", "rank": "prev_rank"})
chk = (weekly.assign(prev_date=weekly["chart_date"] - pd.Timedelta(days=7))
             .merge(prev_ids, on=["prev_date", "song_id"], how="left", validate="many_to_one"))
if len(chk) != len(weekly):
    raise ValueError(f"Join contract broken: {len(weekly):,} rows became {len(chk):,}.")
in_window_prev = chk["prev_date"] >= pd.Timestamp(STUDY_START)
not_flagged = ~chk["flag_inconsistent_chart"]

validation = {
    "rows (expected 135,700)": len(weekly),
    "weekly charts (expected 1,357)": weekly["chart_date"].nunique(),
    "weeks without ranks 1-100 exactly once": int(weekly.groupby("chart_date")["rank"]
                                                  .apply(lambda r: sorted(r) != list(range(1, 101))).sum()),
    "same song twice in one week": int(weekly.duplicated(["chart_date", "song_id"]).sum()),
    "missing values outside last_week_rank": int(weekly.drop(columns="last_week_rank").isna().sum().sum()),
    "missing last_week_rank (expected 14,357)": int(weekly["last_week_rank"].isna().sum()),
    "last-week rank given but song not on previous chart (excl. 2018-01-06)":
        int((in_window_prev & chk["last_week_rank"].notna() & chk["prev_rank"].isna() & not_flagged).sum()),
    "last-week rank differs from previous chart (excl. 2018-01-06)":
        int((in_window_prev & chk["prev_rank"].notna() & (chk["last_week_rank"] != chk["prev_rank"]) & not_flagged).sum()),
    "songs": len(songs),
}

must_be_zero = [k for k in validation if k.startswith(("weeks without", "same song", "missing values outside", "last-week rank"))]
failed = {k: validation[k] for k in must_be_zero if validation[k] != 0}
if failed:
    raise ValueError(f"Validation failed: {failed}")
pd.Series(validation, name="value").to_frame()

### **5.14 Save the clean tables and the cleaning log**

The clean tables are written as CSV files inside this Colab session, where later sections can use them. CSV here is an interim working format; the storage format is chosen and justified in Section 6. The data files are **not** downloaded by default (Section 1.8: the chart data is not redistributed, and anyone can regenerate them by running this notebook). The cleaning log is downloaded, since it is our documentation.

In [ ]:
CLEAN_DIR = OUTPUT_DIR / "clean"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
WEEKLY_FILE = CLEAN_DIR / "hot100_weekly_clean.csv"
SONGS_FILE  = CLEAN_DIR / "hot100_songs.csv"
weekly.to_csv(WEEKLY_FILE, index=False, date_format="%Y-%m-%d")
songs.to_csv(SONGS_FILE, index=False, date_format="%Y-%m-%d")

CLEANING_LOG_FILE = OUTPUT_DIR / "cleaning_log_billboard.json"
CLEANING_LOG_FILE.write_text(json.dumps({
    "record_type": "cleaning log – Billboard Hot 100",
    "source_commit": SOURCE_COMMIT,
    "source_sha256": actual_sha256,
    "cleaned_at_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    "steps": CLEANING_LOG,
    "validation": validation,
    "outputs": {
        "weekly_table": {"file": WEEKLY_FILE.name, "rows": len(weekly), "columns": list(weekly.columns)},
        "songs_table":  {"file": SONGS_FILE.name,  "rows": len(songs),  "columns": list(songs.columns)},
    },
}, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
download_file(CLEANING_LOG_FILE)

DOWNLOAD_CLEAN_DATA = False   # True only for private use within the group (Section 1.8)
if DOWNLOAD_CLEAN_DATA:
    download_file(WEEKLY_FILE)
    download_file(SONGS_FILE)

pd.DataFrame(CLEANING_LOG)[["step", "description", "rows_before", "rows_after", "values_changed"]]

### **5.15 What we deliberately did not change**

| Not done | Why |
|---|---|
| Correct or delete the 2018-01-06 chart | Its ranks are valid; which neighbouring chart is wrong cannot be determined without the original source. Flagged instead (C7). |
| Edit titles or performer names | Original text is information (e.g. "Charli xcx" is the artist's real restyling). Identity uses keys instead (C8). |
| Split performer credits into individual artists | Separators are ambiguous inside band names ("Earth, Wind & Fire", "Tyler, The Creator", "Hootie & The Blowfish"); naive splitting would create false artists. Deferred to the step where we build artist-level information, with a dedicated, tested method. |
| Remove songs that charted before 2000 | They did reach the chart; whether to include them is an analysis decision. Flagged (C10–C11). |
| Remove holiday songs or unusual performers (e.g. Glee Cast) | They are real chart entries, not errors. Whether they need special treatment is an analysis question. |
| Recompute last-week, peak and weeks-on-chart columns | Not needed for our outcome (reaching the chart); the source's values are kept, clearly labelled `_reported`, and the flagged chart marks where they fail. |

### **5.16 Summary of Section 5**

**Observations:**
- All 135,700 rows in the window are kept; only one value type was changed (3,203 `0` codes → missing).
- Normalised keys merged spelling variants: 11,785 exact title+performer pairs → 11,759.
- 34 renamed chart entries were linked using Billboard's own last-week ranks, with none rejected and no song appearing twice in a week: **11,728 songs**.
- 148 songs have a first entry outside our window (133 found before 2000, 15 identified only by the source's weeks counter).
- After cleaning, no week-to-week inconsistency remains except in the flagged 2018-01-06 chart.

**Interpretation:** the clean data gives, for each song, whether and when it first reached the Hot 100 within 2000–2025 (reliable for 11,580 songs), which is what our research question requires from the Billboard data.

**Remaining limitations:**
- Identity across long gaps under a different credit can only be detected through the source's counter (the 15 cases), so some similar cases may remain undetected.
- The 2018-01-06 chart's reported values are unreliable.
- The two collection eras differ in conventions; `source_era` is kept so results can be checked by era.

## **6. Storage**

This section decides how the cleaned Billboard data is stored, and justifies that choice against alternatives.

**What we store:** four related kinds of information: weekly charts (1,357), songs (11,728), the different names under which songs were printed (11,785), and chart entries, i.e. one song at one rank in one week (135,700).

**Requirements, derived from our project:**

| # | Requirement | Why we need it |
|---|---|---|
| R1 | Runs in Colab with nothing to install and no server | Reproducibility: anyone can rerun the notebook (How to reproduce this notebook) |
| R2 | Enforces the integrity rules found in Section 4 | Our main data risks are duplicates, impossible ranks and identity errors (4.6–4.8, C9) |
| R3 | Represents relationships between songs, names and weekly entries | Songs appear in many weeks and under several names; MusicBrainz matches will add more linked tables |
| R4 | Can be queried in a readable, standard way | Analyses must be traceable by a reader (brief: reproducibility) |
| R5 | Keeps data types (dates, whole numbers, flags) | Avoids silent errors when data is reloaded |
| R6 | Proportionate to the data size (~150,000 rows) | The brief warns against choosing technology for its complexity |
| R7 | One private file, easy to keep out of public repositories | No redistribution of chart data (Section 1.8) |

### **6.1 Options considered**

| Option | What it is | Fit with our requirements | Verdict |
|---|---|---|---|
| CSV files | Plain-text tables | R1 ✓, R6 ✓; no rules enforced (R2 ✗), no relationships (R3 ✗), types lost on reload (R5 ✗) | Tested in 6.7 |
| Excel | Spreadsheet file | Same weaknesses as CSV, plus automatic reformatting of values (e.g. dates) and manual-editing risk | Rejected |
| Parquet files | Compressed, typed column files | R1 ✓, R5 ✓, R6 ✓, very small and fast; no rules or relationships (R2 ✗, R3 ✗) | Tested in 6.7 |
| **SQLite** | A relational database in a single file, built into Python | R1–R4 ✓, R6 ✓, R7 ✓; no native date or boolean type (R5 partly) | **Tested in 6.7; chosen (6.8)** |
| DuckDB | An analytical database, also single-file | Strong for analytics and supports constraints, but an extra library to install and maintain; its speed advantage is irrelevant at our size | Not needed |
| PostgreSQL | A full database server | Strongest features, but needs a running server, so a reader cannot rerun the notebook in Colab (R1 ✗); far beyond our data size (R6 ✗) | Rejected for the Billboard data |
| MongoDB | A document (non-relational) database | Our data is tabular with clear relationships; a document model adds nothing and needs a server | Rejected |

### **6.2 Data model**

We split the clean data into four tables so that each fact is stored once (normalisation), and we let the database enforce the rules found in Section 4.

| Table | One row per… | Key | Main columns |
|---|---|---|---|
| `charts` | weekly chart | `chart_date` | `source_era`, `flag_inconsistent_chart` |
| `songs` | song | `song_id` | display title and performer, `charted_before_window`, `first_entry_uncertain` |
| `song_names` | way a song was printed | `name_id` | `song_id`, title and performer as printed, their normalised keys |
| `chart_entries` | song at a rank in a week | (`chart_date`, `rank`) | `song_id`, `name_id`, last-week rank, reported peak and weeks |

**Relationships:** each chart entry belongs to one chart, one song and one printed name; each song has one or more printed names.

```
  charts ──1:many──> chart_entries <──many:1── songs ──1:many──> song_names
                            │                                        ▲
                            └─────────── many:1 (name_id) ───────────┘
```


**Rules the database enforces, and the finding behind each:**

| Rule | Finding |
|---|---|
| (`chart_date`, `rank`) unique | 4.3 / 4.6: one song per rank per week |
| (`chart_date`, `song_id`) unique | 4.6 / C9: a song appears at most once per week |
| `rank`, `peak_rank_reported` between 1 and 100 | 4.5 |
| `last_week_rank` between 1 and 100, or empty | 4.4: the `0` code must never return |
| every entry refers to an existing chart, song and name (foreign keys) | no orphan rows |
| each printed (title, performer) belongs to exactly one song | C8–C9: identity is unambiguous |

**Design choices:**
- **Era and flag are stored once per chart**, not repeated in 100 rows per week.
- **Song statistics (first and last chart date, weeks, best rank) are not stored but calculated by a view** (`song_summary`, a saved query). A stored copy could become outdated; a view is always consistent with the entries.
- **`songs` stores the two "first entry" flags** because they depend on pre-2000 data, which is not in the database.
- **Deliberate redundancy:** `song_id` in `chart_entries` could be looked up through `name_id`. We keep it because it lets the database enforce "one song once per week", and it keeps queries simple. A check in 6.5 confirms the two never disagree.
- **SQLite has no date or true/false type:** dates are stored as `YYYY-MM-DD` text (which sorts correctly and works with SQLite's date functions) and flags as 0/1.
- **`song_names` is designed for later matching with MusicBrainz:** it lists every spelling of every song with its normalised keys.

### **6.3 Prepare the four tables**

In [ ]:
charts_tbl = (weekly[["chart_date", "source_era", "flag_inconsistent_chart"]]
              .drop_duplicates().sort_values("chart_date").reset_index(drop=True))
if charts_tbl["chart_date"].duplicated().any():
    raise ValueError("A chart week has two different era/flag values – charts table would have a duplicate key.")

song_names_tbl = (weekly[["song_id", "title", "performer", "title_key", "performer_key"]]
                  .drop_duplicates(["title", "performer"])
                  .sort_values(["song_id", "title", "performer"]).reset_index(drop=True))
song_names_tbl.insert(0, "name_id", range(1, len(song_names_tbl) + 1))

songs_tbl = songs[["song_id", "title", "performer", "charted_before_window", "first_entry_uncertain"]].rename(
    columns={"title": "display_title", "performer": "display_performer"})

# Join contract: many-to-one (many weekly rows share one name), every row must find its name (no orphans).
entries_tbl = (weekly.merge(song_names_tbl[["name_id", "title", "performer"]], on=["title", "performer"],
                            how="left", validate="many_to_one")
               [["chart_date", "rank", "song_id", "name_id", "last_week_rank",
                 "peak_rank_reported", "weeks_on_chart_reported"]])

if len(entries_tbl) != len(weekly):
    raise ValueError(f"Join contract broken: {len(weekly):,} weekly rows became {len(entries_tbl):,}.")
if entries_tbl["name_id"].isna().any():
    raise ValueError("Some chart entries did not find their name_id.")

for name, tbl in {"charts": charts_tbl, "songs": songs_tbl, "song_names": song_names_tbl,
                  "chart_entries": entries_tbl}.items():
    print(f"{name:14s} {len(tbl):>8,} rows  {tbl.shape[1]} columns")

### **6.4 Create the database**

In [ ]:
import sqlite3   # built into Python

DB_FILE = OUTPUT_DIR / "billboard_hot100.sqlite"
if DB_FILE.exists():
    DB_FILE.unlink()          # start from an empty database on every run

SCHEMA = """
PRAGMA foreign_keys = ON;

CREATE TABLE charts (
    chart_date              TEXT    PRIMARY KEY,               -- 'YYYY-MM-DD', a Saturday
    source_era              TEXT    NOT NULL CHECK (source_era IN ('pre-2022 archive', '2022+ UT scrape')),
    flag_inconsistent_chart INTEGER NOT NULL CHECK (flag_inconsistent_chart IN (0, 1))
);

CREATE TABLE songs (
    song_id                 TEXT    PRIMARY KEY,               -- our identifier, e.g. 'S00001'
    display_title           TEXT    NOT NULL,
    display_performer       TEXT    NOT NULL,
    charted_before_window   INTEGER NOT NULL CHECK (charted_before_window IN (0, 1)),
    first_entry_uncertain   INTEGER NOT NULL CHECK (first_entry_uncertain IN (0, 1))
);

CREATE TABLE song_names (
    name_id                 INTEGER PRIMARY KEY,
    song_id                 TEXT    NOT NULL REFERENCES songs(song_id),
    title                   TEXT    NOT NULL,                  -- exactly as printed on the chart
    performer               TEXT    NOT NULL,
    title_key               TEXT    NOT NULL,
    performer_key           TEXT    NOT NULL,
    UNIQUE (title, performer)
);

CREATE TABLE chart_entries (
    chart_date              TEXT    NOT NULL REFERENCES charts(chart_date),
    rank                    INTEGER NOT NULL CHECK (rank BETWEEN 1 AND 100),
    song_id                 TEXT    NOT NULL REFERENCES songs(song_id),
    name_id                 INTEGER NOT NULL REFERENCES song_names(name_id),
    last_week_rank          INTEGER          CHECK (last_week_rank BETWEEN 1 AND 100),
    peak_rank_reported      INTEGER NOT NULL CHECK (peak_rank_reported BETWEEN 1 AND 100),
    weeks_on_chart_reported INTEGER NOT NULL CHECK (weeks_on_chart_reported >= 1),
    PRIMARY KEY (chart_date, rank),
    UNIQUE (chart_date, song_id)
);

CREATE INDEX idx_entries_song ON chart_entries(song_id);

CREATE VIEW song_summary AS
SELECT  s.song_id,
        s.display_title,
        s.display_performer,
        MIN(e.chart_date)  AS first_chart_date,
        MAX(e.chart_date)  AS last_chart_date,
        COUNT(*)           AS weeks_in_window,
        MIN(e.rank)        AS best_rank_in_window,
        s.charted_before_window,
        s.first_entry_uncertain
FROM songs s
JOIN chart_entries e ON e.song_id = s.song_id
GROUP BY s.song_id;
"""

def to_sql_ready(df):
    """Convert pandas types SQLite cannot store directly: dates → 'YYYY-MM-DD' text, True/False → 1/0."""
    out = df.copy()
    for col in out.columns:
        if pd.api.types.is_datetime64_any_dtype(out[col]):
            out[col] = out[col].dt.strftime("%Y-%m-%d")
        elif pd.api.types.is_bool_dtype(out[col]):
            out[col] = out[col].astype(int)
    return out

con = sqlite3.connect(DB_FILE)
con.executescript(SCHEMA)
for name, tbl in [("charts", charts_tbl), ("songs", songs_tbl),
                  ("song_names", song_names_tbl), ("chart_entries", entries_tbl)]:
    to_sql_ready(tbl).to_sql(name, con, if_exists="append", index=False)
con.commit()

pd.read_sql("SELECT name, type FROM sqlite_master WHERE type IN ('table', 'view') ORDER BY type, name", con)

### **6.5 Check the database against the clean data**

In [ ]:
def q(sql, params=()):
    """Run a SQL query on our database and return the result as a table."""
    return pd.read_sql(sql, con, params=params)

db_checks = {
    "charts":        q("SELECT COUNT(*) AS n FROM charts")["n"][0],
    "songs":         q("SELECT COUNT(*) AS n FROM songs")["n"][0],
    "song_names":    q("SELECT COUNT(*) AS n FROM song_names")["n"][0],
    "chart_entries": q("SELECT COUNT(*) AS n FROM chart_entries")["n"][0],
    "foreign-key violations": len(q("PRAGMA foreign_key_check")),
    "entries whose song_id disagrees with their name_id":
        q("""SELECT COUNT(*) AS n FROM chart_entries e JOIN song_names n ON n.name_id = e.name_id
             WHERE n.song_id <> e.song_id""")["n"][0],
}
display(pd.Series(db_checks, name="value").to_frame())

view = q("SELECT * FROM song_summary ORDER BY song_id")
expected = songs.sort_values("song_id").reset_index(drop=True)
view_matches_python = {
    "first_chart_date":    bool((view["first_chart_date"] == expected["first_chart_date"].dt.strftime("%Y-%m-%d")).all()),
    "last_chart_date":     bool((view["last_chart_date"]  == expected["last_chart_date"].dt.strftime("%Y-%m-%d")).all()),
    "weeks_in_window":     bool((view["weeks_in_window"]  == expected["weeks_in_window"]).all()),
    "best_rank_in_window": bool((view["best_rank_in_window"] == expected["best_rank_in_window"]).all()),
}
pd.Series(view_matches_python, name="view equals Section 5 result").to_frame()

### **6.6 Test that the database rejects invalid data**

A rule is only useful if it works. We try to insert examples of each kind of error found in Section 4 (and others that must never happen). Each attempt runs in a transaction that is undone, so the database is unchanged afterwards. Every attempt must be rejected.

In [ ]:
new_chart = "INSERT INTO charts VALUES ('2026-01-03', '2022+ UT scrape', 0)"
violations = {
    "rank outside 1-100":             [new_chart, "INSERT INTO chart_entries VALUES ('2026-01-03', 101, 'S00001', 1, NULL, 1, 1)"],
    "two songs at the same rank":     [new_chart, "INSERT INTO chart_entries VALUES ('2026-01-03', 1, 'S00001', 1, NULL, 1, 1)",
                                                  "INSERT INTO chart_entries VALUES ('2026-01-03', 1, 'S00002', 2, NULL, 1, 1)"],
    "same song twice in one week":    [new_chart, "INSERT INTO chart_entries VALUES ('2026-01-03', 1, 'S00001', 1, NULL, 1, 1)",
                                                  "INSERT INTO chart_entries VALUES ('2026-01-03', 2, 'S00001', 1, NULL, 1, 1)"],
    "last-week rank coded as 0":      [new_chart, "INSERT INTO chart_entries VALUES ('2026-01-03', 1, 'S00001', 1, 0, 1, 1)"],
    "song that does not exist":       [new_chart, "INSERT INTO chart_entries VALUES ('2026-01-03', 1, 'S99999', 1, NULL, 1, 1)"],
    "chart week that does not exist": ["INSERT INTO chart_entries VALUES ('2026-01-10', 1, 'S00001', 1, NULL, 1, 1)"],
    "same printed name added twice":  ["INSERT INTO song_names VALUES (999999, 'S00001', 'Smooth', 'Santana Featuring Rob Thomas', 'x', 'y')"],
}

results = {}
for label, statements in violations.items():
    try:
        with con:                                   # transaction: undone automatically if anything fails
            for sql in statements:
                con.execute(sql)
            raise RuntimeError("accepted")           # never keep test rows, even if they were accepted
    except sqlite3.IntegrityError as err:
        results[label] = f"rejected – {err}"
    except RuntimeError:
        results[label] = "ACCEPTED (problem!)"

display(pd.Series(results, name="database response").to_frame())
q("SELECT (SELECT COUNT(*) FROM charts) AS charts, (SELECT COUNT(*) FROM chart_entries) AS entries")

### **6.7 Comparison: CSV vs Parquet vs SQLite on our data**

We store the same four tables in each format and measure: total size, time to write and to read everything, and whether data types survive a save-and-reload. We also answer one typical analysis question both ways (debuts per year), to compare the effort and to check that both give the same answer. Times are the median of three runs and vary slightly between runs and machines.

In [ ]:
import statistics

TABLES = {"charts": charts_tbl, "songs": songs_tbl, "song_names": song_names_tbl, "chart_entries": entries_tbl}
CMP_DIR = OUTPUT_DIR / "storage_comparison"
CMP_DIR.mkdir(parents=True, exist_ok=True)

def timed(function, repeats=3):
    """Run a function several times; return its result and the median time in seconds."""
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        result = function()
        times.append(time.perf_counter() - start)
    return result, statistics.median(times)

def write_csv():
    for name, tbl in TABLES.items():
        tbl.to_csv(CMP_DIR / f"{name}.csv", index=False, date_format="%Y-%m-%d")

def write_parquet():
    for name, tbl in TABLES.items():
        tbl.to_parquet(CMP_DIR / f"{name}.parquet", index=False)

def write_sqlite():
    path = CMP_DIR / "test.sqlite"
    if path.exists():
        path.unlink()
    with sqlite3.connect(path) as c:
        c.executescript(SCHEMA)
        for name, tbl in TABLES.items():
            to_sql_ready(tbl).to_sql(name, c, if_exists="append", index=False)
    c.close()

def read_csv():
    return {name: pd.read_csv(CMP_DIR / f"{name}.csv") for name in TABLES}

def read_parquet():
    return {name: pd.read_parquet(CMP_DIR / f"{name}.parquet") for name in TABLES}

def read_sqlite():
    with sqlite3.connect(CMP_DIR / "test.sqlite") as c:
        out = {name: pd.read_sql(f"SELECT * FROM {name}", c) for name in TABLES}
    c.close()
    return out

formats = {"CSV": (write_csv, read_csv, ".csv"),
           "Parquet": (write_parquet, read_parquet, ".parquet"),
           "SQLite": (write_sqlite, read_sqlite, ".sqlite")}
comparison, loaded = {}, {}
for fmt, (writer, reader, ext) in formats.items():
    _, write_s = timed(writer)
    loaded[fmt], read_s = timed(reader)
    entries = loaded[fmt]["chart_entries"]
    comparison[fmt] = {
        "files": len(list(CMP_DIR.glob(f"*{ext}"))),
        "size (MB)": round(sum(f.stat().st_size for f in CMP_DIR.glob(f"*{ext}")) / 1_000_000, 1),
        "write time (s)": round(write_s, 2),
        "read time (s)": round(read_s, 2),
        "chart_date type after reading": str(entries["chart_date"].dtype),
        "last_week_rank type after reading": str(entries["last_week_rank"].dtype),
        "flag type after reading": str(loaded[fmt]["songs"]["first_entry_uncertain"].dtype),
    }
pd.DataFrame(comparison)

In [ ]:
def debuts_pandas(tables):
    entries = tables["chart_entries"].copy()
    entries["chart_date"] = pd.to_datetime(entries["chart_date"])
    first = entries.groupby("song_id")["chart_date"].min().rename("first_chart_date").reset_index()
    first = first.merge(tables["songs"][["song_id", "first_entry_uncertain"]], on="song_id",
                        how="left", validate="one_to_one")          # one row per song on both sides
    observed = first[~first["first_entry_uncertain"].astype(bool)]
    return observed.groupby(observed["first_chart_date"].dt.year).size()

DEBUTS_SQL = """
SELECT CAST(substr(first_chart_date, 1, 4) AS INTEGER) AS year, COUNT(*) AS debuts
FROM song_summary
WHERE first_entry_uncertain = 0
GROUP BY year
ORDER BY year
"""

pandas_result, pandas_s = timed(lambda: debuts_pandas(loaded["Parquet"]))
sql_result, sql_s = timed(lambda: q(DEBUTS_SQL))
print(f"pandas on loaded tables: {pandas_s:.3f} s | SQL on the database: {sql_s:.3f} s")
print("Same answer:", bool((pandas_result.values == sql_result["debuts"].values).all()))
sql_result.set_index("year").T

### **6.8 Decision**

**Observations (6.6–6.7):**
- Parquet is the smallest (1.9 MB), fastest, and the only format that keeps all data types on reload.
- SQLite is the largest (16.7 MB, because of the indexes that enforce its rules and dates stored as text) and the slowest, and it returns dates as text and flags as 0/1.
- All formats write and read our full data in a few seconds at most (SQLite, the slowest, took under three seconds in total in every run; times vary slightly between runs and machines), and both query methods give the same answer.
- Only SQLite rejected all seven kinds of invalid data; CSV and Parquet have no mechanism to reject any of them.

**Decision: SQLite.**

**Justification:**
1. **Integrity (R2) is our main risk, not speed or size.** Section 4 found duplicate-identity, placeholder and continuity problems; SQLite makes these impossible to reintroduce (6.6). At ~150,000 rows, the size and speed differences are a few megabytes and at most a couple of seconds, so they do not affect our work (R6).
2. **Our data is relational (R3).** Songs, printed names and weekly entries are linked, and MusicBrainz matches will add another linked table. A relational database states and checks these links; files do not.
3. **Readable, standard queries (R4).** Recurring logic (such as each song's first week) is defined once in a view and reused, instead of being re-coded in every analysis.
4. **Nothing to install, no server, one file (R1, R7).** SQLite is part of Python, so the notebook stays fully reproducible in Colab; the single file is easy to keep private.
5. **Proportionate (R6).** SQLite is the simplest database available; PostgreSQL or MongoDB would add a server without solving any problem we have.

**Accepted costs and how we handle them:**
- *Types:* dates return as text and flags as 0/1 → a helper restores the types on every read (6.9).
- *Size:* 16.7 MB vs 1.9 MB for Parquet → irrelevant at our scale.
- *When we would choose differently:* for much larger data used only for analysis (e.g. millions of rows), Parquet or DuckDB would be better; for data shared live between many users, PostgreSQL. Neither applies to the Billboard data.
- *MusicBrainz:* the full MusicBrainz data (millions of recordings) is far larger; how it is stored is decided in Section 8. Only the matched information needs to join this database.

### **6.9 Reading data back with correct types**

In [ ]:
TYPE_RULES = {
    "dates":    ["chart_date"],
    "integers": ["rank", "name_id", "last_week_rank", "peak_rank_reported", "weeks_on_chart_reported",
                 "n_matched_recordings", "first_release_year", "days_release_to_chart"],
    "flags":    ["flag_inconsistent_chart", "charted_before_window", "first_entry_uncertain"],
}

def read_table(name_or_sql, connection=con):
    """Read a table, view or SELECT query from the database and restore pandas types."""
    sql = name_or_sql if name_or_sql.strip().upper().startswith("SELECT") else f"SELECT * FROM {name_or_sql}"
    df = pd.read_sql(sql, connection)
    for col in df.columns:
        is_full_date_text = df[col].dropna().astype(str).str.fullmatch(r"\d{4}-\d{2}-\d{2}").all()
        if (col in TYPE_RULES["dates"] or col.endswith("_date")) and is_full_date_text:
            df[col] = pd.to_datetime(df[col], format="%Y-%m-%d")
        elif col in TYPE_RULES["integers"]:
            df[col] = df[col].astype("Int64")
        elif col in TYPE_RULES["flags"] or col.startswith("flag_"):
            df[col] = df[col].astype(bool)
    return df

read_table("song_summary").dtypes.to_frame("type")

### **6.10 Save and download the storage documentation**

In [ ]:
SCHEMA_FILE = OUTPUT_DIR / "schema_billboard.sql"
SCHEMA_FILE.write_text(SCHEMA.strip() + "\n", encoding="utf-8")

STORAGE_LOG_FILE = OUTPUT_DIR / "storage_billboard.json"
STORAGE_LOG_FILE.write_text(json.dumps({
    "record_type": "storage record – Billboard Hot 100",
    "source_commit": SOURCE_COMMIT,
    "source_sha256": actual_sha256,
    "built_at_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    "chosen_storage": "SQLite database (single file), relational schema with constraints",
    "database_file": DB_FILE.name,
    "sqlite_version": sqlite3.sqlite_version,
    "tables": {name: int(q(f"SELECT COUNT(*) AS n FROM {name}")["n"][0]) for name in TABLES},
    "views": ["song_summary"],
    "database_checks": {k: int(v) for k, v in db_checks.items()},
    "constraint_tests": results,
    "format_comparison": comparison,
}, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

download_file(SCHEMA_FILE)
download_file(STORAGE_LOG_FILE)

DOWNLOAD_DATABASE = False   # True only for private use within the group (Section 1.8)
if DOWNLOAD_DATABASE:
    download_file(DB_FILE)
print(f"Database: {DB_FILE} ({DB_FILE.stat().st_size / 1_000_000:.1f} MB)")

### **6.11 Summary of Section 6**

**Observations:** the clean Billboard data is stored in a SQLite database with four tables (charts 1,357; songs 11,728; song_names 11,785; chart_entries 135,700) and one view. The database matches the cleaned data exactly, its calculated song figures equal those of Section 5, and it rejected all seven kinds of invalid data tested.

**Interpretation:** SQLite was chosen over CSV and Parquet because enforcing integrity and relationships matters more for our project than size or speed, which are negligible at this scale. Its weakness (no native date or boolean type) is handled by a reading helper.

**Prepared for later:** `song_names` lists every spelling of every song with its normalised keys, ready for matching with MusicBrainz; a matching table can be added to this database, linked by `song_id`.

## **7. Descriptive analysis of the Billboard data**

This section describes the songs that reached the Hot 100 between 2000 and 2025: how many, when, how they entered, how long they stayed, and who performed them.

**What these results can and cannot show.** All songs in the Billboard data reached the chart, so this section cannot identify what *determines* reaching it: that requires a comparison with songs that never charted (Sections 10–11). It describes the population of successful songs and how the chart itself changed over time, which later analyses must take into account.

**Choices made for this section:**
- **Unit:** the song (`song_summary` view, Section 6).
- **Songs analysed:** the 11,580 songs whose first chart entry falls inside our window. The 148 songs with `first_entry_uncertain` (Section 5.11) are excluded, because their true entry date is earlier than what we observe.
- **Positions and longevity** are used only to describe how songs enter and how the chart changed, not as measures of success (consistent with our definition).
- **Eras.** Where years are grouped, we use four eras whose boundaries follow the main changes to the Hot 100 formula (7.2): 2000–04 (before downloads), 2005–11 (downloads), 2012–17 (on-demand streaming and YouTube), 2018–25 (paid streams weighted more).
- **Hot 100 formula changes** are marked with dotted vertical lines in the yearly charts.
- Each result is followed by **observations** and, separately, **interpretations**.

### **7.1 Data and chart settings**

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
DESCRIPTIVE_RESULTS = {}

# Colour scale for heatmaps, built from the notebook palette: white = low, dark green = high.
CMAP_PALETTE = LinearSegmentedColormap.from_list("palette", ["#FFFFFF", COLOR_THIRD, COLOR_SECOND, COLOR_MAIN])

# Changes to the Hot 100 formula discussed in the text (sources in 7.2): label -> year(s).
FORMULA_CHANGES = {
    "downloads": [2005],
    "streaming &\nYouTube": [2012, 2013],
    "paid streams\nweighted more": [2018],
}

# Eras used when years are grouped; boundaries follow the formula changes above.
ERA_BINS   = [1999, 2004, 2011, 2017, 2025]
ERA_LABELS = ["2000–04", "2005–11", "2012–17", "2018–25"]

def style_axes(ax, title, xlabel, ylabel, title_pad=6):
    """Apply the same labels and light styling to every chart."""
    ax.set_title(title, pad=title_pad)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)

def mark_formula_changes(ax, labels=True):
    """Dotted vertical lines at the Hot 100 formula changes, labelled just above the plot area.
    Use title_pad=30 in style_axes when labels are shown."""
    for label, years in FORMULA_CHANGES.items():
        for year in years:
            ax.axvline(year, color="dimgrey", linestyle=":", linewidth=1.2, zorder=0)
        if labels:
            ax.text(np.mean(years), 1.01, label, transform=ax.get_xaxis_transform(),
                    ha="center", va="bottom", fontsize=8, color="dimgrey")

def save_figure(fig, name):
    """Save a chart as a high-resolution PNG for the report and presentation."""
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, dpi=200, bbox_inches="tight")
    return path

songs_desc   = read_table("song_summary")
entries_desc = read_table("chart_entries")

entry_rank = (entries_desc.sort_values(["chart_date", "rank"])
              .drop_duplicates("song_id").set_index("song_id")["rank"])
songs_desc["entry_rank"] = songs_desc["song_id"].map(entry_rank)
songs_desc["entry_year"] = songs_desc["first_chart_date"].dt.year

entered = songs_desc[~songs_desc["first_entry_uncertain"]].copy()
entered["era"] = pd.cut(entered["entry_year"], ERA_BINS, labels=ERA_LABELS)
print(f"Songs in the database               : {len(songs_desc):,}")
print(f"Songs whose first entry we observe  : {len(entered):,}")
print(f"Excluded (first entry before window): {len(songs_desc) - len(entered):,}")
print(f"Songs per era                       : {entered['era'].value_counts(sort=False).to_dict()}")

### **7.2 How many songs reach the chart each year?**

In [ ]:
entries_per_year = entered.groupby("entry_year").size()
DESCRIPTIVE_RESULTS["entries_per_year"] = entries_per_year.to_dict()

fig, ax = plt.subplots(figsize=(9, 4.3))
ax.plot(entries_per_year.index, entries_per_year.values, color=COLOR_MAIN, linewidth=2, marker="o", markersize=4)
ax.set_ylim(bottom=0)
mark_formula_changes(ax)
style_axes(ax, "Songs reaching the Hot 100 for the first time, per year",
           "Year of first chart appearance", "Number of songs", title_pad=30)
save_figure(fig, "fig1_entries_per_year")
plt.show()

early = entries_per_year.loc[2000:2004].mean()
late  = entries_per_year.loc[2021:2025].mean()
print(f"Average per year 2000–2004: {early:.0f} | 2021–2025: {late:.0f} | ratio: {late / early:.1f}x")
entries_per_year.to_frame("songs").T

**Observations:** about 300 songs per year reached the chart for the first time in 2000–2004, against about 610 per year in 2021–2025 (2.0 times as many). The increase is not steady: rises in 2005–2011, a drop in 2012, and a jump in 2018–2020. Most of the 2010–2011 peak, and so much of the 2012 drop, comes from a single act, Glee Cast (see the check in 7.5).

**Interpretation:** the number of slots is fixed (100 per week), so twice as many new songs means songs pass through the chart faster. The timing matches documented changes to the Hot 100 formula: digital downloads were added in 2005, on-demand streaming in 2012, YouTube views in 2013, and paid streams were weighted more heavily from 2018 (sources below; dotted lines in the chart). This timing is consistent with, but does not prove, an effect of these changes.

**Implication:** reaching the chart was not equally hard in every year, so any later comparison across years must take the year into account.

*Sources:* [Music Business Journal (Berklee), "Modernizing the Hot 100"](https://www.thembj.org/?p=4740); [MusicRow, 14 March 2012](https://musicrow.com/category/salesmarketing/page/86/); [heise, on YouTube and Billboard chart data](https://heise.de/-11123509).

### **7.3 When in the year do songs enter?**

In [ ]:
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
chart_weeks = read_table("charts")["chart_date"]
weeks_per_month = chart_weeks.groupby(chart_weeks.dt.month).size()
entries_per_month = entered.groupby(entered["first_chart_date"].dt.month).size()
per_chart_week = entries_per_month / weeks_per_month
overall_rate = len(entered) / len(chart_weeks)        # average over all chart weeks
DESCRIPTIVE_RESULTS["new_songs_per_chart_week_by_month"] = per_chart_week.round(2).to_dict()
DESCRIPTIVE_RESULTS["new_songs_per_chart_week_overall"] = round(overall_rate, 2)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(per_chart_week.index, per_chart_week.values, color=COLOR_SECOND)
ax.axhline(overall_rate, color=COLOR_MAIN, linestyle="--", linewidth=1.5)
ax.text(12.6, overall_rate + 0.15, f"average of all\nchart weeks: {overall_rate:.1f}", ha="right", va="bottom",
        fontsize=8, color=COLOR_MAIN)
ax.set_ylim(0, per_chart_week.max() * 1.18)
ax.set_xticks(range(1, 13), MONTHS)
style_axes(ax, "New songs per chart week, by month (2000–2025)",
           "Month of first chart appearance", "New songs per chart week (average)")
save_figure(fig, "fig2_entries_per_month")
plt.show()

pd.DataFrame({"chart weeks": weeks_per_month, "new songs": entries_per_month,
              "per chart week": per_chart_week.round(1)}).T

**Observations:** new songs enter at a similar rate throughout the year (about 8–9 per chart week, close to the average of all chart weeks, 8.5, dashed line), except January, which is clearly lower (6.8 per week).

**Interpretation:** a plausible explanation is that fewer songs are released around the holidays and that holiday songs occupy chart places in late December and early January; our data cannot confirm this. Because this average pools all 26 years, we check below whether the pattern holds in every period.

The monthly averages above pool 26 years in which the number of new songs doubled (7.2), so a busy recent year weighs as much as a quiet early one. To check whether the January dip holds in every period, and to see where the busiest months are, we split the same rate (new songs per chart week) by year **and** month.

In [ ]:
year_of_week  = chart_weeks.dt.year.rename("year")
month_of_week = chart_weeks.dt.month.rename("month")
weeks_ym   = chart_weeks.groupby([year_of_week, month_of_week]).size()
entries_ym = entered.groupby([entered["first_chart_date"].dt.year.rename("year"),
                              entered["first_chart_date"].dt.month.rename("month")]).size()
rate_ym = (entries_ym.reindex(weeks_ym.index, fill_value=0) / weeks_ym).unstack("month")

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(rate_ym.values, cmap=CMAP_PALETTE, aspect="auto", vmin=0)
ax.set_xticks(range(12), MONTHS)
ax.set_yticks(range(len(rate_ym)), rate_ym.index)
threshold = rate_ym.values.max() * 0.55
for r in range(rate_ym.shape[0]):
    for c in range(rate_ym.shape[1]):
        v = rate_ym.iat[r, c]
        ax.text(c, r, f"{v:.0f}", ha="center", va="center", fontsize=7,
                color="white" if v > threshold else "black")
ax.set_title("New songs per chart week, by year and month")
ax.set_xlabel("Month of first chart appearance")
ax.set_ylabel("Year of first chart appearance")
fig.colorbar(im, ax=ax, label="New songs per chart week (average)", shrink=0.8)
save_figure(fig, "fig2b_entries_year_month_heatmap")
plt.show()

# How the January dip and the busiest months are distributed
jan_below_year_avg = (rate_ym[1] < (entries_ym.groupby(level="year").sum() / weeks_ym.groupby(level="year").sum()))
jan_rank = rate_ym.rank(axis=1)[1]                          # 1 = lowest month of that year
by_era_month = rate_ym.groupby(pd.cut(rate_ym.index, ERA_BINS, labels=ERA_LABELS), observed=True).mean()
top_cells = rate_ym.stack().nlargest(8)
print(f"Years in which January is below that year's average : {int(jan_below_year_avg.sum())} of {len(rate_ym)}")
print(f"Years in which January is the lowest month          : {int((jan_rank == 1).sum())} of {len(rate_ym)}")
print("\nHighest year-months (new songs per chart week):")
print(top_cells.round(1).to_string())
print("\nAverage per era (new songs per chart week):")
display(by_era_month.round(1).set_axis(MONTHS, axis=1))

DESCRIPTIVE_RESULTS["new_songs_per_chart_week_by_year_month"] = {
    str(y): {MONTHS[m - 1]: round(float(v), 2) for m, v in row.items()} for y, row in rate_ym.iterrows()}
DESCRIPTIVE_RESULTS["january_below_year_average_years"] = int(jan_below_year_avg.sum())

**Observations:**
- January is below its year's average in 21 of 26 years and is the lowest month in 15. It is the lowest month in every era (4.8, 5.0 and 6.3 new songs per chart week in 2000–04, 2005–11 and 2012–17; 10.0 in 2018–25, tied with February).
- The colours darken from top to bottom in every month: the rise in new songs (7.2) happens throughout the year, not in one season. From 2018 on, 85 of 96 months had more than 8 new songs per chart week; before 2005 none did (highest: 7.8).
- All of the eight highest year-months are from 2020 onwards (17–24 new songs per chart week, against about 6 in the early 2000s). Five of them contain one of the busiest weeks of 7.6, when one artist's album entered the chart at once: November 2021 (Taylor Swift), May 2022 (Bad Bunny), April 2024 (Future and Metro Boomin), March 2025 (Playboi Carti) and December 2020 (Taylor Swift).
- Before 2018 the months are much more even; the darkest early cells (13–14 in 2009–2010) fall in Glee Cast's main years (7.5).

**Interpretation:** the January dip is a stable seasonal pattern, not an artefact of pooling, and fits the explanation above (fewer releases around the holidays, holiday songs taking chart places); our data still cannot confirm the cause. In the streaming era the month-to-month pattern is dominated by individual album releases rather than by season: one release can double the number of new songs in a month.

**Implication:** the month in which charting songs enter the chart matters little apart from January; the year (7.2) matters much more. This describes when charting songs arrive, not whether a song charts: whether the month of release is associated with reaching the chart is a separate question, tested among singles in 11.7. Because recent entries come in bursts of songs from one release, later analyses that count songs should keep in mind that many of them share the same artist and release date and are not independent.

### **7.4 How do new songs enter, and how long do they stay?**

In [ ]:
entered["entered_top40"]      = entered["entry_rank"] <= 40
entered["one_week_only"]      = entered["weeks_in_window"] == 1
entered["best_rank_at_entry"] = entered["entry_rank"] == entered["best_rank_in_window"]   # never climbed higher

by_year = entered.groupby("entry_year").agg(
    songs=("song_id", "size"),
    median_entry_rank=("entry_rank", "median"),
    share_entered_top40=("entered_top40", "mean"),
    share_best_rank_at_entry=("best_rank_at_entry", "mean"),
    share_one_week_only=("one_week_only", "mean"),
    median_weeks_on_chart=("weeks_in_window", "median"),
)
DESCRIPTIVE_RESULTS["entry_and_longevity_by_year"] = by_year.round(3).to_dict(orient="index")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.6))
series = [("share_best_rank_at_entry", "best rank reached in entry week", COLOR_MAIN,   "o"),
          ("share_one_week_only",      "stayed only one week",            COLOR_SECOND, "s"),
          ("share_entered_top40",      "entered in the top 40",           COLOR_THIRD,  "^")]
for col, label, color, marker in series:
    ax1.plot(by_year.index, by_year[col] * 100, color=color, linewidth=2, marker=marker, markersize=5,
             label=label)
ax1.set_ylim(0, 100)
ax1.legend(loc="upper left", frameon=True, facecolor="white", edgecolor="none", framealpha=1)
mark_formula_changes(ax1)
style_axes(ax1, "How new songs enter the chart", "Year of first chart appearance",
           "% of songs entering that year", title_pad=30)

ax2.plot(by_year.index, by_year["median_weeks_on_chart"], color=COLOR_MAIN, linewidth=2, marker="o", markersize=4)
ax2.axhline(20, color="dimgrey", linestyle="--", linewidth=1)
ax2.text(2025.4, 19.5, "20 weeks: recurrent rule", ha="right", va="top", fontsize=8, color="dimgrey",
         backgroundcolor="white")
ax2.set_ylim(0, 25)
mark_formula_changes(ax2)
style_axes(ax2, "How long they stay", "Year of first chart appearance",
           "Median weeks on chart", title_pad=30)
fig.text(0.01, -0.03, "2025: 102 songs entered from October onwards and could only be observed for a few weeks "
         "(window ends 2025-12-27).", fontsize=8, color="dimgrey")
save_figure(fig, "fig3_entry_and_longevity")
plt.show()

late_entries  = int((entered["first_chart_date"] >= "2025-10-01").sum())
early_era     = entered[entered["era"] == "2000–04"]
recent_era    = entered[entered["era"] == "2018–25"]
one_week_rec  = recent_era[recent_era["one_week_only"]]
print(f"Songs entering from October 2025 (few weeks left to observe): {late_entries}")
print(f"2000–04: songs that stayed exactly 20 weeks                 : {(early_era['weeks_in_window'] == 20).mean():.0%}")
print(f"2018–25: one-week songs that entered below the top 40      : {(~one_week_rec['entered_top40']).mean():.0%}"
      f" ({len(one_week_rec):,} songs)")
print(f"2018–25: top-40 entries that reached their best rank at entry: "
      f"{recent_era.loc[recent_era['entered_top40'], 'best_rank_at_entry'].mean():.0%}")
by_year.round(2)


**Observations:**
- In 2000–2004, almost no new song entered in the top 40 (0–3%) or stayed only one week (≈1%), and only 6–11% reached their best rank in their entry week: nearly every song climbed after entering. The median song stayed 19–20 weeks, and 31% of the songs entering in 2000–04 stayed *exactly* 20 weeks.
- From 2005 all three shares rise. Since 2020, about three quarters of new songs (74–76%) reached their best rank in their entry week, a fifth to a third (21–34%) entered in the top 40, 36–46% stayed only one week, and the median song stayed 2–3 weeks.
- The rise is not steady: it happens mainly in 2005–2011 and 2017–2020. The 2010–2011 peak in the one-week share and the dip in median weeks (6–7) are mostly Glee Cast (7.5).
- The songs that stay one week are not the ones entering high: in 2018–25, 94% of the 1,869 one-week songs entered below the top 40. Of the songs that did enter in the top 40, 86% reached their best rank in that first week.
- 102 songs entered from October 2025, so 2025's one-week share is partly an effect of the window ending.

**Interpretation:** the chart changed from **slow climbs** (enter low, rise for weeks, stay for months) to **songs entering at their best rank**: today a typical song has its best week first and then declines. Our first reading of this figure ("many songs enter high and leave quickly") was not accurate; 7.7 shows why. Two groups are behind the averages: songs that enter high and still stay for months (7.7), and a large number of songs that enter low and are gone after a week, often album tracks charting in their release week (7.6). The changes start in 2005, when digital downloads were added to the Hot 100, and continue with streaming (sources in 7.2): these let a song's popularity register in its first week. This timing is consistent with, but does not prove, such an effect.

The 19–20-week median of the early 2000s reflects Billboard's **recurrent rule**, in use in various forms since 1991: a song that has spent 20 weeks on the chart and falls below No. 50 is removed ([Wikipedia, "Billboard Hot 100", section on recurrent rules](https://en.wikipedia.org/wiki/Billboard_Hot_100)). The pile-up at exactly 20 weeks shows that many early-2000s songs would have stayed longer without it, so the longevity figures of that period are partly set by a chart rule rather than by listeners.

**Implication:** "reaching the chart" means something different in 2002 and in 2022: in recent years a large share of charting songs appear once, at a low position. Section 11.6 therefore checks whether the main results also hold under a stricter definition (at least two weeks on the chart), as a robustness check rather than a change of our definition.


### **7.5 Who reaches the chart?**

To describe the "artist popularity" factor, we use the **lead credit**: the performer text before "Featuring" (normalised as in Section 5). For example, "Taylor Swift Featuring Ice Spice" has the lead credit "taylor swift".

**Why this definition:** "Featuring" always separates the main artist from guests, whereas "&", "x", "With" and commas also appear inside band names (Section 5.15). **Limitation:** joint credits such as "Future & Metro Boomin" count as their own lead credit, separate from each artist. MusicBrainz, which stores each credited artist separately, is used to identify individual lead artists for the comparison group (Section 10).

A song's lead credit **had charted before** if it appears on the chart before 2000 (anywhere in the raw archive, back to 1958) or with an earlier song in our window.

In [ ]:
def lead_credit(performer_text):
    """Normalised performer text before 'featuring' (the main credited artist or group)."""
    keys = make_key(performer_text)
    return keys.str.split(" featuring ", n=1).str[0].str.strip()

entered["lead_credit"] = lead_credit(entered["display_performer"])
pre_window_leads = set(lead_credit(raw_hot100.loc[raw_hot100["chart_week"] < STUDY_START, "performer"]))
first_entry_of_lead = entered.groupby("lead_credit")["first_chart_date"].transform("min")
entered["lead_charted_before"] = (entered["lead_credit"].isin(pre_window_leads)
                                  | (entered["first_chart_date"] > first_entry_of_lead))

returning_share = entered.groupby("entry_year")["lead_charted_before"].mean()
DESCRIPTIVE_RESULTS["share_from_lead_credits_that_charted_before"] = returning_share.round(3).to_dict()

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(returning_share.index, returning_share.values * 100, color=COLOR_MAIN, linewidth=2, marker="o", markersize=4)
ax.set_ylim(0, 100)
style_axes(ax, "Share of new chart songs whose lead artist had charted before",
           "Year of first chart appearance", "% of songs entering that year")
save_figure(fig, "fig4_returning_artists")
plt.show()

print(f"Overall: {entered['lead_charted_before'].mean():.1%} of entering songs had a lead credit that charted before")
returning_share.round(2).to_frame("share").T

**Observations:** about three quarters (74.5%) of songs entering the chart had a lead credit that had already charted; the share is stable over time (roughly 67–84% each year); the 2010–2011 high (84%) is partly due to Glee Cast (see the check below).

**Interpretation:** among successful songs, most come from artists with a chart history. This does **not** show that chart history *causes* success: artists with past hits also release more songs, and we do not know how many songs by new versus established artists were released but never charted. Our comparison group cannot settle this either, because all of its artists have charted (Section 10); Section 9.7 examines instead whether established artists' songs enter the chart faster.

The share of returning artists hardly changed, but *how* their songs enter may have. We compare, per era, songs whose lead credit had charted before with songs from a lead credit's first chart week, using the measures of 7.4.

In [ ]:
status = entered["lead_charted_before"].map({True: "had charted before", False: "first chart entry"}).rename("artist")
by_era_status = entered.groupby(["era", status], observed=True).agg(
    songs=("song_id", "size"),
    pct_top40=("entered_top40", "mean"),
    median_weeks=("weeks_in_window", "median"),
    pct_one_week=("one_week_only", "mean"),
)
by_era_status[["pct_top40", "pct_one_week"]] *= 100
wide = by_era_status.unstack("artist")

metrics = [("pct_top40",    "Entered in the top 40 (%)", "{:.0f}"),
           ("median_weeks", "Median weeks on chart",     "{:.0f}"),
           ("pct_one_week", "Stayed only one week (%)",  "{:.0f}")]
groups = [("had charted before", COLOR_MAIN), ("first chart entry", COLOR_SECOND)]
x, width = np.arange(len(ERA_LABELS)), 0.38

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
for ax, (col, title, fmt) in zip(axes, metrics):
    for k, (label, color) in enumerate(groups):
        values = wide[(col, label)].reindex(ERA_LABELS)
        bars = ax.bar(x + (k - 0.5) * width, values, width, color=color, label=label)
        ax.bar_label(bars, [fmt.format(v) for v in values], fontsize=8, padding=2)
    ax.set_xticks(x, ERA_LABELS)
    ax.set_ylim(0, wide[col].max().max() * 1.2)
    style_axes(ax, title, "Era of first chart appearance", "")
axes[0].legend(frameon=False, loc="upper left", title="lead credit", title_fontsize=9)
fig.suptitle("New songs by artists with and without a chart history, per era", y=1.02)
save_figure(fig, "fig4b_new_vs_returning_by_era")
plt.show()

DESCRIPTIVE_RESULTS["entry_by_era_and_artist_history"] = {
    f"{era} | {artist}": {k: round(float(v), 2) for k, v in row.items()}
    for (era, artist), row in by_era_status.iterrows()}
by_era_status.round(1)

**Observations:**
- In 2000–04 the two groups behaved the same: 2% of songs entered in the top 40, the median song stayed 18–20 weeks and about 1% stayed one week, whether or not the lead credit had charted before.
- From 2005 on, songs from lead credits with a chart history entered in the top 40 more often than first chart entries (13% vs 8% in 2005–11, 14% vs 8% in 2012–17, 30% vs 18% in 2018–25).
- Until 2017 both groups stayed about as long (median 11–13 weeks; 19–23% stayed one week).
- In 2018–25 the pattern splits: songs from returning lead credits entered higher but stayed *shorter* (median 2 weeks vs 5; 42% vs 29% stayed one week).
- Glee Cast's songs are almost all in the "had charted before" group of 2005–11 (about 175 of that group's 2,224 songs; only its first two songs count as a first chart entry); see the check below.

**Interpretation:** in the streaming era an established artist's existing audience can stream a whole new release at once, so many of its tracks chart in release week, some of them high, and most drop out the following week: the album bursts of 7.6. A first chart entry is more often a single that has to build an audience, enters lower and stays longer. Before downloads, chart history made no visible difference to *how* a song entered. These are descriptions of charting songs only and do not show that chart history causes success (see the interpretation above).

**Limitation (lookback):** "had charted before" uses the whole archive back to 1958, so a song entering in 2000 is checked against 42 years of charts and a song entering in 2025 against 67. Later songs therefore have slightly more chance of counting as "returning". A fixed lookback (e.g. "charted in the previous 10 years") would make the eras strictly comparable; we did not apply it here.

**Implication:** artist history matters for *how* songs reach the chart, and increasingly so since streaming. Section 9.7 examines whether established artists' songs also reach the chart sooner after release, and Section 11.3 whether album tracks and singles differ.


In [ ]:
songs_per_lead = entered.groupby("lead_credit").size().sort_values(ascending=False)
top_1pct = int(np.ceil(len(songs_per_lead) * 0.01))
concentration = {
    "lead credits with at least one entering song": len(songs_per_lead),
    "lead credits with exactly one song": int((songs_per_lead == 1).sum()),
    "share of lead credits with exactly one song": round(float((songs_per_lead == 1).mean()), 3),
    "median songs per lead credit": float(songs_per_lead.median()),
    f"share of all songs by the top 1% of lead credits ({top_1pct})":
        round(float(songs_per_lead.head(top_1pct).sum() / songs_per_lead.sum()), 3),
}
DESCRIPTIVE_RESULTS["artist_concentration"] = concentration
DESCRIPTIVE_RESULTS["top_lead_credits"] = songs_per_lead.head(10).to_dict()
display(pd.Series(concentration).to_frame("value"))
songs_per_lead.head(10).to_frame("songs")

**Observations:** most lead credits (64%) reached the chart with a single song, while 31 credits (1%) account for 22% of all entering songs. Glee Cast, a TV-show cast that recorded cover versions, is third with 205 songs.

**Interpretation:** chart entries are highly concentrated among a few very prolific artists. Such outliers (and special cases like Glee Cast) could dominate any analysis of artist effects, so later analyses should check whether results hold without them.

#### Glee Cast: an outlier

Glee Cast, third in the table above, is not an artist in the usual sense but the cast of a TV series who released cover versions of songs performed in the show. Its songs reached the chart in bursts while the show was on air, and in 7.2–7.5 the years 2010–2011 stand out (a peak in new songs, a jump in one-week songs and in returning artists). We check how much of this is Glee Cast by recomputing those yearly figures without it.

In [ ]:
is_glee = entered["lead_credit"] == "glee cast"
without_glee = entered[~is_glee]

def yearly_figures(df):
    """The yearly figures of 7.2, 7.4 and 7.5 for a set of songs."""
    g = df.groupby("entry_year")
    return pd.DataFrame({"songs":          g.size(),
                         "one_week":       g["one_week_only"].mean() * 100,
                         "median_weeks":   g["weeks_in_window"].median(),
                         "best_at_entry":  g["best_rank_at_entry"].mean() * 100,
                         "top40":          g["entered_top40"].mean() * 100,
                         "charted_before": g["lead_charted_before"].mean() * 100})

all_y, ex_y = yearly_figures(entered), yearly_figures(without_glee)
glee_table = pd.concat({"all songs": all_y, "without Glee Cast": ex_y}, axis=1).swaplevel(axis=1)
glee_table = glee_table[[(col, grp) for col in all_y.columns for grp in ["all songs", "without Glee Cast"]]]
glee_table.insert(0, ("Glee Cast", "songs"), is_glee.groupby(entered["entry_year"]).sum())

print(f"Glee Cast songs in total: {int(is_glee.sum())}, entering {entered.loc[is_glee, 'entry_year'].min()}–"
      f"{entered.loc[is_glee, 'entry_year'].max()}; stayed only one week: "
      f"{entered.loc[is_glee, 'one_week_only'].mean():.0%}; entered in the top 40: "
      f"{entered.loc[is_glee, 'entered_top40'].mean():.0%}")
display(glee_table.loc[2008:2014].round(1))

panels = [("songs",          "New songs per year (7.2)",               "Number of songs", None),
          ("one_week",       "Stayed only one week (7.4)",             "% of songs entering that year", 100),
          ("median_weeks",   "Median weeks on chart (7.4)",            "Weeks", 25),
          ("best_at_entry",  "Best rank reached in entry week (7.4)",  "% of songs entering that year", 100),
          ("top40",          "Entered in the top 40 (7.4)",            "% of songs entering that year", 100),
          ("charted_before", "Lead artist had charted before (7.5)",   "% of songs entering that year", 100)]
fig, axes = plt.subplots(2, 3, figsize=(15, 8.5))
for ax, (col, title, ylabel, ymax) in zip(axes.flat, panels):
    ax.axvspan(2009, 2012, color=COLOR_THIRD, alpha=0.4, linewidth=0)
    ax.plot(all_y.index, all_y[col], color=COLOR_MAIN, linewidth=2, marker="o", markersize=3, label="all songs")
    ax.plot(ex_y.index, ex_y[col], color=COLOR_SECOND, linewidth=2, linestyle="--", label="without Glee Cast")
    ax.set_ylim(0, ymax)
    style_axes(ax, title, "Year of first chart appearance", ylabel)
axes.flat[0].legend(frameon=False, loc="upper left")
fig.suptitle("Yearly figures of 7.2–7.5 with and without Glee Cast (shaded: 2009–2012, Glee Cast's main years)",
             y=1.0)
fig.tight_layout()
save_figure(fig, "fig5_glee_sensitivity")
plt.show()

DESCRIPTIVE_RESULTS["glee_cast_check"] = {
    str(y): {f"{a} | {b}": round(float(v), 1) for (a, b), v in row.items()}
    for y, row in glee_table.loc[2008:2014].iterrows()}

**Observations:**
- Glee Cast had 205 entering songs, all in 2009–2013 (79 in 2010, 73 in 2011, about one in six of each year's new songs). 83% stayed only one week; 23% entered in the top 40.
- Without Glee Cast there is no 2010–2011 peak in new songs (395 and 410 instead of 474 and 483); the numbers stay flat at about 400 from 2008 to 2011. The 2012 drop shrinks from 120 songs (483 → 363) to 73 (410 → 337).
- For 2010–2011, without Glee Cast: one-week share 24–25% instead of 33–34%; median weeks 12–13 instead of 6–7; best rank in entry week 42–43% instead of 50–52%; lead credit charted before 80–81% instead of 84%. The share entering in the top 40 barely changes (15% instead of 16–19%).
- Outside 2009–2012 the lines with and without Glee Cast are identical.

**Interpretation:** the 2010–2011 spikes in 7.2, 7.4 and 7.5 are mostly one TV show, whose cast released covers of the songs in each episode that charted for a week and dropped out. They do not reflect a general change in the chart. Without Glee Cast the trends are smoother: the one-week share rises steadily from 2005 and the median stay declines gradually. The remaining 2012 drop in new songs (410 → 337) coincides with the addition of on-demand streaming (7.2), still a link based on timing only.

**Implication:** we keep Glee Cast in the data, because its songs genuinely charted. Statements about 2009–2012 should be read with this outlier in mind, and later analyses of artists should check whether their results hold without it: Section 11.6 shows that it does not affect the comparison with non-charting songs.


### **7.6 Weeks with many new songs**

In [ ]:
entries_per_week = entered.groupby("first_chart_date").size()
busiest = entries_per_week.nlargest(10, keep="all").sort_index()   # 10 busiest weeks (+ ties), then in date order

def top_credit(week):
    """Most frequent lead credit among a week's new songs (ties broken alphabetically)."""
    counts = entered.loc[entered["first_chart_date"] == week, "lead_credit"].value_counts()
    counts = counts.sort_index().sort_values(ascending=False, kind="stable")
    return counts.index[0], int(counts.iloc[0])

busiest_table = pd.DataFrame(
    [top_credit(d) for d in busiest.index],
    index=busiest.index.strftime("%Y-%m-%d"),
    columns=["most frequent lead credit", "songs by that credit"])
busiest_table.insert(0, "new songs", busiest.values)
busiest_table.index.name = "chart date"

DESCRIPTIVE_RESULTS["new_songs_per_week"] = {"median": float(entries_per_week.median()), "max": int(entries_per_week.max())}
DESCRIPTIVE_RESULTS["busiest_weeks"] = busiest_table.reset_index().to_dict(orient="records")
print(f"New songs per chart week: median {entries_per_week.median():.0f}, max {entries_per_week.max()}")
print(f"Weeks shown: {len(busiest_table)} (the 10 busiest, plus any week tied with the 10th)")
busiest_table

**Observations:** a typical week brings 7 new songs; the 11 busiest weeks (the 10 busiest plus one tied week) bring 28–34, and all of them are from December 2020 onwards. In 6 of these 11 weeks, a single lead credit accounts for more than half of the new songs (e.g. Taylor Swift with 31 of 33 on 2024-05-04, Morgan Wallen with 29 of 31 on 2025-05-31); in the others the top credit still has 11–15 songs.

**Interpretation:** in recent years, a large part of chart entries comes in bursts when a popular artist releases an album and many of its tracks chart at once, often for a single week. This links 7.4 (one-week songs) with 7.5 (concentration), and suggests that album releases by established artists are a major route onto the chart. That none of the busiest weeks predates 2020 is consistent with the streaming-era changes described in 7.2.

**Implication:** later analyses may need to distinguish album tracks from singles (possible with MusicBrainz release information).

### **7.7 Does a song's entry position predict how long it stays?**

7.4 shows two shares that rose together over time (songs entering in the top 40, songs staying one week), but it does not show whether they are the *same* songs: does entering high mean leaving fast, or do high and low entries behave differently? We therefore cross the song's entry position with how long it stays, separately for each era. This also tests the 7.4 interpretation that the chart moved from slow climbs to fast turnover: if so, the change should be visible within each entry position, not just in the overall average.

**Measure:** median weeks on the chart within our window, for songs entering at positions 1–10, 11–40, 41–70 and 71–100. The 40 limit matches 7.4; 70 splits the lower 60 positions roughly in half. Because Glee Cast's many short-lived entries in 2009–2011 can distort single cells (7.5), we also print how many of them fall in the one cell where they are concentrated, and its median without them.

In [ ]:
BAND_LABELS = ["1–10", "11–40", "41–70", "71–100"]
entered["entry_band"] = pd.cut(entered["entry_rank"].astype(float), [0, 10, 40, 70, 100], labels=BAND_LABELS)

median_weeks = entered.pivot_table(index="entry_band", columns="era", values="weeks_in_window",
                                   aggfunc="median", observed=False)
n_songs      = entered.pivot_table(index="entry_band", columns="era", values="song_id",
                                   aggfunc="size", observed=False)
pct_one_week = entered.pivot_table(index="entry_band", columns="era", values="one_week_only",
                                   aggfunc="mean", observed=False) * 100

fig, ax = plt.subplots(figsize=(8, 4.2))
im = ax.imshow(median_weeks.values, cmap=CMAP_PALETTE, aspect="auto", vmin=0)
ax.set_xticks(range(len(ERA_LABELS)), ERA_LABELS)
ax.set_yticks(range(len(BAND_LABELS)), [f"#{b}" for b in BAND_LABELS])
threshold = median_weeks.values.max() * 0.55
for r in range(median_weeks.shape[0]):
    for c in range(median_weeks.shape[1]):
        v = median_weeks.iat[r, c]
        ax.text(c, r, f"{v:.0f} week{'' if v == 1 else 's'}\n(n={n_songs.iat[r, c]:,})", ha="center", va="center", fontsize=8,
                color="white" if v > threshold else "black")
ax.set_title("Median weeks on chart, by entry position and era")
ax.set_xlabel("Era of first chart appearance")
ax.set_ylabel("Entry position")
fig.colorbar(im, ax=ax, label="Median weeks on chart")
save_figure(fig, "fig6_entry_position_longevity_by_era")
plt.show()

print("% of songs that stayed only one week, by entry position and era:")
display(pct_one_week.round(0))
print("Share of each era's new songs by entry position (%):")
display((n_songs / n_songs.sum() * 100).round(0))

# Glee Cast (7.5) among the 2005–11 entries at positions 41–70: how much of this group's short stay does it explain?
group_0511 = entered[(entered["era"].astype(str) == "2005–11") & (entered["entry_band"] == "41–70")]
glee_0511 = group_0511["lead_credit"] == "glee cast"                       # same rule as in 7.5
print(f"2005–11, entries at 41–70: {len(group_0511):,} songs, of which Glee Cast {int(glee_0511.sum())} "
      f"(entering {group_0511.loc[glee_0511, 'entry_year'].min()}–{group_0511.loc[glee_0511, 'entry_year'].max()}; "
      f"{int(group_0511.loc[glee_0511, 'one_week_only'].sum())} stayed only one week). Median weeks: "
      f"{group_0511['weeks_in_window'].median():.0f} with Glee Cast, {group_0511.loc[~glee_0511, 'weeks_in_window'].median():.0f} without.")

DESCRIPTIVE_RESULTS["entry_position_longevity_by_era"] = {
    "median_weeks": {str(b): {str(e): float(v) for e, v in row.items()} for b, row in median_weeks.iterrows()},
    "songs":        {str(b): {str(e): int(v) for e, v in row.items()} for b, row in n_songs.iterrows()},
    "pct_one_week": {str(b): {str(e): round(float(v), 1) for e, v in row.items()} for b, row in pct_one_week.iterrows()},
    "glee_cast_2005_11_entry_41_70": {"songs": len(group_0511), "glee_cast_songs": int(glee_0511.sum()),
                                      "glee_cast_one_week": int(group_0511.loc[glee_0511, "one_week_only"].sum()),
                                      "median_weeks_with_glee": float(group_0511["weeks_in_window"].median()),
                                      "median_weeks_without_glee": float(group_0511.loc[~glee_0511, "weeks_in_window"].median())},
}

**Observations:**
- **Top-10 entries stay long in every era:** median 19–23 weeks since 2005, with almost none staying only one week (0–2%). (2000–04 had only 6 top-10 entries, too few to interpret.)
- **Entries below the top 10 changed completely.** In 2000–04 they stayed 18–20 weeks whatever their entry position (the recurrent rule of 7.4). In 2018–25 the median is 4 weeks for entries at 11–40, 2 weeks at 41–70 and 1 week at 71–100; half of the songs entering at 41–100 (48–50%) stayed only one week, against 12% at 11–40.
- **Where songs enter shifted upwards:** the share of new songs entering in the top 40 rose from 1% (2000–04) to 27% (2018–25), and the share entering at 71–100 fell from 72% to 44%.
- In 2005–11, entries at 41–70 stayed shorter (median 5 weeks) than entries at 71–100 (14 weeks). Glee Cast contributes 68 of these 484 songs (all in 2009–2011), almost all of which stayed only one week (7.5). Without Glee Cast, the median for this group is 9 weeks, so the shorter stay comes mostly from Glee Cast.

**Interpretation:** before downloads, entry position said almost nothing about how long a song would stay: songs entered low and climbed. In the streaming era it says a great deal: there are now two routes. A song that enters in the top 10 is usually a hit that lasts about as long as hits did twenty years ago; a song that enters lower is usually gone within one to four weeks. The faster turnover in 7.4 therefore comes almost entirely from songs entering below the top 10, many of them album tracks charting in their release week (7.6, and returning artists in 7.5). This corrects our first reading of 7.4, that high-entering songs leave quickly.

**Implication:** under our yes/no definition of success, a growing share of recent "successes" are songs that appeared once or twice at a low position. When Section 11 compares charting and non-charting songs, part of the difference may come from this group (album tracks by established artists, see 11.3), so Section 11.6 repeats the comparison with a stricter threshold (at least two weeks on the chart) to show whether the conclusions depend on it.

*Limitation:* songs entering in late 2025 are cut off by the end of the window, which slightly lowers the 2018–25 medians (102 of 4,845 songs).

### **7.8 Save the descriptive results and figures**

In [ ]:
import shutil

DESC_FILE = OUTPUT_DIR / "descriptive_results_billboard.json"
DESC_FILE.write_text(json.dumps({
    "record_type": "descriptive results – Billboard Hot 100 (aggregates only)",
    "source_commit": SOURCE_COMMIT,
    "computed_at_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    "songs_analysed": len(entered),
    "results": DESCRIPTIVE_RESULTS,
}, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

FIG_ZIP = shutil.make_archive(str(OUTPUT_DIR / "figures_billboard"), "zip", FIG_DIR)
download_file(DESC_FILE)
download_file(FIG_ZIP)

### **7.9 Summary of Section 7**

**Observations:**
1. About twice as many songs reached the chart per year in 2021–2025 (≈610) as in 2000–2004 (≈300). The increase happens in every month of the year.
2. New songs enter at a similar rate all year, except January, which is lower in 21 of 26 years. In recent years the busiest months are those with a major album release.
3. The chart moved from slow climbs to songs entering at their best rank: since 2020 about three quarters of new songs never climbed after entering (6–11% in 2000–04), 36–46% stay only one week, and the median song stays 2–3 weeks (vs ≈19–20 weeks in the early 2000s, a figure set partly by Billboard's 20-week recurrent rule).
4. Entry position now predicts how long a song stays: top-10 entries still last a median of about 19 weeks, while songs entering at 41–100 in 2018–25 last 1–2 weeks and half of them only one week.
5. About three quarters of entering songs come from lead credits that had charted before; 1% of lead credits account for 22% of entries. Since 2018, songs from returning lead credits enter higher but stay shorter than first chart entries (median 2 vs 5 weeks).
6. All of the busiest weeks for new entries are from December 2020 onwards, and in about half of them a single artist's release accounts for most new songs.
7. One outlier, Glee Cast (205 songs in 2009–2013, 83% for one week), causes the 2010–2011 spikes in new songs, one-week songs and returning artists.

**Interpretation:** reaching the Hot 100 became more common and more short-lived over the period, but not for all songs equally: hits that enter high still last, while a large and growing group of songs, often album tracks by established artists, appear for a week or two at low positions. The timing of these changes matches documented changes to the Hot 100 formula (downloads from 2005, streaming from 2012–2013, paid streams weighted more from 2018; see 7.2).

**Limitations:**
- Only successful songs are observed, so none of these patterns identify what determines reaching the chart (selection; requires the comparison group of Sections 10–11).
- The link with Billboard's methodology changes is based on timing only; its effect is not tested here. Chart rules (such as the recurrent rule) also shape longevity directly.
- Lead credits treat joint credits as separate artists, and "had charted before" uses a lookback that grows longer over time (7.5).
- Glee Cast distorts 2009–2012; results for those years should be read with it in mind.
- Entries in late 2025 have little time left in the window (right-censoring).

**Next:** add release dates, genres and artist information (Section 8), analyse when songs enter the chart (Section 9), and build a comparison group of songs that did not chart (Section 10).


## **8. Additional source: MusicBrainz**

### **8.1 Export the Billboard song list for matching**

MusicBrainz is matched to our songs in a local PostgreSQL database built from the official MusicBrainz dump (too large for Colab). This cell exports the list of songs to look up: every printed spelling of every song (`song_names`, Section 6), so that a match can be found under any of them. The file contains Billboard titles and performers, so it is kept private and not published (Section 1.8).

In [ ]:
BB_FOR_MB_FILE = OUTPUT_DIR / "billboard_song_names.csv"
bb_for_mb = read_table("SELECT song_id, name_id, title, performer FROM song_names ORDER BY name_id")
bb_for_mb.to_csv(BB_FOR_MB_FILE, index=False)
print(f"{len(bb_for_mb):,} spellings of {bb_for_mb['song_id'].nunique():,} songs written to {BB_FOR_MB_FILE}")
download_file(BB_FOR_MB_FILE)

### **8.2 How the MusicBrainz information was obtained**

**Source.** MusicBrainz, an open music encyclopedia maintained by the non-profit MetaBrainz Foundation. MetaBrainz publishes the full database as downloadable files ("dumps") about twice a week and keeps each one for about two weeks. We used the full export **`20261017-002244`** (17 October 2026), files `mbdump.tar.bz2` (core data, CC0) and `mbdump-derived.tar.bz2` (tags, CC BY-NC-SA 3.0). Both files were verified against MusicBrainz's published SHA-256 checksums; a private copy is kept, because the dump is removed from the server after about two weeks.

**Why outside Colab.** The tables we need total about 25 GB (e.g. 58.5 million tracks, 40.6 million recordings), too large for a Colab session. They were loaded into a local PostgreSQL 18.6 database on a laptop; only the result (one row per song) is brought into this notebook.

**Steps** (scripts and full log in our MusicBrainz repository, linked in 8.3):

| Script | What it does |
|---|---|
| `mb_01_schema.sql` | Creates the 17 needed tables, with definitions copied unchanged from the official MusicBrainz schema (version 31, matching the dump) |
| `mb_02_load.sql` | Loads the dump files |
| `mb_03_indexes.sql` | Adds keys and indexes for fast joins |
| `mb_04_billboard.sql` | Loads our list of 11,785 spellings of 11,728 songs (8.1) |
| `mb_05_match.sql` | Matches songs to MusicBrainz recordings and derives release dates and genres |
| `mb_06_export.sql` | Exports one row per song |

**Matching rules (`mb_05`), and why:**
- **Same normalisation on both sides** as in Section 5 (case, apostrophes, "Feat."/"Ft.", "&", punctuation, spaces), plus accent removal (MusicBrainz writes "Beyoncé", Billboard "Beyonce").
- **Title and artist must both match** (the artist as lead credit, full credit, or any credited artist), so that covers and songs with the same title by other artists are excluded.
- **Excluded:** music videos, and versions described as live, karaoke, instrumental, demo or a cappella, which are not the charting song.
- **Fallback:** for songs without a match, titles are compared without bracketed parts (e.g. "Soak City (Do It)" vs "Soak City"); marked as such.
- **First release date:** the earliest dated release containing any matched recording, excluding bootlegs and pseudo-releases (translations). Dates may be complete or known only to the month or year; the precision is recorded.
- **Genres:** only user tags that are on MusicBrainz's official genre list and have positive votes; from the most specific level available: the recording, else the artist's own releases (never compilations), else the artist.

**Run result:** 11,224 of 11,728 songs matched (95.7%; 11,033 by exact title, 191 by the fallback); 11,190 with a release date and 11,061 with a genre. Total run time about 15 minutes.

**Reproducibility.** Re-running `run_all.sh` on the same dump reproduces the file exactly (its SHA-256 is recorded). Re-running on a later dump reproduces the method, but results may differ slightly, because MusicBrainz is continuously edited.

### **8.3 Download and verify the MusicBrainz extract**

As with the Billboard archive (Section 3), the extract is downloaded from a fixed commit of a public repository and verified with its SHA-256 fingerprint, recorded when it was produced.

**Why a separate repository.** The PostgreSQL step runs outside this notebook (8.2), so its result has to be published somewhere the notebook can download it from automatically, at a fixed and verifiable version. We use a dedicated public repository, [github.com/Isabelle333/billboard-musicbrainz-extract](https://github.com/Isabelle333/billboard-musicbrainz-extract), that contains only the MusicBrainz scripts, the extracts and their run logs:
- **Reproducibility:** each extract is pinned by commit and checked by SHA-256, exactly like the Billboard archive, and the scripts that produced it sit next to it.
- **Licence:** the extract contains MusicBrainz data, including genres under CC BY-NC-SA 3.0; keeping it in its own repository keeps that licence separate from our own code and notebook.
- **No chart data:** the extract contains MusicBrainz data and our own `song_id` only, so publishing it does not redistribute Billboard data (Section 1.6).

The settings are defined here rather than in Section 2.2 because they only exist after the PostgreSQL step.

In [ ]:
MB_REPO        = "Isabelle333/billboard-musicbrainz-extract"   # repository holding the extract
MB_COMMIT      = "c95bcb0ee8312c2a2145ac7bc613417ef250b74a"                 # pinned snapshot
MB_PATH        = "musicbrainz_for_billboard_songs.csv"
MB_EXPECTED_SHA256 = "a7dcab5ba6888d7a0f11da8d6b20f81fde6540f897400a9c09b3db85c3cfb3af"
MB_DUMP        = "20261017-002244"                                       # MusicBrainz full export used
MB_URL         = f"https://raw.githubusercontent.com/{MB_REPO}/{MB_COMMIT}/{MB_PATH}"

MB_ACCESS_TIME_UTC = datetime.now(timezone.utc).replace(microsecond=0)
mb_bytes = download_bytes(MB_URL)
mb_sha256 = hashlib.sha256(mb_bytes).hexdigest()
print(f"Expected SHA-256 : {MB_EXPECTED_SHA256}")
print(f"Actual SHA-256   : {mb_sha256}")
if mb_sha256 != MB_EXPECTED_SHA256:
    raise ValueError("The MusicBrainz extract is NOT the file produced by our PostgreSQL run. Stop here.")

MB_RAW_FILE = RAW_DIR / "musicbrainz_for_billboard_songs.csv"
MB_RAW_FILE.write_bytes(mb_bytes)
print(f"Integrity check passed. Saved to: {MB_RAW_FILE}")

In [ ]:
MB_TYPES = {"song_id": str, "match_method": str, "n_matched_recordings": "Int64",
            "first_recording_mbid": str, "mb_title": str, "mb_artist_credit": str,
            "first_release_group_mbid": str, "first_release_type": str,
            "first_release_year": "Int64", "first_release_date": str, "date_precision": str,
            "genre_level": str, "top_genre": str, "top_genres_votes": str}
mb = pd.read_csv(io.BytesIO(mb_bytes), dtype=MB_TYPES, keep_default_na=False, na_values=[""])

checks = {
    "rows (expected 11,728)": len(mb),
    "columns as expected": list(mb.columns) == list(MB_TYPES),
    "song_id unique": bool(mb["song_id"].is_unique),
    "every song_id exists in our songs table": bool(mb["song_id"].isin(songs["song_id"]).all()),
    "every one of our songs present": bool(songs["song_id"].isin(mb["song_id"]).all()),
}
display(pd.Series(checks, name="value").to_frame())
if not all(v is True or v == 11728 for v in checks.values()):
    raise ValueError("The MusicBrainz extract does not line up with our songs table.")
mb.head(3)

### **8.4 Quality assessment of the MusicBrainz information**

As for the Billboard data (Section 4), we assess the new information before using it: how many songs were matched and whether coverage differs over time (8.4a); how precise the release dates are and whether they are plausible compared with the chart dates (8.4b–c); a manual check of a random sample of matches against musicbrainz.org (8.4d); and the unmatched songs and genre information (8.4e).

#### **8.4a Coverage**

In [ ]:
coverage = pd.DataFrame({
    "songs": mb["match_method"].value_counts(),
    "with release date": mb.dropna(subset=["first_release_year"])["match_method"].value_counts(),
    "with genre": mb.dropna(subset=["top_genre"])["match_method"].value_counts(),
}).fillna(0).astype(int)
coverage.loc["total"] = coverage.sum()
coverage["share of all songs"] = (coverage["songs"] / len(mb)).round(3)
MB_QA = {"coverage": coverage.to_dict(orient="index")}
display(coverage)

# Join contract: one MusicBrainz row and one chart summary per song.
mb_q = mb.merge(read_table("song_summary")[["song_id", "first_chart_date", "first_entry_uncertain"]],
                on="song_id", how="left", validate="one_to_one")
if len(mb_q) != len(mb) or mb_q["first_chart_date"].isna().any():
    raise ValueError("Join contract broken: a song is missing or duplicated after attaching chart dates.")
mb_q["entry_year"] = mb_q["first_chart_date"].dt.year
mb_q["period"] = pd.cut(mb_q["entry_year"], bins=[1999, 2004, 2009, 2014, 2019, 2025],
                        labels=["2000–04", "2005–09", "2010–14", "2015–19", "2020–25"])
by_period = mb_q.groupby("period", observed=True).agg(
    songs=("song_id", "size"),
    matched=("match_method", lambda m: (m != "no match").mean()),
    with_date=("first_release_year", lambda y: y.notna().mean()),
    with_genre=("top_genre", lambda g: g.notna().mean()),
).round(3)
MB_QA["coverage_by_period"] = by_period.to_dict(orient="index")
by_period

**Observations:** 11,224 of 11,728 songs (95.7%) were matched to MusicBrainz: 11,033 by exact title and 191 by the title without brackets; 11,190 songs (95.4%) have a release date and 11,061 (94.3%) a genre. Coverage is high in every period but lower for the most recent songs: 97.6% matched in 2000–04 against 93.1% in 2020–25.

**Interpretation:** MusicBrainz is edited by volunteers, so recent songs are documented less completely, and recent hits more often have long multi-artist credits that are written differently in the two sources (see 8.4e). Analyses using MusicBrainz information therefore slightly under-represent recent songs; we report the number of songs behind every figure.

#### **8.4b Release dates**

In [ ]:
precision = mb["date_precision"].value_counts(dropna=False).rename("songs").to_frame()
precision["share"] = (precision["songs"] / len(mb)).round(3)
MB_QA["date_precision"] = {str(k): int(v) for k, v in precision["songs"].items()}
display(precision)

dated = mb_q[(mb_q["date_precision"] == "day") & (~mb_q["first_entry_uncertain"])].copy()
dated["release_date"] = pd.to_datetime(dated["first_release_date"], format="%Y-%m-%d")
dated["days_release_to_chart"] = (dated["first_chart_date"] - dated["release_date"]).dt.days

timing_flags = {
    "songs with exact release date and observed first entry": len(dated),
    "release more than 7 days AFTER first chart week": int((dated["days_release_to_chart"] < -7).sum()),
    "release within 7 days after chart week (small date error or pre-release airplay)": int(dated["days_release_to_chart"].between(-7, -1).sum()),
    "release 0–365 days before first chart week": int(dated["days_release_to_chart"].between(0, 365).sum()),
    "release more than 1 year before first chart week": int((dated["days_release_to_chart"] > 365).sum()),
    "release more than 5 years before first chart week": int((dated["days_release_to_chart"] > 5 * 365).sum()),
    "median days from release to first chart week": float(dated["days_release_to_chart"].median()),
}
MB_QA["release_vs_chart_timing"] = timing_flags
pd.Series(timing_flags, name="value").to_frame()

**Observations:** 88.8% of songs have an exact release date, 3.7% only a month, 3.0% only a year, and 4.6% none. Among the 10,306 songs with an exact release date and an observed first chart entry: 9,017 (87.5%) were released within a year before their first chart week; 802 more than a year before (95 of them more than five years before); 88 within a week after; and 399 (3.9%) more than a week after their first chart week, which is impossible in reality. The median time from release to first chart week is 20 days.

**Interpretation:** most release dates are consistent with the chart data, and most songs enter the chart within weeks of release, a first indication relevant to our hypothesis. Because a chart is dated after the week it measures (1.9), no song can genuinely be released after its first chart date. The 88 songs within a week are most likely small errors in the release date or radio play before the official release, so we tolerate them; the 399 cases beyond a week and the very old releases need specific treatment (8.7).

#### **8.4c Implausible release dates**

In [ ]:
cols_check = ["song_id", "display_title", "display_performer", "first_chart_date",
              "mb_title", "mb_artist_credit", "first_release_date", "days_release_to_chart"]
dated_named = dated.merge(read_table("songs")[["song_id", "display_title", "display_performer"]],
                          on="song_id", how="left", validate="one_to_one")   # one name per song
display(dated_named[dated_named["days_release_to_chart"] < -7][cols_check].head(10))
dated_named[dated_named["days_release_to_chart"] > 5 * 365][cols_check].sort_values("days_release_to_chart", ascending=False).head(10)

**Observations:** songs released "after" their first chart week fall into two groups. Some are early-2000s singles that charted on radio airplay before a commercial release (e.g. Aaliyah "Try Again", 10 days); others are matching problems where MusicBrainz's earliest known release is a later or different version (e.g. "Same Script, Different Cast" dated 2004 although it charted in 2000; Janet "Doesn't Really Matter" matched only to a club mix). Songs released more than five years before charting are almost all holiday classics from the 1940s–60s (e.g. Gene Autry, Dean Martin, The Ronettes) that entered the Hot 100 for the first time from 2018 onwards.

**Interpretation:** the first group cannot be corrected with our data; the second is a different phenomenon from new songs reaching the chart. Both are flagged rather than deleted (8.7).

#### **8.4d Manual check of 30 matches**

In [ ]:
MANUAL_CHECK_SEED = 2026
matched_songs = mb[mb["match_method"] != "no match"].merge(
    read_table("songs")[["song_id", "display_title", "display_performer"]],
    on="song_id", how="left", validate="one_to_one")                  # one name per song
manual_sample = matched_songs.sample(n=30, random_state=MANUAL_CHECK_SEED)[
    ["song_id", "display_title", "display_performer", "mb_title", "mb_artist_credit",
     "first_release_date", "top_genre", "first_recording_mbid"]].copy()
manual_sample["musicbrainz_link"] = "https://musicbrainz.org/recording/" + manual_sample["first_recording_mbid"]
manual_sample = manual_sample.drop(columns="first_recording_mbid").reset_index(drop=True)
pd.set_option("display.max_colwidth", 80)
manual_sample

**Manual check (30 randomly selected matches, seed 2026).** For each match we compared the MusicBrainz title, artist credit, release date, release type and genre with the Billboard title and performer and with the song's documented release (album or single, and its release date). The `musicbrainz_link` column lets any reader open the record and repeat the check.

- **Correct song and primary artist: 30 / 30.** Differences are only spelling (curly apostrophes, capitals, "feat." instead of "Featuring", "Part 2" instead of "(Part 2)") or credits listed differently (Tommy Torres missing for "Tu Recuerdo"; John Waters added for "The Creep").
- **Plausible release date: 28 / 30.** Two problems:
  - Justin Bieber "Boyfriend" is dated 2012-01-01, although the single was released in March 2012: a 1 January placeholder, treated as known to the year only (M1, 8.7).
  - Ricky Martin "Tu Recuerdo" is dated 2007 (year only), although it charted from December 2006: MusicBrainz misses the earliest release. M2 compares exact dates only, so this song is not flagged.
- **Sensible genre: 26 / 30.** Four questionable tags: Lee Brice "Memory I Don't Mess With" tagged *pop* (a country song), twenty one pilots "Nico and the Niners" tagged *electronic* rather than alternative/hip hop, Christina Perri "A Thousand Years (Part 2)" tagged *musical*, and Jessie Murph "Always Been You" tagged *country* (a pop ballad; this genre comes from the release, not the recording: 8.4e). The other 26 fit the song or the artist.

**Interpretation:** matching is reliable: every checked match is the right song by the right primary artist, so the matching error rate is probably below about 10% (with no error in 30 checks, the "rule of three" gives 3/30 as an approximate upper bound). Release dates are right in most cases. The two errors come from a 1 January placeholder, which M1 turns into a year-only date, and from a missing earliest release, which M2 cannot detect when the date is known only to the year, so some wrong dates remain. Genre is the weakest field: about one tag in seven is doubtful. Three of the four doubtful tags are on the recording itself and one comes from the release group, so the problem lies in the volunteer tags themselves rather than in genres taken from the artist. This supports grouping genres into broad families (8.7) and reading genre results with caution (11.4, 11.6).

#### **8.4e Unmatched songs and genres**

In [ ]:
unmatched = mb[mb["match_method"] == "no match"].merge(
    read_table("songs")[["song_id", "display_title", "display_performer"]],
    on="song_id", how="left", validate="one_to_one")                  # one name per song
print(f"Unmatched songs: {len(unmatched)}")
display(unmatched[["song_id", "display_title", "display_performer"]].sample(n=15, random_state=MANUAL_CHECK_SEED))

genre_levels = mb["genre_level"].value_counts(dropna=False).rename("songs").to_frame()
top_genres = mb["top_genre"].value_counts().head(20).rename("songs").to_frame()
MB_QA["genre_levels"] = {str(k): int(v) for k, v in genre_levels["songs"].items()}
MB_QA["top_genres_top20"] = top_genres["songs"].to_dict()
MB_QA["distinct_top_genres"] = int(mb["top_genre"].nunique())
print(f"Distinct top genres: {MB_QA['distinct_top_genres']}")
display(genre_levels)
top_genres

**Observations:** the 504 unmatched songs are mostly collaborations whose credits are written differently in the two sources (e.g. "Keyshia Cole Duet With Monica", "Juice WRLD x Marshmello", "Pusha T Kanye West", "¥$: Kanye West & Ty Dolla $ign") and recent songs. MusicBrainz uses 218 different genres for our songs; 7,792 genres come from the recording itself, 1,804 from the artist's release, 1,465 from the artist, and 667 songs have none.

**Interpretation:** unmatched songs are not a random 4.3%: they over-represent collaborations and recent releases, a limitation to state when using MusicBrainz information. 218 genres are far too many to compare, so they are grouped into broad families (8.7).

### **8.5 Store the MusicBrainz information in our database**

The extract is added to our SQLite database (Section 6) as a new table, linked to `songs` by `song_id`, with rules that reject impossible values. All original columns are kept; problems found in 8.4 are handled in the analysis, not by changing the stored values.

In [ ]:
con.executescript("""
DROP TABLE IF EXISTS song_musicbrainz;
CREATE TABLE song_musicbrainz (
    song_id                  TEXT    PRIMARY KEY REFERENCES songs(song_id),
    match_method             TEXT    NOT NULL CHECK (match_method IN ('exact title', 'title without brackets', 'no match')),
    n_matched_recordings     INTEGER NOT NULL CHECK (n_matched_recordings >= 0),
    first_recording_mbid     TEXT,
    mb_title                 TEXT,
    mb_artist_credit         TEXT,
    first_release_group_mbid TEXT,
    first_release_type       TEXT,
    first_release_year       INTEGER CHECK (first_release_year BETWEEN 1800 AND 2026),
    first_release_date       TEXT,
    date_precision           TEXT    CHECK (date_precision IN ('day', 'month', 'year')),
    genre_level              TEXT    CHECK (genre_level IN ('recording', 'release group', 'artist')),
    top_genre                TEXT,
    top_genres_votes         TEXT,
    CHECK ((match_method = 'no match') = (n_matched_recordings = 0))
);
""")
mb.to_sql("song_musicbrainz", con, if_exists="append", index=False)
con.commit()

db_mb = {
    "rows in song_musicbrainz": q("SELECT COUNT(*) AS n FROM song_musicbrainz")["n"][0],
    "foreign-key violations": len(q("PRAGMA foreign_key_check(song_musicbrainz)")),
    "songs without a MusicBrainz row": q("""SELECT COUNT(*) AS n FROM songs s
                                           LEFT JOIN song_musicbrainz m ON m.song_id = s.song_id
                                           WHERE m.song_id IS NULL""")["n"][0],
}
MB_QA["database_checks"] = {k: int(v) for k, v in db_mb.items()}
pd.Series(db_mb, name="value").to_frame()

### **8.6 Save the provenance and quality record**

In [ ]:
MB_PROV_FILE = OUTPUT_DIR / "provenance_quality_musicbrainz.json"
MB_PROV_FILE.write_text(json.dumps({
    "record_type": "provenance and quality assessment – MusicBrainz information for Billboard songs",
    "musicbrainz_dump": MB_DUMP,
    "export_repository": f"https://github.com/{MB_REPO}",
    "export_commit": MB_COMMIT,
    "export_url": MB_URL,
    "export_sha256": mb_sha256,
    "accessed_at_utc": MB_ACCESS_TIME_UTC.isoformat(),
    "manual_check_seed": MANUAL_CHECK_SEED,
    "results": MB_QA,
}, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
download_file(MB_PROV_FILE)

### **8.7 Cleaning the MusicBrainz information**

Based on 8.4, we clean the MusicBrainz information in four logged steps (M1–M4, "M" for MusicBrainz, like C1–C12 in Section 5). As before, the stored original values are not changed: the cleaned values go into a separate table, `song_mb_clean`.

| Step | What | Responds to |
|---|---|---|
| M1 | Exact dates on 1 January treated as known to the year only | placeholder dates (8.4d) |
| M2 | Flag songs released more than 7 days after their first chart week | impossible timing (8.4b–c) |
| M3 | Flag songs released more than 5 years before their first chart week | old songs charting late (8.4c) |
| M4 | Group the 218 genres into 10 broad families | too many genres (8.4e) |

In [ ]:
MB_CLEANING_LOG = []

def log_mb_step(step, description, values_changed, justification):
    """Record one MusicBrainz cleaning step (same idea as log_step in Section 5)."""
    MB_CLEANING_LOG.append({"step": step, "description": description,
                            "values_changed": int(values_changed), "justification": justification})
    print(f"[{step}] {description} | values changed: {values_changed:,}")

mb_raw = read_table("SELECT * FROM song_musicbrainz")
mbc = mb_raw.merge(read_table("song_summary")[["song_id", "first_chart_date", "first_entry_uncertain"]],
                   on="song_id", how="left", validate="one_to_one")   # contract: one partner per song
if len(mbc) != len(mb_raw) or mbc["first_chart_date"].isna().any():
    raise ValueError("Join contract broken: a song is missing or duplicated after attaching chart dates.")

day_dates = mbc["date_precision"] == "day"
jan_first = day_dates & mbc["first_release_date"].str.endswith("-01-01", na=False)
expected_jan_first = day_dates.sum() / 365.25
print(f"Exact dates on 1 January: {int(jan_first.sum())} "
      f"(about {expected_jan_first:.0f} expected if release dates were spread evenly over the year)")

In [ ]:
# M1: 1 January dates count as "year only"; only reliable exact dates get a release_date
mbc["release_date"] = pd.to_datetime(mbc["first_release_date"].where(day_dates & ~jan_first), format="%Y-%m-%d")
mbc["release_precision"] = mbc["date_precision"].where(~jan_first, "year")
log_mb_step("M1", "Treat exact dates on 1 January as known to the year only", jan_first.sum(),
            f"{int(jan_first.sum())} dates fall on 1 January, about {jan_first.sum() / expected_jan_first:.0f} times "
            "more than expected; most are placeholders for 'released that year'.")

# M2: impossible timing
mbc["days_release_to_chart"] = (mbc["first_chart_date"] - mbc["release_date"]).dt.days
after_chart = mbc["days_release_to_chart"] < -7
mbc["flag_release_after_chart"] = after_chart
log_mb_step("M2", "Flag songs whose earliest known release is more than 7 days after their first chart week",
            after_chart.sum(), "A song cannot chart before it exists: MusicBrainz lacks the earliest release "
            "(e.g. radio-only singles) or the match is wrong. Kept, but excluded from time-to-chart analyses.")

# M3: old songs charting decades later
old_release = mbc["days_release_to_chart"] > 5 * 365
mbc["flag_old_song"] = old_release
log_mb_step("M3", "Flag songs first released more than 5 years before their first chart week",
            old_release.sum(), "Mostly old holiday classics and revivals entering the chart decades later; a "
            "different phenomenon from new releases, analysed separately.")

usable = (mbc["days_release_to_chart"].notna() & ~after_chart & ~old_release & ~mbc["first_entry_uncertain"])
pd.Series({"release after chart (M2)": int(after_chart.sum()),
           "old songs (M3)": int(old_release.sum()),
           "usable for time-to-chart analysis": int(usable.sum())}, name="songs").to_frame()

In [ ]:
GENRE_FAMILY_RULES = [   # checked in this order; the first rule with a keyword found in the genre name wins
    ("Holiday",            ["christmas", "holiday"]),
    ("Latin",              ["latin", "reggaeton", "corrido", "bachata", "salsa", "cumbia", "banda",
                            "norteño", "regional mexican", "dembow", "mariachi", "urbano", "bolero"]),
    ("Hip hop / rap",      ["hip hop", "rap", "trap", "drill", "crunk", "dirty south", "grime",
                            "bounce", "phonk", "boom bap", "g-funk", "chipmunk soul"]),
    ("Country",            ["country", "bluegrass", "americana"]),
    ("R&B / soul",         ["r&b", "soul", "funk", "new jack swing", "quiet storm"]),
    ("Other",              ["dancehall", "reggae", "afrobeat"]),
    ("Pop",                ["indie pop", "alternative pop", "art pop", "dream pop", "bedroom pop"]),
    ("Rock / alternative", ["rock", "metal", "punk", "emo", "grunge", "indie", "alternative",
                            "post-hardcore", "new wave", "industrial", "aor"]),
    ("Pop",                ["pop", "ballad", "boy band", "girl group"]),
    ("Electronic / dance", ["electro", "house", "dance", "edm", "techno", "trance", "dubstep", "brostep",
                            "disco", "drum and bass", "club", "big beat", "downtempo", "ambient"]),
]

def genre_family(genre):
    """Broad family of a detailed MusicBrainz genre (first matching rule wins; otherwise 'Other')."""
    if pd.isna(genre):
        return pd.NA
    for family, keywords in GENRE_FAMILY_RULES:
        if any(k in genre for k in keywords):
            return family
    return "Other"

mbc["genre_family"] = mbc["top_genre"].map(genre_family)
log_mb_step("M4", "Group the detailed genres into broad families", mbc["top_genre"].notna().sum(),
            f"{mbc['top_genre'].nunique()} distinct genres are too many to compare; keyword rules in a fixed "
            "order assign each to one of 10 families (full mapping saved).")

genre_map = (mbc.dropna(subset=["top_genre"]).groupby(["genre_family", "top_genre"]).size()
             .rename("songs").reset_index().sort_values(["genre_family", "songs"], ascending=[True, False]))
family_counts = mbc["genre_family"].value_counts(dropna=False).rename("songs").to_frame()
family_counts["share"] = (family_counts["songs"] / len(mbc)).round(3)
display(family_counts)
genre_map.groupby("genre_family").head(6)

**Genre votes: one row per song and genre.** The extract stores each song's five highest-ranked genres in one text field (`top_genres_votes`, e.g. `dance:3|r&b:3|soul:2`), and our analyses use only the first (`top_genre`). The second peer review pointed out that this hides two things: which genre is chosen when several have the same number of votes, and how many top genres rest on a single vote. We therefore (1) store the votes as one row per song and genre (`song_genre_votes`), so that they can be queried, (2) check the rule used by the extraction scripts (`mb_05`, `mb_07`): most votes first, ties broken alphabetically, and (3) measure how often the choice is uncertain. This is done here, after M4, because a tie matters only when the tied genres belong to different families. The original text field stays unchanged in `song_musicbrainz`, as delivered by the extract; the new table is rebuilt from it on every run.

In [ ]:
import unicodedata   # standard library: removes accents when comparing names

# One row per song and genre, instead of one text field per song ("dance:3|r&b:3|soul:2").
# The extract keeps the five highest-ranked genres of each song, at the level the genre was taken from (8.2).
def genre_votes_long(df, id_col):
    """Split the top_genres_votes text into one row per song and genre, keeping MusicBrainz's rank order."""
    s = df[[id_col, "top_genres_votes"]].dropna()
    long = s.assign(pair=s["top_genres_votes"].str.split("|")).explode("pair")
    long["rank"] = long.groupby(id_col).cumcount() + 1
    parts = long["pair"].str.rsplit(":", n=1, expand=True)
    long["genre"], long["votes"] = parts[0], parts[1].astype(int)
    return long[[id_col, "rank", "genre", "votes"]].reset_index(drop=True)

def check_tie_rule(long, top, id_col):
    """The rule used in mb_05/mb_07: most votes first, ties in alphabetical order (PostgreSQL ignores accents
    when sorting, e.g. "alté" before "alternative r&b", so we compare names without accents). Stop if the data disagree."""
    plain = long["genre"].map(lambda g: unicodedata.normalize("NFKD", g).encode("ascii", "ignore").decode())
    stored = long.sort_values([id_col, "rank"], kind="stable")
    expected = long.assign(plain=plain).sort_values([id_col, "votes", "plain"], ascending=[True, False, True], kind="stable")
    if not ((stored[id_col].values == expected[id_col].values) & (stored["genre"].values == expected["genre"].values)).all():
        raise ValueError("Genres are not ordered by votes, then alphabetically.")
    first = stored[stored["rank"] == 1].set_index(id_col)["genre"]
    if not (first == top.reindex(first.index)).all():
        raise ValueError("top_genre is not the first genre of top_genres_votes.")

def tie_profile(long, id_col):
    """Per song: is the top genre tied, does the tie cross genre families, does the top genre rest on one vote?"""
    d = long.assign(family=long["genre"].map(genre_family))
    leaders = d[d["votes"] == d.groupby(id_col)["votes"].transform("max")]
    return pd.DataFrame({
        "tied_top": leaders.groupby(id_col).size() > 1,
        "tie_across_families": leaders.groupby(id_col)["family"].nunique() > 1,
        "top_from_one_vote": d[d["rank"] == 1].set_index(id_col)["votes"] == 1,
    })

song_genre_votes = genre_votes_long(mb, "song_id")
check_tie_rule(song_genre_votes, mb.set_index("song_id")["top_genre"], "song_id")

con.executescript("""
DROP TABLE IF EXISTS song_genre_votes;
CREATE TABLE song_genre_votes (
    song_id  TEXT    NOT NULL REFERENCES songs(song_id),
    rank     INTEGER NOT NULL CHECK (rank BETWEEN 1 AND 5),
    genre    TEXT    NOT NULL,
    votes    INTEGER NOT NULL CHECK (votes >= 1),
    PRIMARY KEY (song_id, genre),
    UNIQUE (song_id, rank)
);
""")
song_genre_votes.to_sql("song_genre_votes", con, if_exists="append", index=False)
con.commit()

ties = tie_profile(song_genre_votes, "song_id").join(mb.set_index("song_id")["genre_level"])
own_level = ties["genre_level"].isin(["recording", "release group"])
tie_table = pd.DataFrame({
    "all songs with a genre": ties[["tied_top", "tie_across_families", "top_from_one_vote"]].mean() * 100,
    "songs with a genre of their own": ties.loc[own_level, ["tied_top", "tie_across_families", "top_from_one_vote"]].mean() * 100,
}).round(1).rename(index={"tied_top": "% top genre tied with another",
                          "tie_across_families": "% tie between different families",
                          "top_from_one_vote": "% top genre from a single vote"})
tie_table.loc["songs"] = [len(ties), int(own_level.sum())]
GENRE_TIES = tie_table.to_dict()        # saved in the MusicBrainz cleaning log (next cell)
print(f"Rows in song_genre_votes: {len(song_genre_votes):,} for {song_genre_votes['song_id'].nunique():,} songs "
      "(rule checked: most votes first, ties alphabetical)")
tie_table

**Observations:** the stored order follows the rule for every song (most votes first, ties alphabetical). Of the 11,061 charting songs with a genre, 24.7% have a top genre tied with another genre, 12.7% with a genre of another family (e.g. dance and R&B), and 27.7% have a top genre that rests on a single vote. Among the 9,596 songs with a genre of their own, the shares are 25.3%, 14.0% and 29.4%.

**Interpretation:** the tie rule is fixed and reproducible, but alphabetical order has no musical meaning: for about one song in eight, the family we assign depends on it. Together with the single-vote genres, this confirms that genre is our least certain variable (8.4d). We keep the rule, because any other choice would be equally arbitrary, and instead test whether the genre results depend on these songs (11.6).

In [ ]:
song_mb_clean = mbc[["song_id", "release_date", "release_precision", "days_release_to_chart",
                     "flag_release_after_chart", "flag_old_song", "genre_family"]].copy()
con.executescript("""
DROP TABLE IF EXISTS song_mb_clean;
CREATE TABLE song_mb_clean (
    song_id                  TEXT    PRIMARY KEY REFERENCES songs(song_id),
    release_date             TEXT,
    release_precision        TEXT    CHECK (release_precision IN ('day', 'month', 'year')),
    days_release_to_chart    INTEGER,
    flag_release_after_chart INTEGER NOT NULL CHECK (flag_release_after_chart IN (0, 1)),
    flag_old_song            INTEGER NOT NULL CHECK (flag_old_song IN (0, 1)),
    genre_family             TEXT
);
""")
to_sql_ready(song_mb_clean).to_sql("song_mb_clean", con, if_exists="append", index=False)
con.commit()

GENRE_MAP_FILE = OUTPUT_DIR / "genre_family_mapping.csv"
genre_map.to_csv(GENRE_MAP_FILE, index=False)
MB_CLEAN_LOG_FILE = OUTPUT_DIR / "cleaning_log_musicbrainz.json"
MB_CLEAN_LOG_FILE.write_text(json.dumps({
    "record_type": "cleaning log – MusicBrainz information",
    "export_sha256": mb_sha256,
    "cleaned_at_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    "steps": MB_CLEANING_LOG,
    "genre_family_rules": GENRE_FAMILY_RULES,
    "genre_family_counts": {str(k): int(v) for k, v in family_counts["songs"].items()},
    "genre_ties": GENRE_TIES,
}, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
download_file(MB_CLEAN_LOG_FILE)
download_file(GENRE_MAP_FILE)

q("""SELECT COUNT(*) AS songs, SUM(release_date IS NOT NULL) AS with_exact_date,
            SUM(flag_release_after_chart) AS release_after_chart, SUM(flag_old_song) AS old_songs,
            SUM(genre_family IS NOT NULL) AS with_genre_family
     FROM song_mb_clean""")

**A stored derived value.** `days_release_to_chart` is computed from the release date and the first chart date, and stored in `song_mb_clean` because the cleaning flags M2 and M3 are defined on it and 9.1 compares it across periods. Storing a value that can be computed from other tables risks an update anomaly: if a chart entry were corrected or two songs merged, the first chart date would change but the stored number would not. In our workflow this cannot happen, because the database is rebuilt from the raw files on every run and never edited afterwards. To make this assumption executable, the final checks (12.3) recompute the value from `song_summary` and stop the notebook if any stored value differs. If the database were ever updated in place, this column should instead be computed in a view, like the song statistics in `song_summary` (Section 6).

### **8.8 Summary of Section 8**

**Observations:** MusicBrainz information was obtained for 95.7% of the charting songs (95.4% with a release date, 94.3% with a genre) from the official dump of 17 October 2026, through a documented PostgreSQL process (8.2) and a verified, published extract (8.3). After cleaning, 10,176 songs have a reliable exact release date and 9,589 can be used for time-to-chart analysis; 399 songs have an impossible release date and 118 are old songs that charted decades later (both flagged); the 218 genres are grouped into 10 families. The genre votes are also stored as one row per song and genre: for 12.7% of songs with a genre, the top genre is tied with a genre of another family, and 27.7% rest on a single vote.

**Interpretation:** the matched information is complete and consistent enough for the planned analyses, with three limitations: coverage is lower for recent songs and collaborations; about 4% of release dates are implausible and excluded from timing analyses; genres come from volunteer tags, for some songs only from the artist, and for about one song in eight the family depends on how a tie is broken (tested in 11.6).

## **9. When do songs reach the chart?**

This section tests our hypothesis (About this study): most songs that reach the Hot 100 do so soon after release, and beyond a certain time a song becomes much less likely to enter.

**Rules used throughout:** release dates from the cleaned MusicBrainz table (8.7); songs grouped by release year; every song is followed for the same fixed number of charts; medians with the 25th–75th percentiles; the number of songs shown for every figure.

### **9.1 How to measure time to chart**

**First approach.** In 8.4 and 8.7 we measured time to chart in days, from the release date to the first chart date (`days_release_to_chart`, median 20 days). Before using it to test our hypothesis, we checked that one day means the same thing in every year. If it does, songs released on the same weekday should show the same typical gap in every period. We compare the most common gap for the three most frequent release weekdays, grouped by the chart rules in force when the song entered (A, B, C: explained below).

In [ ]:
timing = read_table("song_mb_clean").merge(
    read_table("song_summary")[["song_id", "first_chart_date", "first_entry_uncertain"]],
    on="song_id", how="left", validate="one_to_one")
timing = timing[timing["release_date"].notna() & ~timing["flag_release_after_chart"]
                & ~timing["flag_old_song"] & ~timing["first_entry_uncertain"]].copy()

entry_date = timing["first_chart_date"]
timing["chart rules at entry"] = np.select(
    [entry_date < pd.Timestamp("2015-07-25"), entry_date < pd.Timestamp("2018-01-06")],
    ["A: to Jul 2015", "B: Jul 2015 – Dec 2017"], default="C: from Jan 2018")
timing["release weekday"] = timing["release_date"].dt.day_name()

def most_common_gap(days):
    """Most frequent number of days from release to first chart entry, and how many songs have it."""
    mode = days.mode().iloc[0]
    return pd.Series({"songs": len(days), "most common gap (days)": int(mode),
                      "songs with that gap": int((days == mode).sum())})

gap_table = (timing[timing["release weekday"].isin(["Monday", "Tuesday", "Friday"])]
             .groupby(["chart rules at entry", "release weekday"])["days_release_to_chart"]
             .apply(most_common_gap).unstack())
display(gap_table)

**Observations:** the typical gap depends on the period, not only on the song. Friday releases most often entered 22 days after release up to the end of 2017, but 15 days from 2018 (3,223 songs; 2,560 of them at exactly 15 days). Before July 2015, Tuesday releases (then the usual US release day) most often entered after 18 days and Monday releases after 19.

**Explanation:** each chart counts a *tracking week* that ends some days before the date printed on the chart, and Billboard changed these rules twice in our period:

| Rules | Charts dated | Tracking week | Chart dated … after the tracking week ends |
|---|---|---|---|
| A | before 25 July 2015 | Monday–Sunday | 13 days |
| B | 25 July 2015 – 30 December 2017 | Friday–Thursday | 16 days |
| C | from 6 January 2018 | Friday–Thursday | 9 days |

Under rules A, a Tuesday release counts from that week's Sunday (+5 days), and its chart is dated 13 days later: 18 days. Under B and C, Friday releases have the same tracking week, but in 2018 Billboard moved chart dates one week earlier, so the same situation is 22 days under B and 15 days under C. This re-dating also explains the irregular 2018-01-06 chart found in 4.7c.

**Decision:** we do not measure time to chart in days, because 22 days meant "the first chart the song could enter" in 2016 but "the second" in 2020; songs released before 2018 would look one week slower for purely administrative reasons. Instead, for each song we find its **first possible chart** (the first chart whose tracking week ends on or after the release date) and count **how many charts it waited** after that (0 = it entered on its first possible chart). This measure means the same in every year.

*Sources:* [AOL/Billboard, "Billboard to alter chart tracking week for global release date" (2015)](https://www.aol.com/billboard-alter-chart-tracking-week-080000277.html); [Hypebot, "Billboard shifts magazine, chart dates to reflect actual release week" (2017)](https://www.hypebot.com/billboard-shifts-magazine-chart-dates-to-reflect-actual-relkease-week).

### **9.2 How soon after release do charting songs enter?**

In [ ]:
ANALYSIS_RESULTS = {}

# When could a song first have charted? Each chart counts a "tracking week" that ends some days before
# the date printed on the chart. Billboard changed this twice in our period (sources in the text above):
#   charts dated before 2015-07-25: Monday–Sunday tracking week, chart dated 13 days after it ends;
#   2015-07-25 to 2017-12-30:       Friday–Thursday tracking week, chart dated 16 days after it ends;
#   from 2018-01-06:                Friday–Thursday tracking week, chart dated 9 days after it ends.
TRACKING_RULES = [(pd.Timestamp("2015-07-25"), 13), (pd.Timestamp("2018-01-06"), 16), (pd.Timestamp.max, 9)]
FOLLOW_UP_CHARTS = 52                          # fixed observation window: 52 weekly charts (one year)

def days_after_tracking_end(chart_date):
    """Days between the end of a chart's tracking week and the date printed on the chart."""
    for dated_before, lag in TRACKING_RULES:
        if chart_date < dated_before:
            return lag

chart_dates = read_table("charts")["chart_date"].sort_values().reset_index(drop=True)
tracking_end = chart_dates - pd.to_timedelta(chart_dates.map(days_after_tracking_end), unit="D")
if not tracking_end.is_monotonic_increasing:
    raise ValueError("Tracking-week ends are not in order – check TRACKING_RULES.")
chart_position = pd.Series(range(len(chart_dates)), index=chart_dates)   # chart date -> 0, 1, 2, ...

songs_ttc = read_table("song_mb_clean").merge(
    read_table("song_summary")[["song_id", "display_title", "display_performer", "first_chart_date", "first_entry_uncertain"]],
    on="song_id", how="left", validate="one_to_one")
if len(songs_ttc) != len(read_table("songs")) or songs_ttc["first_chart_date"].isna().any():
    raise ValueError("Join contract broken: a song is missing or duplicated.")

dated = songs_ttc["release_date"].notna()
songs_ttc["first_possible_pos"] = pd.Series(
    np.searchsorted(tracking_end.values, songs_ttc["release_date"].values, side="left"), index=songs_ttc.index).where(dated)
songs_ttc["entry_pos"] = songs_ttc["first_chart_date"].map(chart_position)
songs_ttc["charts_waited"] = (songs_ttc["entry_pos"] - songs_ttc["first_possible_pos"]).astype("Int64")
songs_ttc["release_year"] = songs_ttc["release_date"].dt.year.astype("Int64")

# Selection ledger: every filter, with the number of songs left and the reason.
last_pos = len(chart_dates) - 1
steps = [
    ("all charting songs", pd.Series(True, index=songs_ttc.index), "Section 6 database"),
    ("reliable exact release date", dated, "M1: no date, month/year only, or 1 January placeholder"),
    ("release not after first chart week", ~songs_ttc["flag_release_after_chart"], "M2: impossible timing"),
    ("not an old song charting much later", ~songs_ttc["flag_old_song"], "M3: different phenomenon"),
    ("first chart entry observed", ~songs_ttc["first_entry_uncertain"], "first entry may lie before 2000"),
    ("released 2000 or later", songs_ttc["release_year"] >= 2000,
     "same release years as the comparison group; earlier songs may have had chances before our data starts"),
    (f"full {FOLLOW_UP_CHARTS}-chart window inside our data",
     songs_ttc["first_possible_pos"] + FOLLOW_UP_CHARTS - 1 <= last_pos,
     "every song gets the same chance to be seen entering: excludes releases from about December 2024"),
]
keep = pd.Series(True, index=songs_ttc.index)
ledger = []
for label, condition, reason in steps:
    keep &= condition.fillna(False).astype(bool)
    ledger.append({"filter": label, "songs left": int(keep.sum()), "reason": reason})
ttc = songs_ttc[keep].copy()
ttc["waited"] = ttc["charts_waited"].clip(lower=0).astype(int)   # entries before the first possible chart count as 0
ttc_ledger = pd.DataFrame(ledger)
ANALYSIS_RESULTS["selection_ledger"] = ttc_ledger.to_dict("records")
print(f"Release years covered: {ttc['release_year'].min()}–{ttc['release_year'].max()}")
display(ttc_ledger)

In [ ]:
LONG_FOLLOW_UP = 260                                   # 5 years of weekly charts, the same limit as M3
long_term = ttc[ttc["first_possible_pos"] + LONG_FOLLOW_UP - 1 <= last_pos]
print(f"Songs with a full 5-year window: {len(long_term):,} (released "
      f"{long_term['release_year'].min()}–{long_term['release_year'].max()})")

WAIT_BINS   = [-1, 0, 3, 12, 25, 51, LONG_FOLLOW_UP]   # charts waited; each bin includes its right edge
WAIT_LABELS = ["1st possible chart (or earlier)", "2nd–4th chart", "5th–13th chart",
               "14th–26th chart", "27th–52nd chart", "53rd–260th chart"]
wait_table = (pd.cut(long_term["waited"], WAIT_BINS, labels=WAIT_LABELS)
              .value_counts(sort=False).rename("songs").rename_axis("entered on").to_frame())
wait_table["share"] = (wait_table["songs"] / len(long_term)).round(3)
wait_table["cumulative share"] = (wait_table["songs"].cumsum() / len(long_term)).round(3)
ANALYSIS_RESULTS["wait_distribution_5y"] = wait_table.reset_index().astype({"entered on": str}).to_dict("records")
display(wait_table)

cumulative = pd.Series([(long_term["waited"] < k).mean() for k in range(1, FOLLOW_UP_CHARTS + 1)],
                       index=range(1, FOLLOW_UP_CHARTS + 1))
ANALYSIS_RESULTS["cumulative_share_by_chart_5y"] = {int(k): round(float(v), 3) for k, v in cumulative[[1, 2, 4, 13, 26, 52]].items()}

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.step(cumulative.index, cumulative.values * 100, where="post", color=COLOR_MAIN)
for k in [1, 4, 13, 26, 52]:
    ax.annotate(f"{cumulative[k] * 100:.0f}% by chart {k}", (k, cumulative[k] * 100),
                textcoords="offset points", xytext=(6, -14), fontsize=9)
ax.set_ylim(0, 100)
style_axes(ax, "How soon after release do charting songs enter the Hot 100?",
           "Weekly charts since the song's first possible chart (1 = first possible)",
           "% of songs that have entered (cumulative)")
save_figure(fig, "9_2_cumulative_entry")
plt.show()

**Observations:** we followed 6,813 charting songs released in 2000–2020 for five years after their first possible chart. 38% entered on the very first chart they could have entered, 54% within four charts (about a month), 80% within 26 charts (six months) and 92% within a year; 8% entered between one and five years after release.

**Interpretation:** consistent with our hypothesis, most charting songs enter soon after release. A minority enter much later; part of this may be measurement rather than behaviour, because MusicBrainz gives the first release of the *recording*, often the album, while a song may only have been promoted as a single months later. 11.6 checks this: entries after more than a year are three times as common among songs first released on an album (9.8%) as among songs first released as singles (3.3%). But singles, whose release date is the single itself, enter even faster (46% on their first possible chart), so the early concentration of entries does not come from album dates.

### **9.3 Has the speed of entry changed over time?**

In [ ]:
in_window = ttc[ttc["waited"] < FOLLOW_UP_CHARTS].assign(
    period=lambda d: pd.cut(d["release_year"].astype(int), [1999, 2004, 2009, 2014, 2019, 2024],
                            labels=["2000–04", "2005–09", "2010–14", "2015–19", "2020–24"]))
print(f"Songs entering within {FOLLOW_UP_CHARTS} charts of their first possible chart: {len(in_window):,} "
      f"(of {len(ttc):,}; {len(ttc) - len(in_window):,} entered later)")

def timing_summary(group):
    """Size, median and spread of charts waited, and the share entering early, for one group of songs."""
    w = group["waited"]
    return pd.Series({"songs": len(w), "median charts waited": w.median(),
                      "25th pct": w.quantile(.25), "75th pct": w.quantile(.75),
                      "% on 1st possible chart": round((w == 0).mean() * 100, 1),
                      "% within 4 charts": round((w <= 3).mean() * 100, 1)})

by_period = in_window.groupby("period", observed=True)[["waited"]].apply(timing_summary).astype({"songs": int})
by_year = in_window.groupby("release_year")[["waited"]].apply(timing_summary).astype({"songs": int})
ANALYSIS_RESULTS["timing_by_period"] = by_period.reset_index().astype({"period": str}).to_dict("records")
ANALYSIS_RESULTS["timing_by_release_year"] = by_year.reset_index().to_dict("records")
display(by_period)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(by_year.index, by_year["% on 1st possible chart"], marker="o", color=COLOR_MAIN, label="on 1st possible chart")
ax.plot(by_year.index, by_year["% within 4 charts"], marker="o", color=COLOR_SECOND, label="within 4 charts")
ax.set_ylim(0, 100)
ax.legend(frameon=False)
style_axes(ax, "Charting songs enter sooner after release than they used to",
           "Release year", f"% of songs entering within {FOLLOW_UP_CHARTS} charts")
save_figure(fig, "9_3_speed_by_release_year")
plt.show()

**Observations:** among songs entering within a year, the median wait fell from 14 charts for songs released in 2000–2004 to 0 for 2020–2024: the share entering on the first possible chart rose from 14% to 70%, and the share entering within four charts from 23% to 86%. The rise is not steady: it peaks in 2010 (52% on the first chart), dips in 2012–2015 (36–41%), then rises again from 2016.

**Interpretation:** a song's first weeks after release have become much more decisive. The timing matches the changes to the Hot 100 formula described in 7.2: downloads (2005) and streaming (from 2012, weighted more from 2018) are counted from release day, whereas in the early 2000s songs typically built up radio airplay over weeks. This is consistent with, but does not prove, an effect of these changes. It also matches 7.4 (more new songs entering high, more songs lasting one week).

### **9.4 Is there a threshold after which songs rarely enter?**

9.2 shows *how many* songs entered by each chart. To test for a threshold, we ask a different question for each chart *k*: **of the songs that had not entered yet, what share entered at chart *k*?** (the "entry rate"). A threshold would appear as a sharp drop in this rate. We use the 6,813 songs followed for five years, so that songs entering after a year are still counted among those "not entered yet".

In [ ]:
def entry_rates(waited, charts):
    """For each chart k: of the songs that had not entered before chart k, the share entering at chart k."""
    not_yet = np.array([(waited >= k).sum() for k in range(charts)])
    entering = np.array([(waited == k).sum() for k in range(charts)])
    return pd.DataFrame({"chart": range(1, charts + 1), "songs not yet entered": not_yet,
                         "entered at this chart": entering, "entry rate": (entering / not_yet).round(3)})

entry_rate = entry_rates(long_term["waited"], FOLLOW_UP_CHARTS)
ANALYSIS_RESULTS["entry_rate_by_chart_5y"] = entry_rate.to_dict("records")
display(entry_rate.head(8))
print(f"Average entry rate, charts 3–13: {entry_rate.loc[2:12, 'entry rate'].mean():.3f}; "
      f"charts 14–52: {entry_rate.loc[13:, 'entry rate'].mean():.3f}")

long_periods = long_term[long_term["release_year"] <= 2019].assign(
    period=lambda d: pd.cut(d["release_year"].astype(int), [1999, 2004, 2009, 2014, 2019],
                            labels=["2000–04", "2005–09", "2010–14", "2015–19"]))
rate_by_period = (long_periods.groupby("period", observed=True)["waited"]
                  .apply(lambda w: entry_rates(w, 6).set_index("chart")["entry rate"])
                  .unstack().rename(columns=lambda k: f"chart {k}").rename_axis(columns=None))
rate_by_period.insert(0, "songs", long_periods.groupby("period", observed=True).size())
ANALYSIS_RESULTS["entry_rate_by_period_5y"] = rate_by_period.reset_index().astype({"period": str}).to_dict("records")
display(rate_by_period)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(entry_rate["chart"], entry_rate["entry rate"] * 100, color=COLOR_MAIN)
style_axes(ax, "Charting songs only: after the first two possible charts, the weekly chance of entering stays low",
           "Weekly chart since the song's first possible chart (1 = first possible)",
           "% of songs not yet entered that enter at this chart")
save_figure(fig, "9_4_entry_rate")
plt.show()

**Observations:** of the songs that had not entered yet, 38% entered on the first possible chart and 19% on the second; from the third chart onwards the rate falls to about 4% per chart and stays at that level for the rest of the year (4.0% on average for charts 3–13, 3.5% for charts 14–52). The rate on the first chart rose from 12% for songs released in 2000–2004 to 47% in 2015–2019; in 2000–2004, the rate was low from the start (3–4% from the second chart).

**Interpretation:** the data show a clear threshold after the first two possible charts: a song that has not entered by then is about five to ten times less likely to enter in any given later week. The second chart is still high partly because some releases only had a few days in their first tracking week (e.g. Thursday releases since 2015). After the threshold the chance does not fall further but stays low and roughly constant: a steady trickle of "slow-burning" songs. The threshold has become sharper over time: in the early 2000s it was much weaker, because only 12% of songs entered on their first possible chart and most charting songs entered gradually over the following months.

**Limitation:** all these songs eventually charted, so the analysis shows *when* charting songs enter, not how likely *any* song is to chart. That requires songs that never charted: the comparison group (Section 10), analysed in Section 11.

### **9.5 Does the speed of entry differ by genre?**

We compare the genre families from 8.7 (M4). Because the genre mix of the chart changed strongly over time (7.2) and entry became faster over time (9.3), a difference between genres could simply reflect *when* their songs were released. We therefore also compare genres **within each release period**. Shares are only shown for groups of at least 30 songs; smaller groups are too noisy to compare.

This describes how charting songs of each genre reached the chart. It does not tell us whether a genre makes a song more likely to chart; that needs the comparison group (Sections 10–11).

In [ ]:
MIN_SONGS = 30            # smallest group for which a share is shown; smaller groups are too noisy

by_family = (in_window.dropna(subset=["genre_family"])
             .groupby("genre_family")[["waited"]].apply(timing_summary)
             .astype({"songs": int}).sort_values("songs", ascending=False))
print(f"Songs with a genre family: {int(in_window['genre_family'].notna().sum()):,} of {len(in_window):,}")
ANALYSIS_RESULTS["timing_by_genre_family"] = by_family.reset_index().to_dict("records")
display(by_family)

In [ ]:
MAIN_FAMILIES = by_family.index[by_family["songs"] >= 300].tolist()   # families large enough to split by period
fam = in_window[in_window["genre_family"].isin(MAIN_FAMILIES)]

def share_first_chart(w):
    """% of songs entering on their first possible chart, or missing if the group is too small."""
    return round((w == 0).mean() * 100, 1) if len(w) >= MIN_SONGS else np.nan

first_by_family_period = fam.pivot_table(index="genre_family", columns="period", values="waited",
                                         aggfunc=share_first_chart, observed=True)
songs_by_family_period = fam.pivot_table(index="genre_family", columns="period", values="waited",
                                         aggfunc="size", observed=True)
ANALYSIS_RESULTS["first_chart_share_by_family_period"] = first_by_family_period.reset_index().to_dict("records")
print("% entering on their first possible chart (empty: fewer than 30 songs)")
display(first_by_family_period)
print("Number of songs")
display(songs_by_family_period)

composition = (pd.crosstab(in_window["genre_family"].fillna("no genre"), in_window["period"], normalize="columns") * 100).round(1)
ANALYSIS_RESULTS["genre_mix_by_period"] = composition.reset_index().to_dict("records")
print("Genre mix of charting songs, % of each release period")
display(composition)

fig, ax = plt.subplots(figsize=(9, 4.5))
for family in MAIN_FAMILIES:
    row = first_by_family_period.loc[family]
    ax.plot([str(p) for p in row.index], row.values, marker="o", label=family)
ax.set_ylim(0, 100)
ax.legend(frameon=False, ncol=2)
style_axes(ax, "Entry on the first possible chart, by genre family",
           "Release period", "% of songs entering on their 1st possible chart")
save_figure(fig, "9_5_first_chart_by_genre")
plt.show()

**Observations:** among charting songs, hip hop entered fastest (60% on the first possible chart, 79% within four charts) and country slowest (32% and 41%; median wait 9 charts). Within each release period, country is the slowest family in every period (e.g. 55% on the first chart in 2020–2024 against 79% for hip hop), and hip hop the fastest from 2015 onwards; in 2010–2014, pop and rock were fastest (52%). Meanwhile hip hop's share of charting songs rose from 27% (2000–2004) to 46% (2020–2024), while rock fell from 17% to 4%. Holiday songs (22) are too few to compare.

**Interpretation:** genre is clearly associated with *how* songs reach the chart. The difference between country and hip hop holds within every period, so it is not only an effect of when songs were released. A plausible explanation, which our data cannot test, is that country songs are promoted through radio and climb over weeks, while hip hop is driven by streaming from release day. Part of the overall speed-up in 9.3 comes from the growing share of hip hop, but every family also became faster, so the speed-up is not only a change in genre mix.

**Limitations:** genres are volunteer tags, for 1,465 songs taken from the artist rather than the song (8.4e), and the families are our own grouping (8.7). All songs here charted, so these results say nothing about whether a genre helps a song chart (Sections 10–11).

### **9.6 Single or album track?**

MusicBrainz records the type of each song's first release (8.2): a single, an album or an EP. We compare how quickly songs entered depending on how they were first released, again within release periods. The type is taken unchanged from `song_musicbrainz` (M1–M4 did not change it); rare types (broadcast, other, unknown) are grouped as "Other / unknown".

In [ ]:
release_types = read_table("SELECT song_id, first_release_type FROM song_musicbrainz")
typed = in_window.merge(release_types, on="song_id", how="left", validate="one_to_one")
if len(typed) != len(in_window):
    raise ValueError("Join contract broken: rows changed when attaching release types.")
typed["release_type"] = typed["first_release_type"].where(
    typed["first_release_type"].isin(["Single", "Album", "EP"]), "Other / unknown")

by_type = (typed.groupby("release_type")[["waited"]].apply(timing_summary)
           .astype({"songs": int}).sort_values("songs", ascending=False))
first_by_type_period = typed.pivot_table(index="release_type", columns="period", values="waited",
                                         aggfunc=share_first_chart, observed=True)
type_mix = (pd.crosstab(typed["release_type"], typed["period"], normalize="columns") * 100).round(1)
ANALYSIS_RESULTS["timing_by_release_type"] = by_type.reset_index().to_dict("records")
ANALYSIS_RESULTS["first_chart_share_by_type_period"] = first_by_type_period.reset_index().to_dict("records")
ANALYSIS_RESULTS["release_type_mix_by_period"] = type_mix.reset_index().to_dict("records")
display(by_type)
print("% entering on their first possible chart (empty: fewer than 30 songs)")
display(first_by_type_period)
print("Type of first release, % of each release period")
display(type_mix)

**Observations:** 59% of charting songs were first released on an album, 38% as a single. Until 2010–2014, songs first released as singles entered faster (60% on the first possible chart in 2010–2014, against 26% for album tracks). From 2015 this reversed: in 2020–2024, 84% of album tracks entered on their first possible chart, against 47% of singles. The share of charting songs first released on an album fell from 86% (2000–2004) to 42% (2010–2014), then rose again to 62% (2020–2024).

**Interpretation:** in the streaming era, an album release can put several of its tracks on the chart in the same week, so album tracks that chart do so immediately. Singles, often released ahead of an album, more often enter later, possibly when the album comes out or once they are promoted. In the early 2000s, the slow entries of album tracks (9.2–9.3) may reflect songs that charted only once they were promoted to radio; our data contain no information on promotion, so this cannot be tested.

**Limitations:** "first release" is the earliest dated release in MusicBrainz; if a single and its album were released on the same day, either can be counted first. Early-2000s singles may also be less documented in MusicBrainz than albums, which would inflate the share of album first releases in that period.

### **9.7 Do established artists' songs enter faster?**

7.5 showed that about three quarters of charting songs come from lead credits that had charted before. Here we ask whether those songs also enter *faster* than songs by artists reaching the chart for the first time. We reuse the 7.5 definition: a lead credit "had charted before" if it appeared on the Hot 100 before this song's first entry, including before 2000. Comparisons are again made within release periods, with at least 30 songs per share.

In [ ]:
artist_history = entered[["song_id", "lead_charted_before"]]
history = in_window.merge(artist_history, on="song_id", how="left", validate="one_to_one")
if len(history) != len(in_window) or history["lead_charted_before"].isna().any():
    raise ValueError("Join contract broken: a song is missing its artist history.")
history["lead artist"] = np.where(history["lead_charted_before"], "had charted before", "first chart entry")

by_history = (history.groupby("lead artist")[["waited"]].apply(timing_summary).astype({"songs": int}))
first_by_history_period = history.pivot_table(index="lead artist", columns="period", values="waited",
                                              aggfunc=share_first_chart, observed=True)
songs_by_history_period = history.pivot_table(index="lead artist", columns="period", values="waited",
                                              aggfunc="size", observed=True)
ANALYSIS_RESULTS["timing_by_artist_history"] = by_history.reset_index().to_dict("records")
ANALYSIS_RESULTS["first_chart_share_by_history_period"] = first_by_history_period.reset_index().to_dict("records")
display(by_history)
print("% entering on their first possible chart (empty: fewer than 30 songs)")
display(first_by_history_period)
print("Number of songs")
display(songs_by_history_period)

fig, ax = plt.subplots(figsize=(9, 4.5))
for label, color in [("had charted before", COLOR_MAIN), ("first chart entry", COLOR_SECOND)]:
    row = first_by_history_period.loc[label]
    ax.plot([str(p) for p in row.index], row.values, marker="o", color=color, label=f"lead artist {label}")
ax.set_ylim(0, 100)
ax.legend(frameon=False)
style_axes(ax, "Entry on the first possible chart: established vs new artists",
           "Release period", "% of songs entering on their 1st possible chart")
save_figure(fig, "9_7_first_chart_by_artist_history")
plt.show()

**Observations:** 80% of the songs (6,692 of 8,416) came from lead credits that had charted before. Their songs entered faster: 52% on the first possible chart (median wait 0 charts) against 38% for artists entering the chart for the first time (median 2 charts). The gap appeared over time: in 2000–2004 newcomers were not slower (18% against 13% on the first chart), but from 2010–2014 established artists were clearly faster (48% against 29%), and in 2020–2024 74% against 54%.

**Interpretation:** for established artists, chart entry has become largely a matter of release week: an existing audience streams a new release immediately, which also explains the album "bursts" in 7.6. Newcomers more often build up over several weeks, for example through radio or viral spread. In the early 2000s, when most songs climbed slowly through radio (9.3), an artist's history made no difference to the speed of entry.

**Limitations:** "had charted before" is based on the lead credit text (7.5), so a collaboration credit such as "A & B" counts as a separate credit from "A". All songs here charted: whether being established makes a song more likely to chart at all cannot be tested with our comparison group, because all its artists have charted (Section 10).

### **9.8 Save the analysis results**

In [ ]:
def without_nan(obj):
    """Replace missing numbers (NaN) by null, so the file is valid JSON for any program."""
    if isinstance(obj, dict):
        return {k: without_nan(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [without_nan(v) for v in obj]
    if isinstance(obj, float) and np.isnan(obj):
        return None
    return obj

ANALYSIS_RESULTS["release_to_chart_gap_by_rules"] = gap_table.reset_index().to_dict("records")
ANALYSIS_FILE = OUTPUT_DIR / "analysis_results_time_to_chart.json"
ANALYSIS_FILE.write_text(json.dumps(without_nan({
    "record_type": "analysis results – time to chart of Billboard Hot 100 songs (aggregates only)",
    "source_commit": SOURCE_COMMIT,
    "musicbrainz_export_sha256": mb_sha256,
    "computed_at_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    "tracking_rules": [{"charts_dated_before": None if d == pd.Timestamp.max else str(d.date()),
                        "days_after_tracking_week": lag} for d, lag in TRACKING_RULES],
    "follow_up_charts": FOLLOW_UP_CHARTS,
    "long_follow_up_charts": LONG_FOLLOW_UP,
    "min_songs_per_share": MIN_SONGS,
    "results": ANALYSIS_RESULTS,
}), indent=2, ensure_ascii=False, allow_nan=False, default=str), encoding="utf-8")
download_file(ANALYSIS_FILE)   # the figures are zipped and downloaded once, with all others, in 11.8

### **9.9 Summary of Section 9**

**Observations:**
1. Because Billboard changed how charts are dated (2015, 2018), time to chart is measured in charts waited after a song's first possible chart, not in days (9.1).
2. Of charting songs released 2000–2020, 38% entered on their first possible chart, 54% within four charts and 92% within a year (9.2).
3. Entry became much faster: the share entering on the first possible chart rose from 14% (released 2000–2004) to 70% (2020–2024) (9.3).
4. There is a threshold after the first two possible charts: the weekly chance of entering falls from 38% and 19% to about 4% and stays there (9.4).
5. Country songs entered slowest and hip hop fastest, in every period (9.5); album tracks entered fastest in the streaming era, singles before it (9.6); established artists' songs entered faster from 2010 onwards (9.7).

**Interpretation:** for songs that reach the Hot 100, our hypothesis is supported: success is mostly decided in the first weeks after release, and increasingly so in the streaming era. Genre, release type and artist history are associated with how quickly songs enter, but these are associations among charting songs only. They may partly reflect how songs are promoted, which our data do not record, and they do not show what makes a song chart at all. That requires songs that did not chart (Sections 10–11).

## **10. Comparison group: songs that did not chart**

Section 9 described *when* charting songs enter the Hot 100, but every song analysed there reached the chart (9.4). To study *whether* a song reaches the chart, we also need songs that did not. This section builds that comparison group and takes it through the same steps as the Billboard data: collection (10.1–10.2), checks (10.3–10.4), cleaning (10.5) and storage (10.6–10.7). The comparison itself follows in Section 11.

### **10.1 How the comparison group was obtained (Job B)**

As a comparison group we use **songs by the same lead artists that never charted**, released 2000–2024 (decision log, rows 12–14). Using the same artists means every song in the group comes from an artist with proven access to the US market, rather than from the millions of obscure recordings in MusicBrainz. The songs were extracted from the same MusicBrainz database as in Section 8, with a second script (`run_comparison.sh`, `mb_07`, `mb_08`), using the same rules as for the charting songs:

- **Lead artist** of each charting song: the first credited artist on its matched recordings.
- **Candidate songs:** every recording on which one of these artists is credited first, without music videos and live, karaoke, instrumental, demo or a cappella versions.
- **One song = artist + title without brackets**, so that remixes and re-releases count once.
- **Excluded:** any title equal to a Billboard title (by any artist), anything matched to a charting song, and intros, outros, interludes and skits.
- **First release date, release type and genre:** the same rules as for the charting songs (8.2).

How many recordings each rule removed (from `run_log_comparison.txt`):

| Step | Recordings or songs |
|---|---|
| Candidate recordings of the 2,102 lead artists | 964,829 |
| Removed: same title as a Billboard song | 230,028 |
| Removed: matched to a charting song | 52,867 |
| Removed: intro, outro, interlude or skit | 6,981 |
| Removed: empty title | 5,740 |
| Recordings kept | 725,119 |
| Songs (artist + title) | 288,889 |
| … with a release date | 243,274 |
| … first released 2000–2024 | **186,819** |

A recording can be removed for several reasons, so the removals add up to more than the difference. Running the script twice produced byte-identical files (same SHA-256), which confirms that the extraction is reproducible.

**A limitation of the title rule.** Excluding every title equal to a Billboard title by *any* artist is deliberately wide: it makes sure that no hit, in any version or credit, is counted as a non-charting song. But it also removes songs that merely share a common title with someone else's hit (e.g. a song called "Home"), and it is the largest single filter (230,028 of 964,829 candidate recordings, about 24%). Because it removes only non-charting candidates, it can only make chart rates too high. Re-running `mb_07` with the rule limited to the same artist would change the published extract, its fingerprints and every result of Sections 10–13, so we keep the rule and estimate its largest possible effect instead (11.6; decision log row 30).

### **10.2 Download and verify**

In [ ]:
CMP_COMMIT = "6d45e7be9a9920e85e3d8feaf14c92e648c0d9c9"     # pinned snapshot containing the Job B files
CMP_FILES = {                                               # file -> SHA-256 recorded by run_comparison.sh
    "comparison_songs_musicbrainz.csv.gz": "fa4fb810e4533bd452a0c0aa8d3ec56a2aa863da50beadd1de16dfefdde7bc4d",
    "billboard_song_lead_artist.csv":      "603571bea2d887dd460556b5f39dd40864934b4728b03084291387e847ef4206",
}

CMP_ACCESS_TIME_UTC = datetime.now(timezone.utc).replace(microsecond=0)
cmp_bytes = {}
for path, expected_sha256 in CMP_FILES.items():
    url = f"https://raw.githubusercontent.com/{MB_REPO}/{CMP_COMMIT}/{path}"
    data = download_bytes(url)
    file_sha256 = hashlib.sha256(data).hexdigest()   # own name: actual_sha256 is the Billboard file's (3.2)
    print(f"{path}\n  expected SHA-256: {expected_sha256}\n  actual SHA-256  : {file_sha256}")
    if file_sha256 != expected_sha256:
        raise ValueError(f"{path} is NOT the file produced by our PostgreSQL run. Stop here.")
    (RAW_DIR / path).write_bytes(data)
    cmp_bytes[path] = data
print(f"Integrity checks passed. Both files saved in: {RAW_DIR}")

### **10.3 Structure**

In [ ]:
CMP_TYPES = {"comparison_id": str, "artist_mbid": str, "mb_title": str, "n_recordings": "Int64",
             "first_release_type": str, "first_release_year": "Int64", "first_release_date": str,
             "date_precision": str, "genre_level": str, "top_genre": str, "top_genres_votes": str}
LEAD_TYPES = {"song_id": str, "artist_mbid": str, "artist_name": str}
cmp_raw = pd.read_csv(io.BytesIO(cmp_bytes["comparison_songs_musicbrainz.csv.gz"]), compression="gzip",
                      dtype=CMP_TYPES, keep_default_na=False, na_values=[""])
lead_raw = pd.read_csv(io.BytesIO(cmp_bytes["billboard_song_lead_artist.csv"]),
                       dtype=LEAD_TYPES, keep_default_na=False, na_values=[""])

matched_ids = set(read_table("SELECT song_id FROM song_musicbrainz WHERE match_method <> 'no match'")["song_id"])
structure = {
    "comparison songs (expected 186,819)": len(cmp_raw),
    "comparison columns as expected": list(cmp_raw.columns) == list(CMP_TYPES),
    "comparison_id unique": bool(cmp_raw["comparison_id"].is_unique),
    "every release year within 2000–2024": bool(cmp_raw["first_release_year"].between(2000, 2024).all()),
    "lead-artist rows (expected 11,224)": len(lead_raw),
    "lead-artist columns as expected": list(lead_raw.columns) == list(LEAD_TYPES),
    "song_id unique in lead-artist file": bool(lead_raw["song_id"].is_unique),
    "lead-artist file covers exactly our matched songs": set(lead_raw["song_id"]) == matched_ids,
    "every comparison artist is a lead artist": bool(cmp_raw["artist_mbid"].isin(lead_raw["artist_mbid"]).all()),
}
CMP_QA = {"structure": {k: (v if isinstance(v, bool) else int(v)) for k, v in structure.items()}}
display(pd.Series(structure, name="value").to_frame())
if not all(v is True or v in (186_819, 11_224) for v in structure.values()):
    raise ValueError("The comparison files do not have the expected structure.")
cmp_raw.head(3)

### **10.4 Quality assessment**

In [ ]:
import unicodedata   # standard library: removes accents (e.g. "Beyoncé" -> "Beyonce")

def title_key(titles):
    """Title without bracketed parts, accents, case and punctuation (the rule used in mb_07)."""
    plain = titles.map(lambda t: unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode())
    return make_key(plain.str.replace(r"\s*[\(\[][^\)\]]*[\)\]]", "", regex=True))

# 1. Contamination: does any comparison song have the same artist and title as one of our charting songs?
charting_titles = (read_table("song_names")[["song_id", "title"]]
                   .merge(lead_raw[["song_id", "artist_mbid"]], on="song_id", how="inner", validate="many_to_one")
                   .assign(key=lambda d: title_key(d["title"]))[["artist_mbid", "key"]].drop_duplicates())
same_song = (cmp_raw.assign(key=title_key(cmp_raw["mb_title"]))
                    .merge(charting_titles, on=["artist_mbid", "key"], how="inner", validate="many_to_one"))
print(f"Comparison songs with the same artist and title as a charting song: {len(same_song)}")

# 2. Who are the lead artists, and how many comparison songs does each contribute?
artist_names = lead_raw.drop_duplicates("artist_mbid").set_index("artist_mbid")["artist_name"]
per_artist = pd.DataFrame({
    "artist_name": artist_names,
    "charting_songs": lead_raw.groupby("artist_mbid").size(),
    "comparison_songs": cmp_raw.groupby("artist_mbid").size(),
}).fillna({"comparison_songs": 0}).astype({"comparison_songs": int})
placeholder = per_artist["artist_name"].str.fullmatch(r"\[.*\]")

released = (lead_raw.merge(read_table("SELECT song_id, first_release_year FROM song_musicbrainz"), on="song_id",
                           how="left", validate="one_to_one")
                    .merge(read_table("SELECT song_id, first_chart_date FROM song_summary"), on="song_id",
                           how="left", validate="one_to_one"))
released["new_release"] = released["first_release_year"] >= released["first_chart_date"].dt.year - 5
only_old_songs = ~released.groupby("artist_mbid")["new_release"].any()
per_artist["only_old_songs"] = only_old_songs.reindex(per_artist.index).fillna(False).astype(bool)

top_share = per_artist["comparison_songs"].nlargest(20).sum() / len(cmp_raw)
artist_checks = {
    "lead artists": len(per_artist),
    "lead artists without comparison songs": int((per_artist["comparison_songs"] == 0).sum()),
    "placeholder 'artists' (names in [brackets])": int(placeholder.sum()),
    "  comparison songs of these placeholders": int(per_artist.loc[placeholder, "comparison_songs"].sum()),
    "artists that charted only with songs released 5+ years earlier": int(per_artist["only_old_songs"].sum()),
    "  comparison songs of these artists": int(per_artist.loc[per_artist["only_old_songs"], "comparison_songs"].sum()),
    "median comparison songs per artist": float(per_artist["comparison_songs"].median()),
    "share of comparison songs from the 20 largest catalogues": round(float(top_share), 3),
}
CMP_QA["contamination_same_artist_and_title"] = len(same_song)
CMP_QA["artists"] = artist_checks
display(pd.Series(artist_checks, name="value").to_frame())
per_artist.sort_values("comparison_songs", ascending=False).head(12)

In [ ]:
charting_2000_2024 = read_table("""SELECT * FROM song_musicbrainz
                                    WHERE match_method <> 'no match' AND first_release_year BETWEEN 2000 AND 2024""")

NOT_MUSIC = ["non-music", "interview", "spoken word", "field recording"]   # MusicBrainz genres that are not songs
not_music = pd.DataFrame({"charting songs": charting_2000_2024["top_genre"].value_counts().reindex(NOT_MUSIC),
                          "comparison songs": cmp_raw["top_genre"].value_counts().reindex(NOT_MUSIC)}).fillna(0).astype(int)
CMP_QA["main_genre_not_music"] = not_music.to_dict()
print("Songs whose main genre says they are not music:")
display(not_music)

def measurement_profile(df):
    """How completely the songs of one group are described in MusicBrainz (shares in %)."""
    day = df["date_precision"] == "day"
    return pd.Series({
        "songs": len(df),
        "% exact release date": round(day.mean() * 100, 1),
        "% of exact dates on 1 January": round((day & df["first_release_date"].str.endswith("-01-01")).sum()
                                               / day.sum() * 100, 1),
        "% genre from the recording": round((df["genre_level"] == "recording").mean() * 100, 1),
        "% genre from the release group": round((df["genre_level"] == "release group").mean() * 100, 1),
        "% genre from the artist only": round((df["genre_level"] == "artist").mean() * 100, 1),
        "% no genre": round(df["genre_level"].isna().mean() * 100, 1),
        "% first released on an album": round((df["first_release_type"] == "Album").mean() * 100, 1),
        "% first released as a single": round((df["first_release_type"] == "Single").mean() * 100, 1),
    })

measurement = pd.DataFrame({"charting songs (released 2000–2024)": measurement_profile(charting_2000_2024),
                            "comparison songs": measurement_profile(cmp_raw)})
CMP_QA["measurement_by_group"] = measurement.to_dict()
measurement

**Observations:** no comparison song has the same artist and title as a charting song, so the exclusion of charting songs worked. The 2,102 lead artists contribute a median of 57 comparison songs each; the 20 largest catalogues hold 10.5% of all comparison songs (e.g. Gucci Mane 1,661, the film composer James Newton Howard 1,557). The check also revealed three problems: two placeholder "artists" used by MusicBrainz for cast and soundtrack recordings, `[theatre]` and `[Disney]` (3,739 songs); 54 artists that reached the chart only with songs released more than five years earlier, such as holiday classics by Bing Crosby or Nat King Cole and revivals such as Kate Bush (6,450 songs); and 511 recordings that are not music (interviews, spoken word, field recordings). Finally, the two groups are not described equally well: comparison songs less often have an exact release date (86% against 93%), their genre comes much more often from the artist alone (42% against 13%), and they were much more often first released on an album (82% against 63%) and much less often as a single (10% against 34%).

**Interpretation:** the three problems are cleaned below (K1–K3). The other differences must be kept in mind in the analysis: an artist-level genre is the same for all of an artist's songs, so genre comparisons between charting and non-charting songs are only meaningful for songs with their own genre; and because most non-charting songs are album tracks, a difference in release type partly reflects that labels choose likely hits as singles (the release type is not independent of expected success). Because a few artists contribute very many songs, comparisons should give each artist equal weight rather than each song.

### **10.5 Cleaning (K1–K5)**

We clean the comparison group in five logged steps ("K" for comparison), mirroring the charting songs. The published extract and the database table `comparison_musicbrainz` stay unchanged; excluded artists are recorded with their reason in `comparison_artists`, and the cleaned values go into `comparison_clean`.

| Step | What | Responds to |
|---|---|---|
| K1 | Remove songs of placeholder artists | `[theatre]`, `[Disney]` (10.4) |
| K2 | Remove artists that charted only with songs released 5+ years earlier | reissues, not new songs (10.4) |
| K3 | Remove recordings whose main genre says they are not music | interviews etc. (10.4) |
| K4 | Exact dates on 1 January treated as known to the year only | same as M1 |
| K5 | Genres grouped into the same 10 families | same as M4 |

In [ ]:
CMP_CLEANING_LOG = []

def log_cmp_step(step, description, rows_before, rows_after, values_changed, justification):
    """Record one cleaning step of the comparison group (same idea as log_step in Section 5)."""
    CMP_CLEANING_LOG.append({"step": step, "description": description, "rows_before": int(rows_before),
                             "rows_after": int(rows_after), "values_changed": int(values_changed),
                             "justification": justification})
    print(f"[{step}] {description}\n      rows {rows_before:,} → {rows_after:,} | values changed: {values_changed:,}")

per_artist["exclusion_reason"] = np.select(
    [placeholder, per_artist["only_old_songs"]],
    ["placeholder artist (K1)", "charted only with songs released 5+ years earlier (K2)"], default=None)
per_artist["included"] = per_artist["exclusion_reason"].isna()

cmpc = cmp_raw.copy()
n0 = len(cmpc)
cmpc = cmpc[~cmpc["artist_mbid"].isin(per_artist.index[placeholder])]
log_cmp_step("K1", "Remove songs credited to placeholder artists ([theatre], [Disney])", n0, len(cmpc), 0,
             "MusicBrainz uses these names for cast and soundtrack recordings by many different performers; "
             "they are not artists. mb_07 skipped other placeholders but not these two.")

n1 = len(cmpc)
cmpc = cmpc[~cmpc["artist_mbid"].isin(per_artist.index[per_artist["only_old_songs"]])]
log_cmp_step("K2", "Remove artists that reached the chart only with songs released 5+ years earlier", n1, len(cmpc), 0,
             "E.g. Bing Crosby, Nat King Cole, Frank Sinatra (holiday classics) or Kate Bush (revival): their "
             "2000–2024 releases are mostly reissues, not new songs competing for the chart (same idea as M3).")

n2 = len(cmpc)
cmpc = cmpc[~cmpc["top_genre"].isin(NOT_MUSIC)]
log_cmp_step("K3", "Remove recordings whose main genre says they are not music", n2, len(cmpc), 0,
             "Interviews, spoken word and field recordings (10.4) are not songs that could chart; "
             "no charting song has these genres.")

day_dates = cmpc["date_precision"] == "day"
jan_first = day_dates & cmpc["first_release_date"].str.endswith("-01-01")
cmpc["release_date"] = pd.to_datetime(cmpc["first_release_date"].where(day_dates & ~jan_first), format="%Y-%m-%d")
cmpc["release_precision"] = cmpc["date_precision"].where(~jan_first, "year")
log_cmp_step("K4", "Treat exact dates on 1 January as known to the year only (as M1)", len(cmpc), len(cmpc),
             jan_first.sum(), f"{int(jan_first.sum()):,} dates on 1 January against about "
             f"{day_dates.sum() / 365.25:,.0f} expected: placeholders, as for the charting songs.")

cmpc["genre_family"] = cmpc["top_genre"].map(genre_family)
log_cmp_step("K5", "Group the detailed genres into broad families (rules of M4)", len(cmpc), len(cmpc),
             cmpc["top_genre"].notna().sum(), "Same rules as for the charting songs, so that both groups are comparable.")
charting_genres = set(read_table("song_musicbrainz")["top_genre"].dropna())
new_genres = (cmpc[~cmpc["top_genre"].isin(charting_genres) & cmpc["top_genre"].notna()]
              .groupby(["top_genre", "genre_family"]).size().rename("songs").reset_index()
              .sort_values("songs", ascending=False))
CMP_QA["genres_not_seen_among_charting_songs"] = {"genres": len(new_genres), "songs": int(new_genres["songs"].sum())}
print(f"Genres not seen among charting songs: {len(new_genres)} ({new_genres['songs'].sum():,} songs); "
      "the 15 most frequent and the family the rules give them:")
display(new_genres.head(15))
family_cmp = cmpc["genre_family"].value_counts(dropna=False).rename("songs").to_frame()
family_cmp["share"] = (family_cmp["songs"] / len(cmpc)).round(3)
family_cmp

**Observations:** after cleaning, the comparison group contains 176,210 songs by 2,009 artists. 56 artists were excluded (2 placeholders, 54 that charted only with old songs; 3,739 + 6,450 songs removed), 420 recordings that are not music were removed, and 37 of the 2,046 remaining artists have no comparison song released 2000–2024. 2,468 release dates on 1 January were set to year precision. 250 genres occur only among comparison songs (2,350 songs, 1.3%); inspecting the most frequent showed one gap in our rules (“brostep”, a dubstep style), which was added to the electronic family in 8.7 without changing any charting song.

**Interpretation:** the cleaned group consists of real songs by artists who were active in our period, measured with the same rules as the charting songs. Film composers (e.g. James Newton Howard, A. R. Rahman) and very prolific rappers remain, because by our definition they are charting lead artists; their weight is limited in the analysis by giving each artist equal weight (Section 11).

#### **Manual check of 30 comparison songs**

All results in Section 11 compare charting songs with this group, but so far the group has only been checked inside the data (counts, keys, duplicates). As for the charting songs (8.4d), we check a random sample by hand against the source: 30 songs used in Section 11 (included artists, first released 2000–2023), drawn with the same seed. For each, we check against independent sources and the Hot 100 archive (1) that it is a real song by this artist, not an interview, skit, medley or a different artist's recording, (2) that its first release date is plausible, and (3) that it is not one of the artist's charting songs under another title.

In [ ]:
from urllib.parse import quote

# 30 randomly selected comparison songs among those used in Section 11 (included artists, released 2000–2023),
# with links to look each one up on musicbrainz.org (recording search restricted to the artist's MBID).
used = cmpc[cmpc["artist_mbid"].isin(per_artist.index[per_artist["included"]])
            & cmpc["first_release_year"].between(2000, 2023)]
cmp_sample = (used.sample(n=30, random_state=MANUAL_CHECK_SEED)
                  .merge(per_artist[["artist_name"]], left_on="artist_mbid", right_index=True,
                         how="left", validate="many_to_one")
                  [["artist_name", "mb_title", "first_release_date", "first_release_type", "top_genre", "artist_mbid"]])
cmp_sample["musicbrainz_search"] = [
    "https://musicbrainz.org/search?type=recording&method=advanced&query="
    + quote(f'recording:"{t}" AND arid:{a}') for t, a in zip(cmp_sample["mb_title"], cmp_sample["artist_mbid"])]
cmp_sample = cmp_sample.drop(columns="artist_mbid").reset_index(drop=True)
CMP_QA["manual_check_sample"] = cmp_sample.astype(object).where(cmp_sample.notna(), None).to_dict("records")   # no NaN in JSON
pd.set_option("display.max_colwidth", 120)
cmp_sample

In [ ]:
# How common are titles that suggest an entry is not a separate song? Same patterns for both groups, on
# MusicBrainz titles of songs first released 2000–2023 (charting: matched songs; comparison: as used in Section 11).
NOT_A_SONG_PATTERNS = {
    "intro / outro / interlude / skit": r"\b(?:intro|outro|interlude|skit)s?\b",
    "live (in brackets or after a dash)": r"\(live\b|\blive\)|[-–] live\b",
    "medley":                            r"\bmedley\b",
    "two titles joined by ' / '":        r" / ",
}
def not_a_song_flags(titles):
    """One True/False column per pattern, plus 'any pattern'."""
    flags = pd.DataFrame({name: titles.str.contains(p, case=False, regex=True)
                          for name, p in NOT_A_SONG_PATTERNS.items()})
    flags["any pattern"] = flags.any(axis=1)
    return flags

charting_titles = mb.loc[(mb["match_method"] != "no match") & mb["first_release_year"].between(2000, 2023), "mb_title"]
title_patterns = pd.DataFrame({
    "comparison songs (%)": not_a_song_flags(used["mb_title"]).mean() * 100,
    "charting songs (%)":   not_a_song_flags(charting_titles).mean() * 100,
}).round(2)
title_patterns.loc["songs"] = [len(used), len(charting_titles)]
CMP_QA["title_patterns"] = title_patterns.to_dict()
title_patterns

**Manual check (30 randomly selected comparison songs, seed 2026).** We checked the 30 sampled comparison songs against independent sources (Wikipedia, streaming services, press) and the full Hot 100 archive; the `musicbrainz_search` links let any reader repeat the check. We found sources for 23 of them; for the other 7 we found no evidence against the record.

- **Real song by this artist: 25 / 30.** Three entries are not songs: a 15-second spoken promo intro (Elvis Presley, "Welcome to LidRock"), a live medley (Thalía) and a live introduction (A Tribe Called Quest). Two more are borderline: DJ Khaled's "(Intro) I'm So Grateful" and Meek Mill's "Lord Knows / Fighting Stronger" medley.
- **Plausible release date: 28 / 30.** Elvis Presley's "Just for Old Time's Sake" is a 1962 recording dated 2015 (a reissue), and Iyaz's "Friend" may carry an earlier EP date.
- **Never on the Hot 100: 29 / 30.** The Meek Mill medley contains "Lord Knows", which charted in July 2015.

**Interpretation:** the comparison group consists mostly of real songs that never charted, but 5 of the 30 sampled entries are intros, live tracks or medleys; thirty songs give only a rough estimate of this share. The title patterns above give a firmer lower bound: 3.0% of comparison songs, against 0.6% of charting songs, have an intro, live, medley or joined title (entries such as the Elvis promo, whose title says nothing, are not caught). Because such entries almost never chart, they make the share of songs that charted (Section 11) slightly too low, the opposite direction to the title rule (10.1); 11.6 repeats the main results without them. The reissue shows that an old recording can enter the comparison group with a later release date; we found one such case.

### **10.6 Store in the database**

In [ ]:
con.executescript("""
DROP TABLE IF EXISTS comparison_clean;          -- tables that refer to others are dropped first
DROP TABLE IF EXISTS song_lead_artist;
DROP TABLE IF EXISTS comparison_musicbrainz;
DROP TABLE IF EXISTS comparison_artists;

CREATE TABLE comparison_artists (
    artist_mbid       TEXT    PRIMARY KEY,
    artist_name       TEXT    NOT NULL,
    charting_songs    INTEGER NOT NULL CHECK (charting_songs >= 1),
    comparison_songs  INTEGER NOT NULL CHECK (comparison_songs >= 0),
    included          INTEGER NOT NULL CHECK (included IN (0, 1)),
    exclusion_reason  TEXT,
    CHECK ((included = 1) = (exclusion_reason IS NULL))
);

CREATE TABLE song_lead_artist (
    song_id      TEXT PRIMARY KEY REFERENCES songs(song_id),
    artist_mbid  TEXT NOT NULL REFERENCES comparison_artists(artist_mbid)
);

CREATE TABLE comparison_musicbrainz (
    comparison_id       TEXT    PRIMARY KEY CHECK (length(comparison_id) = 32),
    artist_mbid         TEXT    NOT NULL REFERENCES comparison_artists(artist_mbid),
    mb_title            TEXT    NOT NULL,
    n_recordings        INTEGER NOT NULL CHECK (n_recordings >= 1),
    first_release_type  TEXT,
    first_release_year  INTEGER NOT NULL CHECK (first_release_year BETWEEN 2000 AND 2024),
    first_release_date  TEXT    NOT NULL,
    date_precision      TEXT    NOT NULL CHECK (date_precision IN ('day', 'month', 'year')),
    genre_level         TEXT    CHECK (genre_level IN ('recording', 'release group', 'artist')),
    top_genre           TEXT,
    top_genres_votes    TEXT
);

CREATE TABLE comparison_clean (
    comparison_id      TEXT PRIMARY KEY REFERENCES comparison_musicbrainz(comparison_id),
    release_date       TEXT,
    release_precision  TEXT NOT NULL CHECK (release_precision IN ('day', 'month', 'year')),
    genre_family       TEXT
);
""")
artists_tbl = per_artist.rename_axis("artist_mbid").reset_index()[
    ["artist_mbid", "artist_name", "charting_songs", "comparison_songs", "included", "exclusion_reason"]]
to_sql_ready(artists_tbl).to_sql("comparison_artists", con, if_exists="append", index=False)
lead_raw[["song_id", "artist_mbid"]].to_sql("song_lead_artist", con, if_exists="append", index=False)
cmp_raw.to_sql("comparison_musicbrainz", con, if_exists="append", index=False)
to_sql_ready(cmpc[["comparison_id", "release_date", "release_precision", "genre_family"]]).to_sql(
    "comparison_clean", con, if_exists="append", index=False)
con.commit()

db_cmp = {
    "comparison_artists": q("SELECT COUNT(*) AS n FROM comparison_artists")["n"][0],
    "  of which included": q("SELECT SUM(included) AS n FROM comparison_artists")["n"][0],
    "song_lead_artist": q("SELECT COUNT(*) AS n FROM song_lead_artist")["n"][0],
    "comparison_musicbrainz": q("SELECT COUNT(*) AS n FROM comparison_musicbrainz")["n"][0],
    "comparison_clean": q("SELECT COUNT(*) AS n FROM comparison_clean")["n"][0],
    "foreign-key violations": len(q("PRAGMA foreign_key_check")),
    "clean songs of excluded artists": q("""SELECT COUNT(*) AS n FROM comparison_clean c
                                           JOIN comparison_musicbrainz m USING (comparison_id)
                                           JOIN comparison_artists a USING (artist_mbid)
                                           WHERE a.included = 0""")["n"][0],
}
CMP_QA["database_checks"] = {k: int(v) for k, v in db_cmp.items()}
display(pd.Series(db_cmp, name="value").to_frame())
if db_cmp["foreign-key violations"] or db_cmp["clean songs of excluded artists"]:
    raise ValueError("The comparison tables break a database rule.")

In [ ]:
# Same one-row-per-genre table for the comparison group, with the same checks as in 8.7.
comparison_genre_votes = genre_votes_long(cmp_raw, "comparison_id")
check_tie_rule(comparison_genre_votes, cmp_raw.set_index("comparison_id")["top_genre"], "comparison_id")
con.executescript("""
DROP TABLE IF EXISTS comparison_genre_votes;
CREATE TABLE comparison_genre_votes (
    comparison_id TEXT    NOT NULL REFERENCES comparison_musicbrainz(comparison_id),
    rank          INTEGER NOT NULL CHECK (rank BETWEEN 1 AND 5),
    genre         TEXT    NOT NULL,
    votes         INTEGER NOT NULL CHECK (votes >= 1),
    PRIMARY KEY (comparison_id, genre),
    UNIQUE (comparison_id, rank)
);
""")
comparison_genre_votes.to_sql("comparison_genre_votes", con, if_exists="append", index=False)
con.commit()
cmp_ties = tie_profile(comparison_genre_votes, "comparison_id")
CMP_QA["genre_ties"] = (cmp_ties.mean() * 100).round(1).to_dict()
print(f"Rows in comparison_genre_votes: {len(comparison_genre_votes):,} "
      f"| foreign-key violations: {len(q('PRAGMA foreign_key_check'))}")
(cmp_ties.mean() * 100).round(1).rename("% of comparison songs with a genre").to_frame()

**Observations:** the genre votes of the comparison songs follow the same rule as those of the charting songs (8.7) and are stored the same way (`comparison_genre_votes`, 560,659 rows). Among comparison songs with a genre, 30.9% have a tied top genre, 13.2% a tie between families and 36.7% a top genre from a single vote, more than for charting songs, because non-charting songs are tagged by fewer people.

**Interpretation:** genre is even less certain for the comparison group, so the genre comparison is repeated without these songs in 11.6.

### **10.7 Save the provenance, quality and cleaning record**

In [ ]:
CMP_PROV_FILE = OUTPUT_DIR / "provenance_quality_comparison.json"
CMP_PROV_FILE.write_text(json.dumps({
    "record_type": "provenance, quality assessment and cleaning log – comparison group (Job B)",
    "musicbrainz_dump": MB_DUMP,
    "export_repository": f"https://github.com/{MB_REPO}",
    "export_commit": CMP_COMMIT,
    "export_files_sha256": CMP_FILES,
    "accessed_at_utc": CMP_ACCESS_TIME_UTC.isoformat(),
    "quality": CMP_QA,
    "cleaning_steps": CMP_CLEANING_LOG,
    "excluded_artists": artists_tbl.loc[~artists_tbl["included"], ["artist_name", "exclusion_reason"]].to_dict("records"),
}, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
download_file(CMP_PROV_FILE)

### **10.8 Summary of Section 10**

**Observations:** a comparison group of 186,819 songs by the 2,102 lead artists of our matched charting songs, never charted and first released 2000–2024, was extracted reproducibly from the same MusicBrainz database (10.1) and verified (10.2–10.3). No comparison song has the same artist and title as a charting song. After removing placeholder artists, artists that charted only with old songs, and recordings that are not music, 176,210 songs by 2,009 artists remain (10.5).

**Interpretation:** the group gives us what the chart data alone cannot: songs that did not reach the chart, by artists who demonstrably could. Three limitations follow for Section 11: non-charting songs are less well documented (fewer exact dates; genres more often from the artist alone), so only equally measured variables can be compared; release type is partly a consequence of expected success (labels release likely hits as singles); and catalogue sizes differ widely between artists, so each artist must get equal weight. Because every artist in the group has charted, the comparison cannot test whether being a known artist helps a song chart.

## **11. What distinguishes the songs that charted?**

This section answers our research question directly: among songs by the same artists, released in the same years, how do the songs that reached the Hot 100 differ from those that did not?

**Measure.** The **chart rate**: the share of songs that charted.

**Design choices** (decision log, rows 18, 21, 22, 27):
- **Same artists, same years.** Only songs by the 2,046 included lead artists (Section 10), first released 2000–2023 in both groups. 2023 is the last release year that gives every song at least two years to chart, even songs whose release is known only to the year.
- **Each artist counts equally.** Catalogue sizes differ widely (10.4), so shares are computed per artist and then averaged. Only artists with at least 10 songs in the group being compared are used; with fewer, a single song swings an artist's share by 10 points or more.
- **Within-artist comparisons.** For release type and genre, we compare each category's chart rate with *the same artist's own average*. A positive difference means: when an artist releases a song of this kind, it charts more often than that artist's songs usually do. This removes differences between artists, such as how famous they are.
- **Uncertainty.** Next to each average difference we give an approximate 95% range (the average ± 1.96 standard errors across artists). If the range does not include 0, the difference is unlikely to be explained by chance variation between artists. This is a simple descriptive measure, not a model.
- **Robustness.** The main comparisons are repeated under alternative choices in 11.6.
- **Exploratory.** 11.7 looks at the release month, a question added after the main analysis; it is reported separately, as a secondary result to be confirmed.

As before, every result is followed by observations and, separately, interpretations. All comparisons describe associations, not causes.

### **11.1 The songs compared, and how many charted**

In [ ]:
COMPARISON_RESULTS = {}
LAST_RELEASE_YEAR = 2023      # every song, even one known only to the year, has at least 2 years to chart
MIN_ARTIST_SONGS = 10         # artist-level shares need at least this many songs

SONG_COLUMNS = """artist_mbid, first_release_year, first_release_type, genre_level,
                  c.genre_family, c.release_date, c.release_precision"""
charting = read_table(f"""
    SELECT m.song_id AS song_key, {SONG_COLUMNS}, c.flag_release_after_chart
    FROM song_musicbrainz m
    JOIN song_lead_artist l    USING (song_id)
    JOIN comparison_artists a  USING (artist_mbid)
    JOIN song_mb_clean c       USING (song_id)
    WHERE a.included = 1""")
not_charting = read_table(f"""
    SELECT m.comparison_id AS song_key, {SONG_COLUMNS}
    FROM comparison_musicbrainz m
    JOIN comparison_clean c USING (comparison_id)""")

# Concatenate: same kind of observation (a song of a lead artist), with a column saying where each row comes from.
all_songs = pd.concat([charting.assign(charted=True),
                       not_charting.assign(charted=False, flag_release_after_chart=False)], ignore_index=True)
if not all_songs["song_key"].is_unique:
    raise ValueError("A song appears twice after concatenating the two groups.")

all_songs = all_songs.rename(columns={"first_release_year": "release_year"})
all_songs["release_type"] = all_songs["first_release_type"].where(
    all_songs["first_release_type"].isin(["Single", "Album", "EP"]), "Other / unknown")
all_songs["period"] = pd.cut(all_songs["release_year"].astype(float), [1999, 2004, 2009, 2014, 2019, LAST_RELEASE_YEAR],
                             labels=["2000–04", "2005–09", "2010–14", "2015–19", f"2020–{str(LAST_RELEASE_YEAR)[2:]}"])

ledger = []
def add_to_ledger(label, df, reason):
    ledger.append({"step": label, "charting songs": int(df["charted"].sum()),
                   "non-charting songs": int((~df["charted"]).sum()), "artists": df["artist_mbid"].nunique(),
                   "reason": reason})
add_to_ledger("songs of included lead artists", all_songs, "Sections 8 and 10; charting songs need a MusicBrainz match")
songs_2000_2023 = all_songs[all_songs["release_year"].between(2000, LAST_RELEASE_YEAR)].copy()
add_to_ledger(f"first released 2000–{LAST_RELEASE_YEAR}", songs_2000_2023,
              "same release years for both groups, each song observed for at least 2 years")
cmp_ledger = pd.DataFrame(ledger)
COMPARISON_RESULTS["selection_ledger"] = cmp_ledger.to_dict("records")
display(cmp_ledger)

artist_rates = songs_2000_2023.groupby("artist_mbid")["charted"].agg(songs="size", chart_rate="mean")
big = artist_rates[artist_rates["songs"] >= MIN_ARTIST_SONGS]
overall = {
    "songs": len(songs_2000_2023),
    "% of songs that charted (all songs pooled)": round(songs_2000_2023["charted"].mean() * 100, 1),
    f"artists with at least {MIN_ARTIST_SONGS} songs": len(big),
    "% charted, average over these artists": round(big["chart_rate"].mean() * 100, 1),
    "% charted, median artist": round(big["chart_rate"].median() * 100, 1),
    "% charted, 25th–75th percentile of artists": f"{big['chart_rate'].quantile(.25) * 100:.1f}–{big['chart_rate'].quantile(.75) * 100:.1f}",
}
COMPARISON_RESULTS["overall"] = overall
pd.Series(overall, name="value").to_frame()

**Observations:** we compare 9,798 charting and 168,540 non-charting songs by 2,029 artists, first released 2000–2023. Of all these songs, 5.5% reached the Hot 100. Among the 1,884 artists with at least 10 songs, the average artist placed 6.1% of their songs on the chart, the median artist 3.5%, and the middle half of artists between 1.7% and 7.5%.

**Interpretation:** even for artists who have proven they can reach the Hot 100, most of their songs never do: the typical artist charts with only about one song in thirty. Reaching the chart is the exception, not the rule. The exact level depends on how completely MusicBrainz lists each artist's catalogue, so comparisons *within* these data (below) are more reliable than this overall level. The level is also probably slightly too high, because the comparison group excludes every song whose title equals a Billboard title by any artist (10.1); in the worst case, 4.2% instead of 5.5% of songs charted (11.6).

### **11.2 Has the chart rate changed over time?**

In [ ]:
def artist_weighted_rate(df, by):
    """Share of songs that charted per group, giving each artist equal weight (artists with enough songs only)."""
    cells = df.groupby([by, "artist_mbid"], observed=True)["charted"].agg(songs="size", rate="mean").reset_index()
    cells = cells[cells["songs"] >= MIN_ARTIST_SONGS]
    return cells.groupby(by, observed=True).agg(artists=("rate", "size"), songs=("songs", "sum"),
                                                artist_weighted_pct=("rate", lambda r: round(r.mean() * 100, 1)))

by_period_rate = artist_weighted_rate(songs_2000_2023, "period")
by_period_rate["pooled_pct"] = (songs_2000_2023.groupby("period", observed=True)["charted"].mean() * 100).round(1)
COMPARISON_RESULTS["chart_rate_by_period"] = by_period_rate.reset_index().astype({"period": str}).to_dict("records")
by_period_rate

**Observations:** the artist-weighted chart rate stayed between 5.0% and 6.0% in every period. The pooled rate rose from 4.7–5.2% before 2010 to 7.2% in 2020–2023.

**Interpretation:** although about twice as many songs entered the chart per year in the 2020s as in the early 2000s (7.2), the typical artist's chance of placing a given song on the chart barely changed. That the pooled rate rose while the artist-weighted rate did not suggests that, recently, artists with many songs charted a larger share of them, which fits the album "bursts" of 7.6 and 9.6. A limitation: the share also depends on how completely MusicBrainz documents each period's releases, which may differ over time.

### **11.3 Single or album track?**

In [ ]:
def within_artist(df, column, min_songs=MIN_ARTIST_SONGS):
    """For each category: how much more (or less) often an artist's songs in this category charted than
    that same artist's songs overall. Only artists with at least min_songs songs and at least two categories
    count; each artist has equal weight."""
    d = df.dropna(subset=[column])
    d = d[d.groupby("artist_mbid")["charted"].transform("size") >= min_songs]
    d = d[d.groupby("artist_mbid")[column].transform("nunique") >= 2]
    artist_rate = d.groupby("artist_mbid")["charted"].mean()
    cells = d.groupby(["artist_mbid", column], observed=True)["charted"].agg(songs="size", rate="mean").reset_index()
    cells["difference"] = cells["rate"] - cells["artist_mbid"].map(artist_rate)
    summary = cells.groupby(column, observed=True).agg(
        artists=("rate", "size"), songs=("songs", "sum"),
        artist_weighted_pct=("rate", "mean"), difference_pts=("difference", "mean"),
        sd=("difference", "std"), share_above=("difference", lambda x: (x > 0).mean()))
    margin = 1.96 * summary["sd"] / np.sqrt(summary["artists"])        # approximate 95% range of the mean
    summary["artist_weighted_pct"] = (summary["artist_weighted_pct"] * 100).round(1)
    summary["difference_pts"] = (summary["difference_pts"] * 100).round(1)
    summary["range_pts"] = (margin * 100).round(1)
    summary["approx_95pct_range"] = [f"{d - m:+.1f} to {d + m:+.1f}"
                                     for d, m in zip(summary["difference_pts"], summary["range_pts"])]
    summary["% of artists above own average"] = (summary["share_above"] * 100).round(0)
    return summary.drop(columns=["sd", "share_above"]).sort_values("difference_pts", ascending=False)

def plot_within_artist(summary, title, name):
    """Horizontal bars: difference from the artist's own average, with the approximate 95% range."""
    s = summary.sort_values("difference_pts")
    fig, ax = plt.subplots(figsize=(9, 0.5 * len(s) + 1.5))
    ax.barh([str(i) for i in s.index], s["difference_pts"], xerr=s["range_pts"], color=COLOR_SECOND, ecolor="black", capsize=3)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel("Percentage points above (+) or below (−) the same artist's average")
    ax.grid(axis="x", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    save_figure(fig, name)
    plt.show()

by_type_within = within_artist(songs_2000_2023, "release_type")
COMPARISON_RESULTS["release_type_within_artist"] = by_type_within.reset_index().to_dict("records")
display(by_type_within)
plot_within_artist(by_type_within, "How often songs chart, by type of first release (same artists)", "11_3_release_type_within_artist")

type_share = (pd.crosstab(songs_2000_2023["release_type"], songs_2000_2023["charted"], normalize="columns") * 100).round(1)
type_share.columns = ["non-charting songs (%)", "charting songs (%)"]
type_share

**Observations:** songs first released as singles charted far more often than the same artists' other songs: on average 17.5% of an artist's singles reached the Hot 100, 11.3 percentage points above the artist's own average (approximate range +10.4 to +12.2), and this held for 60% of artists. Album tracks (4.5%), EP tracks (3.1%) and other releases (3.3%) charted less often than the artist's average. Singles make up 34% of charting songs but only 10% of non-charting songs.

**Interpretation:** release as a single is the strongest difference we found between charting and non-charting songs of the same artist. It should not be read as "releasing a song as a single makes it chart". Artists and labels usually choose as singles the songs they expect to succeed, and then promote them (radio, playlists, videos); our data record neither this choice nor the promotion. The difference may therefore come from the format itself, from which songs were chosen, or from how they were promoted, and our data cannot separate these. It is consistent with 9.6, where singles and album tracks also differed in how quickly they entered.

### **11.4 Which genres chart more often than the same artist's other songs?**

An artist-level genre is the same for all of an artist's songs, so it cannot distinguish songs of the same artist (10.4). This comparison therefore uses only songs **with a genre of their own** (from the recording or from the artist's own release), in both groups. Only families with at least 30 artists are shown.

In [ ]:
own_genre = songs_2000_2023[songs_2000_2023["genre_level"].isin(["recording", "release group"])]
print(f"Songs with a genre of their own: {len(own_genre):,} of {len(songs_2000_2023):,} "
      f"(charting {own_genre['charted'].mean() * 100:.1f}% of them)")
by_genre_within = within_artist(own_genre, "genre_family")
by_genre_within = by_genre_within[by_genre_within["artists"] >= 30]
COMPARISON_RESULTS["genre_within_artist"] = by_genre_within.reset_index().to_dict("records")
display(by_genre_within)
plot_within_artist(by_genre_within, "How often songs chart, by genre family (same artists, songs with their own genre)",
                   "11_4_genre_within_artist")

**Observations:** among songs with a genre of their own, an artist's songs tagged as pop charted 12.1 percentage points more often than that artist's average (approximate range +10.2 to +14.0; 812 artists), and R&B/soul (+4.1) and rock/alternative (+3.7) songs also charted more often, although these two differences are not robust (11.6). For country, electronic/dance, hip hop, Latin and "other", the range includes 0, so no clear difference. Holiday songs charted 5.0 points less often than the artist's average.

**Interpretation:** when the same artist releases songs in different genres, the ones tagged pop are clearly the most likely to chart. Two cautions apply. First, tags are added by volunteers after release, and a song that became a hit may be tagged "pop" partly *because* it was popular, which would inflate this difference. Second, this compares songs *within* artists: it does not say that pop artists chart more often than hip hop artists (hip hop is the largest family among charting songs, 9.5). Holiday songs (mostly tracks on Christmas albums) rarely chart, apart from a few classics.

### **11.5 Revisiting the threshold: the chance that a song will still chart**

In 9.4 we could only study songs that did chart. With the comparison group we can now test our hypothesis as stated: **beyond a certain time after release, is a song much less likely to *ever* enter the chart?** For each chart *k* after release, we take all songs that had not charted before chart *k* (charting and non-charting) and ask what share of them still charted later. We use songs with an exact release date that can be followed for five years (released 2000–2020), measured in charts waited as in Section 9, with songs whose release date is impossible (M2) excluded, as in Section 9.

In [ ]:
dated_songs = songs_2000_2023[(songs_2000_2023["release_precision"] == "day") & songs_2000_2023["release_date"].notna()
                              & ~songs_2000_2023["flag_release_after_chart"]].copy()
dated_songs["first_possible_pos"] = np.searchsorted(tracking_end.values, dated_songs["release_date"].values, side="left")
entry = read_table("SELECT song_id AS song_key, first_chart_date FROM song_summary")
dated_songs = dated_songs.merge(entry, on="song_key", how="left", validate="one_to_one")
dated_songs["waited"] = (dated_songs["first_chart_date"].map(chart_position) - dated_songs["first_possible_pos"])
followed = dated_songs[(dated_songs["first_possible_pos"] + LONG_FOLLOW_UP - 1 <= last_pos)
                       & (dated_songs["waited"].isna() | (dated_songs["waited"] < LONG_FOLLOW_UP))].copy()
followed["waited"] = followed["waited"].clip(lower=0)          # NaN = never charted
print(f"Songs with an exact release date and 5 years of charts after it: {len(followed):,} "
      f"({int(followed['charted'].sum()):,} charted), released "
      f"{followed['release_year'].min()}–{followed['release_year'].max()}")

def still_possible(waited, charts):
    """For each chart k: of the songs that had not charted before chart k, the share that ever charted later."""
    rows = []
    for k in range(charts):
        not_yet = waited.isna() | (waited >= k)
        rows.append({"chart": k + 1, "songs not yet charted": int(not_yet.sum()),
                     "% that still charted later": round(waited[not_yet].notna().mean() * 100, 2)})
    return pd.DataFrame(rows)

later = still_possible(followed["waited"], FOLLOW_UP_CHARTS)
by_artist = []
for k in [0, 1, 2, 4, 12, 25, 51]:
    nyt = followed[followed["waited"].isna() | (followed["waited"] >= k)]
    rates = nyt.groupby("artist_mbid")["waited"].agg(n="size", rate=lambda w: w.notna().mean())
    by_artist.append(round(rates.loc[rates["n"] >= MIN_ARTIST_SONGS, "rate"].mean() * 100, 2))
key_rows = later.iloc[[0, 1, 2, 4, 12, 25, 51]].assign(**{"% artist-weighted": by_artist})
COMPARISON_RESULTS["chance_to_still_chart"] = later.to_dict("records")
COMPARISON_RESULTS["chance_to_still_chart_key_points"] = key_rows.to_dict("records")
display(key_rows)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.step(later["chart"], later["% that still charted later"], where="post", color=COLOR_MAIN)
ax.set_ylim(bottom=0)
style_axes(ax, "Chance that a song which has not charted yet will still chart",
           "Weekly chart since the song's first possible chart (1 = at release)",
           "% of songs not yet charted that charted later")
save_figure(fig, "11_5_chance_to_still_chart")
plt.show()

**Observations:** at release, 5.2% of these songs went on to chart. Of the songs that missed their first possible chart, 3.3% still charted later; of those that missed their first two, 2.7%; after about a quarter of a year without entering (12 charts), 1.8%; after half a year, 1.1%; after a year, 0.45%. The artist-weighted figures are slightly higher but follow the same pattern (5.6% at release, 0.7% after a year).

**Interpretation:** this confirms our hypothesis with songs that did not chart included: the chance that a song ever reaches the Hot 100 halves once it has missed its first two possible charts, and after a year without charting it is about eleven times lower than at release. Together with 9.4, the first two charts after release are decisive; after them, the chance keeps shrinking steadily rather than dropping to zero at a fixed point, so the "threshold" is a sharp early drop followed by a slow decline.

**Limitations:** only songs with an exact release date are included, which is more often the case for charting songs (93%) than for non-charting songs (86%, 10.4); the shares are therefore slightly too high, although the shape of the curve should not change much. Songs released before 2000 and after 2020 are not included. The release date of an album track is the album's, which may be months before the song was promoted as a single; among songs first released as singles, dated by the single itself, the early drop is even sharper (11.6). The title rule of the comparison group (10.1) makes every point of the curve slightly too high, by at most about a quarter, without changing its shape (11.6).

### **11.6 Robustness checks**

The comparisons of 11.3–11.5 depend on choices we made ourselves: what counts as success, the minimum number of songs per artist, keeping Glee Cast, how genres and release dates are taken from MusicBrainz, and how the comparison group was built. 7.4 and 7.7 also showed that a large share of recent chart entries stayed only one week, and genre and release type may overlap (pop songs are more often released as singles). We therefore repeat the main comparisons, changing one choice at a time:

- **Success = at least 2 weeks on the chart:** songs that charted for one week only count as not charting.
- **At least 5 / at least 20 songs per artist** instead of 10.
- **Without Glee Cast** (7.5).
- **Genre within one release type:** the genre comparison of 11.4 among singles only and among album tracks only, so that genre is not confused with release type (artists with at least 5 such songs, because fewer artists have 10 singles).
- **Without uncertain genres:** the genre comparison of 11.4 without songs whose top genre is tied with a genre of another family, or rests on a single vote (8.7).
- **Songs first released as singles:** the time-to-chart results of 9.2 and 11.5 for songs whose first release is the single itself, because an album track's release date may be months before the song was promoted.
- **The title rule of the comparison group:** the largest possible effect of excluding every title equal to a Billboard title by any artist (10.1).
- **Without entries that are not songs:** the main results without songs whose title marks an intro, a live track, a medley or two joined titles (10.5), removed from both groups.

The last four checks were added in response to the second peer review. If the conclusions hold in every variant, they do not depend on these choices. Categories with fewer than 30 artists are shown as "–".

In [ ]:
# Variants of the main comparison: each changes one of our own choices.
weeks = read_table("SELECT song_id AS song_key, weeks_in_window FROM song_summary")
base = songs_2000_2023.merge(weeks, on="song_key", how="left", validate="one_to_one")   # one row per song on both sides
if len(base) != len(songs_2000_2023):
    raise ValueError("Join contract broken: songs were added or lost.")
two_weeks = base.assign(charted=base["charted"] & (base["weeks_in_window"] >= 2))

glee_song_ids = read_table("songs").loc[lambda s: s["display_performer"].str.lower().str.startswith("glee cast"), "song_id"]
glee_artists = set(base.loc[base["song_key"].isin(glee_song_ids), "artist_mbid"])
without_glee = base[~base["artist_mbid"].isin(glee_artists)]
print(f"Charting songs that charted only one week: {int((base['charted'] & (base['weeks_in_window'] == 1)).sum()):,} "
      f"of {int(base['charted'].sum()):,} | Glee Cast artist entries removed: {len(glee_artists)} "
      f"({len(base) - len(without_glee):,} songs)")

VARIANTS = [  # label, songs, minimum songs per artist
    ("Main result (11.3–11.4)",              base,         MIN_ARTIST_SONGS),
    ("Success = at least 2 weeks on chart",  two_weeks,    MIN_ARTIST_SONGS),
    ("Artists with at least 5 songs",        base,         5),
    ("Artists with at least 20 songs",       base,         20),
    ("Without Glee Cast",                    without_glee, MIN_ARTIST_SONGS),
]
OWN_GENRE = ["recording", "release group"]
SHOWN_TYPES  = ["Single", "Album"]
SHOWN_GENRES = ["Pop", "R&B / soul", "Rock / alternative", "Hip hop / rap", "Holiday"]

def cell(summary, category):
    """'difference (range)' for one category, or '–' if the category has fewer than 30 artists."""
    if category not in summary.index or summary.loc[category, "artists"] < 30:
        return "–"
    row = summary.loc[category]
    return f"{row['difference_pts']:+.1f} ({row['approx_95pct_range']})"

rows = {}
for label, df, min_songs in VARIANTS:
    types  = within_artist(df, "release_type", min_songs)
    genres = within_artist(df[df["genre_level"].isin(OWN_GENRE)], "genre_family", min_songs)
    rows[label] = {**{t: cell(types, t) for t in SHOWN_TYPES}, **{g: cell(genres, g) for g in SHOWN_GENRES},
                   "artists": int(df.groupby("artist_mbid").size().ge(min_songs).sum())}
robustness = pd.DataFrame(rows).T
COMPARISON_RESULTS["robustness"] = robustness.reset_index().rename(columns={"index": "variant"}).to_dict("records")
print("Difference from the same artist's average, in percentage points (approximate 95% range):")
display(robustness)

# Genre and release type overlap: is the genre result only a single-vs-album effect?
own = base[base["genre_level"].isin(OWN_GENRE)]
by_type_genre = {}
for t in SHOWN_TYPES:
    g = within_artist(own[own["release_type"] == t], "genre_family", 5)
    by_type_genre[f"{t}s only" if t == "Single" else "Album tracks only"] = {
        **{x: cell(g, x) for x in SHOWN_GENRES},
        "artists": int(own[own["release_type"] == t].groupby("artist_mbid").size().ge(5).sum())}
genre_within_type = pd.DataFrame(by_type_genre).T
COMPARISON_RESULTS["genre_within_release_type"] = genre_within_type.reset_index().rename(columns={"index": "songs"}).to_dict("records")
print("Genre comparison within one release type (artists with at least 5 such songs):")
genre_within_type


**Observations:**
- 2,409 of the 9,798 charting songs (25%) charted for only one week. Counting them as not charting lowers the difference for singles from +11.3 to +9.8 points and for pop from +12.1 to +11.1; both ranges stay far above 0. R&B/soul (+3.1) and rock/alternative (+2.7) shrink but stay above 0.
- With at least 5 or at least 20 songs per artist instead of 10, no difference changes by more than 0.8 points.
- Without Glee Cast (one artist, 502 songs) nothing changes by more than 0.1 points.
- Hip hop's range includes 0 and holiday songs chart less often in every variant.
- **Within one release type**, pop songs still chart more often than the same artist's other songs: +7.5 points among singles (range +5.2 to +9.8) and +7.7 among album tracks (+5.9 to +9.5), against +12.1 overall. R&B/soul and rock/alternative show no clear difference among singles (ranges include 0) and a small one among album tracks (+2.2 each, R&B's range touching 0). Holiday songs chart less often in both.

**Interpretation:** our two main results, singles and pop songs charting more often than the same artist's other songs, do not depend on our thresholds, on the many one-week entries of the streaming era, or on Glee Cast. Roughly a third of pop's advantage comes from pop songs being released as singles more often, but pop songs chart more often even among songs of the same release type, so genre and release type are two separate associations. The smaller differences for R&B/soul and rock/alternative are not robust: they shrink with the stricter definition and largely disappear within singles, so we do not treat them as findings.

**Limitations:** these checks re-use the same data, so they cannot correct problems of measurement such as genre tags that may follow success (11.4, 8.4d). The within-type comparison uses fewer artists (820 with at least 5 singles), so its ranges are wider.

In [ ]:
# Genre result without songs whose genre assignment is uncertain (8.7): a tie between two families,
# or a top genre that rests on a single vote.
all_ties = pd.concat([tie_profile(song_genre_votes, "song_id"),
                      tie_profile(comparison_genre_votes, "comparison_id")]).rename_axis("song_key").reset_index()
own_t = base[base["genre_level"].isin(OWN_GENRE)].merge(all_ties, on="song_key", how="left", validate="one_to_one")
GENRE_VARIANTS = [
    ("Main result (11.4)",                       own_t),
    ("Without ties between families",            own_t[~own_t["tie_across_families"]]),
    ("Without top genres from a single vote",    own_t[~own_t["top_from_one_vote"]]),
    ("Without both",                             own_t[~own_t["tie_across_families"] & ~own_t["top_from_one_vote"]]),
]
genre_rows = {}
for label, df in GENRE_VARIANTS:
    g = within_artist(df, "genre_family")
    genre_rows[label] = {**{x: cell(g, x) for x in SHOWN_GENRES}, "songs": len(df),
                         "charting songs": int(df["charted"].sum())}
genre_certainty = pd.DataFrame(genre_rows).T
COMPARISON_RESULTS["genre_without_uncertain_tags"] = genre_certainty.reset_index().rename(columns={"index": "variant"}).to_dict("records")
print("Share of songs with a genre of their own (%):")
display((own_t.groupby("charted")[["tie_across_families", "top_from_one_vote"]].mean() * 100).round(1)
        .rename(index={False: "non-charting", True: "charting"}))
print("Difference from the same artist's average, in percentage points (approximate 95% range):")
genre_certainty

**Observations (uncertain genres):**
- Among songs with a genre of their own, 13.3% of charting and 17.2% of non-charting songs have a top genre tied with a genre of another family, and 28.5% and 48.5% a top genre that rests on a single vote: non-charting songs are tagged by fewer people.
- Pop charts more often than the same artist's average in every variant: +13.2 points without ties between families, +11.1 without single-vote genres, +12.2 without both (all ranges far above 0). Holiday songs chart less often in every variant (−5.2 to −6.9).
- Rock/alternative falls from +3.7 to between +0.1 and +1.4, with ranges that include 0. R&B/soul stays positive (+3.9 to +6.9), with wider ranges; hip hop's range includes 0 throughout.

**Interpretation:** the pop result does not depend on how ties are broken or on genres with a single vote. The small rock/alternative difference came partly from such uncertain tags, which confirms that it is not a finding. R&B/soul does not hold under the other checks of this section, so we still do not treat it as one.

**Limitation:** removing single-vote genres keeps only about half of the songs (54,863 of 103,171), mostly the better-documented ones, so these variants describe a better-tagged subset rather than all songs.

In [ ]:
# Time to chart for songs first released as singles. For an album track, the first release date is the
# album's, which may be months before the song was promoted as a single; for a single, it is the single itself.
types = read_table("SELECT song_id, first_release_type FROM song_musicbrainz")
lt = long_term.merge(types, on="song_id", how="left", validate="one_to_one")     # one type per song
def entry_shares(w):
    return pd.Series({"songs": len(w), "% on 1st possible chart": round((w == 0).mean() * 100, 1),
                      "% within 4 charts": round((w <= 3).mean() * 100, 1),
                      "% within 26 charts": round((w <= 25).mean() * 100, 1),
                      "% after more than a year": round((w > 51).mean() * 100, 1)})
charting_by_type = pd.DataFrame({"all charting songs (9.2)": entry_shares(lt["waited"]),
                                 "first released as a single": entry_shares(lt.loc[lt["first_release_type"] == "Single", "waited"]),
                                 "first released on an album": entry_shares(lt.loc[lt["first_release_type"] == "Album", "waited"])})
print("Charting songs followed for five years (as in 9.2):")
display(charting_by_type)

curves = {"all songs (11.5)": followed}
curves.update({f"first released as {'a single' if t == 'Single' else 'an album track'}": followed[followed["release_type"] == t]
               for t in ["Single", "Album"]})
still_by_type = {}
for label, df in curves.items():
    s = still_possible(df["waited"], FOLLOW_UP_CHARTS).set_index("chart")["% that still charted later"]
    still_by_type[label] = {"songs": len(df), "at release": s[1], "after missing 2 charts": s[3],
                            "after 12 charts": s[13], "after a year": s[52],
                            "drop after 2 charts (%)": round((1 - s[3] / s[1]) * 100, 0)}
still_by_type = pd.DataFrame(still_by_type)
COMPARISON_RESULTS["time_to_chart_by_release_type"] = {"charting": charting_by_type.to_dict(), "still_chart": still_by_type.to_dict()}
print("% of songs not yet charted that charted later (as in 11.5):")
still_by_type

**Observations (songs first released as singles):**
- Among charting songs followed for five years, those first released as singles entered faster than album-first songs: 46% on their first possible chart (34% of album tracks), 67% within four charts (48%), and only 3.3% more than a year after release (9.8%).
- Including songs that never charted, a single's chance of ever charting falls from 18.5% at release to 8.1% once it has missed its first two possible charts (−56%) and to 0.80% after a year; for album tracks it falls from 3.8% to 2.2% (−43%) and 0.39%. Over all songs the drop is 49% (11.5).

**Interpretation:** the peer reviewers' concern is partly confirmed: late entries are three times more common among songs first released on an album, which fits album release dates that come before the moment a song was promoted (although some album tracks may also genuinely climb later). But the threshold does not come from album dates: for singles, dated by their own release, the early drop is even sharper than for all songs. Our hypothesis therefore holds most clearly with the most reliable release dates. We keep one date rule for both groups (decision log row 31), because using single dates only where they exist would measure some songs differently from others.

**Limitation:** singles are a selected group (chosen and promoted as likely hits), and even a single's release date may precede its radio promotion.

In [ ]:
# The comparison group excludes every title that equals a Billboard title by ANY artist (10.1). This removes only
# non-charting candidates, so it can only make the chart rate too high. Worst case: every recording removed by this
# rule was a wrongly removed non-charting song, turned into songs at the same rate as the recordings that were kept.
REMOVED_SAME_TITLE, RECORDINGS_KEPT = 230_028, 725_119          # run_log_comparison.txt (table in 10.1)
extra = (RECORDINGS_KEPT + REMOVED_SAME_TITLE) / RECORDINGS_KEPT   # factor for the number of non-charting songs
n_chart, n_non = int(songs_2000_2023["charted"].sum()), int((~songs_2000_2023["charted"]).sum())

def still_worst_case(df, k):
    """11.5 curve at chart k if every non-charting song counted `extra` times."""
    not_yet = df["waited"].isna() | (df["waited"] >= k - 1)
    charted_later = df.loc[not_yet, "waited"].notna().sum()
    return round(charted_later / (charted_later + extra * df.loc[not_yet, "waited"].isna().sum()) * 100, 2)

reported = later.set_index("chart")["% that still charted later"]     # the 11.5 curve
title_bound = pd.DataFrame({
    "as reported": [round(n_chart / (n_chart + n_non) * 100, 1), reported[1], reported[3], reported[52]],
    "worst case": [round(n_chart / (n_chart + extra * n_non) * 100, 1)] + [still_worst_case(followed, k) for k in (1, 3, 52)],
}, index=["% of songs that charted (11.1)", "still charted, at release (11.5)",
          "still charted, after missing 2 charts", "still charted, after a year"])
title_bound["worst case / as reported"] = (title_bound["worst case"] / title_bound["as reported"]).round(2)
COMPARISON_RESULTS["title_rule_worst_case"] = {"factor": round(extra, 3), **title_bound.to_dict()}
print(f"Worst case: {extra:.2f} times as many non-charting songs")
title_bound

**Observations (title rule):** in the worst case, the comparison group would contain 1.32 times as many non-charting songs. The share of songs that charted would then be 4.2% instead of 5.5%, and the 11.5 curve would start at 4.0% instead of 5.2%, reach 2.1% instead of 2.7% after two missed charts, and 0.34% instead of 0.45% after a year. Every value is multiplied by about the same factor (0.76–0.77).

**Interpretation:** the title rule can only make chart rates too high, and by at most about a quarter. Because the added songs would never chart, they enlarge every point of the curve alike, so the shape of 11.5 does not change: the chance still halves after two missed charts and is still about eleven times lower after a year. The worst case is pessimistic, because part of the removed recordings are other versions of the artists' own hits, which are rightly excluded.

**Limitation:** this bound covers overall levels only. If titles shared with Billboard hits were more common in some genres or release types, the within-artist comparisons of 11.3–11.4 could also shift; testing this requires re-running the extraction with the rule limited to the same artist (13.4).

In [ ]:
# Main results without entries whose title suggests they are not a separate song (10.5), removed from BOTH groups.
titles_all = pd.concat([mb.set_index("song_id")["mb_title"], cmp_raw.set_index("comparison_id")["mb_title"]])
not_song = not_a_song_flags(titles_all.reindex(base["song_key"]).fillna(""))["any pattern"].values
songs_only = base[~not_song]
rate_all  = base["charted"].mean() * 100
rate_only = songs_only["charted"].mean() * 100
types_only  = within_artist(songs_only, "release_type")
genres_only = within_artist(songs_only[songs_only["genre_level"].isin(OWN_GENRE)], "genre_family")
not_song_check = pd.DataFrame({
    "main result": [f"{rate_all:.1f}%", cell(within_artist(base, "release_type"), "Single"),
                    cell(within_artist(base[base["genre_level"].isin(OWN_GENRE)], "genre_family"), "Pop"), f"{len(base):,}"],
    "without these titles": [f"{rate_only:.1f}%", cell(types_only, "Single"), cell(genres_only, "Pop"), f"{len(songs_only):,}"],
}, index=["% of songs that charted (11.1)", "Single (difference, range)", "Pop (difference, range)", "songs"])
COMPARISON_RESULTS["without_not_a_song_titles"] = not_song_check.to_dict()
print(f"Removed: {int(not_song.sum()):,} songs ({int((not_song & base['charted'].values).sum()):,} of them charting)")
not_song_check

**Observations (entries that are not songs):** 5,037 songs have an intro, live, medley or joined title (54 of them charting). Without them, 5.6% of songs charted instead of 5.5%; singles chart +11.4 points above the same artist's average (+11.3 in the main result) and pop +12.1 (unchanged), with almost identical ranges.

**Interpretation:** these entries lower the overall chart rate only slightly and do not affect the within-artist comparisons. They partly offset the title rule, which works in the other direction and can matter more (worst case 5.5% → 4.2%, above).

### **11.7 Exploratory: does the release month matter?**

**Status: exploratory, added after the main analysis.** The release month was not part of our analysis plan (About this study). We added it after the main analysis because of our own earlier result: 7.3 showed that fewer charting songs enter the chart in January, which raised the question whether the month of release, not only the month of entry, is associated with reaching the chart. Release timing is also one of the few factors studied here that the people releasing a song choose (as with release type, 11.3), which makes the answer of practical interest. A factor chosen after seeing the data is more likely to show a pattern by chance than a planned one, so we report every comparison we made, including a first approach that we abandoned (below), and present the result as a **secondary, exploratory part of our answer** (Section 13), to be confirmed on other data.

**Measure.** The same within-artist comparison as in 11.3–11.4, applied to the month of first release. Only songs with an exact release date are used, because dates known only to the month have no date after cleaning (8.7), and songs dated after their first chart week are left out (M2). This is the month in which a song was first released, not the month in which it entered the chart (7.3), and it includes singles that never charted.

**First approach.** We start with the simplest comparison: all songs released in a month, compared with the same artist's songs released in other months.

**Many comparisons.** We compare 12 months and 20 genre-by-quarter cells. With approximate 95% ranges, about one comparison in twenty is expected to exclude 0 by chance alone, so a single cell should not be over-read; we only interpret patterns that hold across neighbouring months and across the checks.

In [ ]:
MONTH_NAMES = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
QUARTERS    = ["Jan–Mar", "Apr–Jun", "Jul–Sep", "Oct–Dec"]

# Songs with an exact release date (dates known only to the month have no date after cleaning, 8.7),
# without songs dated after their first chart week (M2).
month_songs = songs_2000_2023[(songs_2000_2023["release_precision"] == "day") & songs_2000_2023["release_date"].notna()
                        & ~songs_2000_2023["flag_release_after_chart"].astype(bool)].copy()
month_number = pd.to_datetime(month_songs["release_date"]).dt.month
month_songs["release_month"]   = pd.Categorical(month_number.map(lambda m: MONTH_NAMES[m - 1]), categories=MONTH_NAMES, ordered=True)
month_songs["release_quarter"] = pd.Categorical(((month_number - 1) // 3).map(lambda i: QUARTERS[i]), categories=QUARTERS, ordered=True)
print(f"Songs with an exact release date: {len(month_songs):,} ({int(month_songs['charted'].sum()):,} charted)")

# First approach: all songs, each compared with the same artist's songs released in other months
first_try = within_artist(month_songs, "release_month").reindex(MONTH_NAMES)
first_table = pd.DataFrame({
    "songs": first_try["songs"],
    "difference": first_try["difference_pts"],
    "approx. 95% range": first_try["approx_95pct_range"],
    "share of singles (%)": (month_songs.groupby("release_month", observed=True)["release_type"]
                             .apply(lambda t: (t == "Single").mean() * 100).round(1)),
})
COMPARISON_RESULTS["release_month_all_songs_first_approach"] = first_table.reset_index().rename(columns={"index": "month"}).to_dict("records")
print("First approach – all songs: difference from the same artist's average, in percentage points:")
first_table

**Observations (first approach):** every month came out *above* the artists' averages, from +0.8 points in December to +6.1 in January (all ranges above 0 except December's). The share of singles among the songs released in a month falls from 16% in January to 10% in December.

**Why we changed the approach:** every month cannot be better than average, so this comparison does not measure what we want. The cause is release type: an artist's album tracks come out together, in one month, and rarely chart, while singles are spread over the year and chart much more often (11.3). Because the measure gives each of an artist's months equal weight, the many months that hold only a few singles outweigh the one month that holds an album. Since singles are also unevenly spread over the year, the comparison of all songs mostly repeats the single-versus-album result. We keep it here to show why the final approach compares singles only.

**Final approach.** We compare **singles only**: each single with the same artist's other singles (artists with at least 5 singles with an exact date), so that release type is held constant. We repeat it without holiday songs, which are mostly released in November–December and rarely chart (11.4), and we split it by genre family and quarter to see whether the best time differs by genre.

In [ ]:
# Final approach: singles only, each compared with the same artist's other singles
MIN_ARTIST_SINGLES = 5        # fewer artists have 10 singles (as in 11.6)
singles = month_songs[month_songs["release_type"] == "Single"]
singles_no_holiday = singles[singles["genre_family"] != "Holiday"]
print(f"Singles with an exact release date: {len(singles):,} ({int(singles['charted'].sum()):,} charted)")

by_month = within_artist(singles, "release_month", MIN_ARTIST_SINGLES).reindex(MONTH_NAMES)
by_month_nh = within_artist(singles_no_holiday, "release_month", MIN_ARTIST_SINGLES).reindex(MONTH_NAMES)
month_table = pd.DataFrame({
    "singles: songs": by_month["songs"],
    "singles: difference": by_month["difference_pts"],
    "singles: approx. 95% range": by_month["approx_95pct_range"],
    "singles without holiday songs: difference": by_month_nh["difference_pts"],
    "singles without holiday songs: approx. 95% range": by_month_nh["approx_95pct_range"],
})
by_quarter = within_artist(singles, "release_quarter", MIN_ARTIST_SINGLES).reindex(QUARTERS)
COMPARISON_RESULTS["release_month_singles"] = month_table.reset_index().rename(columns={"index": "month"}).to_dict("records")
COMPARISON_RESULTS["release_quarter_singles"] = by_quarter.reset_index().to_dict("records")
display(month_table)
display(by_quarter[["artists", "songs", "difference_pts", "approx_95pct_range"]])

# Checks: does the result depend on our thresholds or on holiday songs?
month_checks = {}
for label, df, min_singles in [(f"Main result (at least {MIN_ARTIST_SINGLES} singles per artist)", singles, MIN_ARTIST_SINGLES),
                               ("Without holiday songs", singles_no_holiday, MIN_ARTIST_SINGLES),
                               ("At least 3 singles per artist", singles, 3),
                               ("At least 10 singles per artist", singles, 10)]:
    months   = within_artist(df, "release_month", min_singles)     # own names: q is our SQL helper (6.2)
    quarters = within_artist(df, "release_quarter", min_singles)
    month_checks[label] = {**{m: f"{months.loc[m, 'difference_pts']:+.1f} ({months.loc[m, 'approx_95pct_range']})"
                              for m in ["Jan", "Nov", "Dec"]},
                           "Oct–Dec": f"{quarters.loc['Oct–Dec', 'difference_pts']:+.1f} ({quarters.loc['Oct–Dec', 'approx_95pct_range']})",
                           "artists": int(df.groupby("artist_mbid").size().ge(min_singles).sum())}
month_checks = pd.DataFrame(month_checks).T
COMPARISON_RESULTS["release_month_checks"] = month_checks.reset_index().rename(columns={"index": "variant"}).to_dict("records")
print("Checks: difference from the same artist's other singles (approximate 95% range):")
display(month_checks)

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(MONTH_NAMES))
ax.bar(x, by_month["difference_pts"], yerr=by_month["range_pts"], color=COLOR_SECOND, ecolor="black", capsize=3)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x, MONTH_NAMES)
style_axes(ax, "Singles released in November–December chart less often",
           "Month of release", "Percentage points above (+) or below (−)\nthe same artist's other singles")
save_figure(fig, "11_7_singles_by_release_month")
plt.show()

# Does the best time differ by genre? Singles with their own genre, by quarter (artists with at least 3 such singles).
own_singles = singles_no_holiday[singles_no_holiday["genre_level"].isin(["recording", "release group"])]
genre_quarter = {}
for family in ["Pop", "Hip hop / rap", "R&B / soul", "Rock / alternative", "Country"]:
    s = within_artist(own_singles[own_singles["genre_family"] == family], "release_quarter", 3)
    genre_quarter[family] = {q: (f"{s.loc[q, 'difference_pts']:+.1f} ({s.loc[q, 'approx_95pct_range']})"
                                 if q in s.index and s.loc[q, "artists"] >= 30 else "–") for q in QUARTERS}
    genre_quarter[family]["artists"] = int(s["artists"].max()) if len(s) else 0
genre_quarter = pd.DataFrame(genre_quarter).T
COMPARISON_RESULTS["genre_by_release_quarter_singles"] = genre_quarter.reset_index().rename(columns={"index": "genre"}).to_dict("records")
print("Singles: difference from the same artist's other singles of that genre, by release quarter (approximate 95% range):")
genre_quarter

**Observations:**
- Of 18,551 singles with an exact release date, 3,141 (16.9%) charted.
- Singles released in **November and December** charted 3.6 points less often than the same artist's other singles (ranges −5.3 to −1.9 and −5.7 to −1.5). By quarter, October–December is the only quarter below the artists' average (−2.6, range −3.8 to −1.4).
- Without holiday songs the results are almost the same (November −3.8, December −3.2), so the year-end dip is not caused by Christmas songs.
- Singles released in **January** charted 5.2 points more often (+2.8 to +7.6). The other months lie between −1.1 and +2.3, most with ranges that include 0.
- **By genre** (singles with their own genre, 56–281 artists per family), the ranges are wide. The October–December difference is negative for all five families (−0.4 to −5.6), but excludes 0 only for hip hop (−5.6) and rock/alternative (−2.6). Two other cells exclude 0: R&B/soul in January–March (+9.2, from fewer than 60 artists) and rock/alternative in July–September (+2.7). Four of 20 cells is more than the one expected by chance, but each rests on few artists.

**Interpretation:** a single released at the end of the year is associated with a lower chance of charting: about one sixth lower in October–December and one fifth lower in November–December, compared with the same artist's other singles (16.9% on average). January is the best month. One possible explanation, consistent with Section 7 but not tested here, is competition: holiday songs return to the chart every December and big album releases can fill many places in one week (7.6), while January has fewer new entries than its year's average in 21 of 26 years (7.3), leaving more room for a new single. We find no clear evidence that the best time differs by genre: the year-end dip appears in every genre, and the genre-specific peaks rest on too few artists to tell them apart from chance.

**Two cautions apply.** First, our outcome is whether a single ever reaches the weekly Hot 100, not its place in a year-end ranking, so a song released early in the year gains nothing from having more weeks left in that calendar year. Its only possible advantage is a longer follow-up, and that advantage is small: every single released 2000–2023 could be followed for almost two years or more, while 92% of charting songs enter within one year (9.2); the limitations below discuss it. Second, holiday and summer songs are usually planned releases. Removing holiday songs leaves the year-end dip unchanged, and summer shows no clear peak (June–August between +0.5 and +2.0, with ranges that include or touch 0), but the timing of strong singles may still reflect the label's expectations rather than an effect of the month itself.

**Practical implication:** if the pattern is confirmed, it would favour releasing a single in January rather than in November or December. Because the question was added after the main analysis and release dates are themselves strategic choices (strong singles may be released in different months from weaker ones), this needs confirming on other data, such as later years, before it is acted on.

**Limitations:** only singles with an exact release date (93% of charting and 86% of non-charting songs have one, 10.4); the first release date in MusicBrainz may differ from the date a single was promoted; and release habits changed over 2000–2023 (since 2015 new music is released on Fridays worldwide), which this comparison does not separate. We have not confirmed the pattern on songs outside this analysis. 2025 releases are not in the comparison group (Section 10), so their chart rate cannot be measured. 2024 releases are in both groups, but they could only be followed until December 2025: a single released in January 2024 had almost two years to chart, one released in December about one year. This alone would make late-year singles look less successful, in the same direction as our result, so a simple comparison of 2024 singles by month could not confirm it. A fair test would count, for every single, only chart entries within the same time after release (as in Section 9); we leave it to further work (13.4).

### **11.8 Save the comparison results and figures**

In [ ]:
COMPARISON_FILE = OUTPUT_DIR / "analysis_results_comparison.json"
COMPARISON_FILE.write_text(json.dumps(without_nan({
    "record_type": "analysis results – charting vs non-charting songs of the same artists (aggregates only)",
    "source_commit": SOURCE_COMMIT,
    "musicbrainz_export_sha256": mb_sha256,
    "comparison_export_commit": CMP_COMMIT,
    "computed_at_utc": datetime.now(timezone.utc).replace(microsecond=0).isoformat(),
    "last_release_year": LAST_RELEASE_YEAR,
    "min_songs_per_artist": MIN_ARTIST_SONGS,
    "results": COMPARISON_RESULTS,
}), indent=2, ensure_ascii=False, allow_nan=False, default=str), encoding="utf-8")

ALL_FIG_ZIP = shutil.make_archive(str(OUTPUT_DIR / "figures_all"), "zip", FIG_DIR)
download_file(COMPARISON_FILE)
download_file(ALL_FIG_ZIP)

### **11.9 Summary of Section 11**

**Observations:**
1. Of 178,338 songs released 2000–2023 by 2,029 artists who have charted, 5.5% reached the Hot 100; the median artist charted with 3.5% of their songs (11.1). This share changed little over time (11.2).
2. Songs first released as singles charted 11 percentage points more often than the same artist's average; album and EP tracks less often (11.3).
3. Among songs with their own genre, those tagged pop charted 12 points more often than the same artist's average, and still about 7.5 points more often among songs of the same release type; holiday songs charted less often (11.4, 11.6).
4. A song's chance of ever charting halves within its first two possible charts and is about eleven times lower after a year without charting (11.5).
5. The results for singles and pop hold when success requires at least two weeks on the chart, with 5 or 20 songs per artist, without Glee Cast, and (for pop) without songs whose genre is uncertain; the small differences for R&B/soul and rock/alternative do not. The early drop holds, even more sharply, among songs first released as singles, and the comparison group's title rule can lower chart levels by at most about a quarter without changing the shape of the curve (11.6).
6. *Exploratory:* singles released in November–December charted about 3.6 points less often than the same artist's other singles, and January singles 5.2 points more often (11.7).

**Interpretation:** for songs by artists who can reach the chart, what most distinguishes the songs that do is how they are released (as singles), the genre they are perceived as (pop), and time to chart: a song's chance of ever charting halves if it misses its first two possible charts, and keeps falling after that. These are associations, not causes: singles are usually chosen and promoted because they are expected to succeed, which our data do not record; genre tags may partly follow success; and every artist compared has already charted, so the effect of being a known artist cannot be tested.

## **12. Generated documentation: README and file inventory**

This last section documents the run itself. From the notebook's own settings and results it produces two files:
- **`file_inventory.csv`**: every file the notebook wrote to `outputs/`, with its size, its SHA-256 fingerprint, whether it was downloaded, and whether it contains song-level chart data.
- **`README.md`**: a short description of the project for someone who receives our files without the notebook: the question, the sources with their fixed versions and fingerprints, how to reproduce the work, the main results and limitations, the files produced and the software versions.

**Why generated rather than written by hand** (decision log, row 23). A hand-written README has to be updated every time a result changes, and its numbers can silently go out of date. Here every number, version and fingerprint is read from the run that produced the results. The README repeats the main interpretations of Sections 9–11 only briefly; the full reasoning stays in this notebook.

### **12.1 File inventory**

Every file in `outputs/` is listed. Whether a file was downloaded is recorded by `download_file` itself (2.3), so the table shows what actually happened in this run, not what we intended. A file counts as containing **song-level chart data** if it holds Billboard chart rows or song lists derived from them; these are the files that Section 1.8 keeps private. The README and the inventory itself are not listed, because they are written in this section (and a file cannot contain its own fingerprint).

In [ ]:
# Files that contain song-level chart data, or full extracts that include it: never to be published (Section 1.8).
CHART_DATA_PREFIXES = ("raw/hot-100", "clean/", "storage_comparison/")   # raw Billboard file, cleaned tables, storage tests
CHART_DATA_FILES    = {"billboard_hot100.sqlite", "billboard_song_names.csv"}
GENERATED_HERE      = {"README.md", "file_inventory.csv"}     # written in this section, so not listed

def sha256_of(path):
    """SHA-256 fingerprint of a file, read in 1 MB blocks so that large files do not fill the memory."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

downloaded = {Path(p).resolve() for p in DOWNLOADED}
rows = []
for path in sorted(p for p in OUTPUT_DIR.rglob("*") if p.is_file()):
    rel = path.relative_to(OUTPUT_DIR).as_posix()
    if rel in GENERATED_HERE:
        continue
    if path.resolve() in downloaded:
        sent = "yes"
    elif rel.startswith("figures/"):
        sent = "yes, inside figures_all.zip"
    else:
        sent = "no, kept in Colab only"
    rows.append({"file": rel,
                 "size (KB)": round(path.stat().st_size / 1000, 1),
                 "downloaded": sent,
                 "song-level chart data": "yes" if rel.startswith(CHART_DATA_PREFIXES) or rel in CHART_DATA_FILES else "no",
                 "sha256": sha256_of(path)})
inventory = pd.DataFrame(rows)

private_sent = inventory.loc[(inventory["song-level chart data"] == "yes") & (inventory["downloaded"] == "yes"), "file"]
print(f"Files in {OUTPUT_DIR}/: {len(inventory)}, of which downloaded: {(~inventory['downloaded'].str.startswith('no')).sum()}, "
      f"with song-level chart data: {(inventory['song-level chart data'] == 'yes').sum()}")
print("Files with chart data that were downloaded:", ", ".join(private_sent) or "none")

INVENTORY_FILE = OUTPUT_DIR / "file_inventory.csv"
inventory.to_csv(INVENTORY_FILE, index=False)
download_file(INVENTORY_FILE)
inventory.drop(columns="sha256")

**Observations:** the only downloaded file that contains chart data is `billboard_song_names.csv`, the list of song names needed for the PostgreSQL step (8.1), which stays within the group. The database, the raw Billboard file, the cleaned tables and the storage-test files remain in Colab only.

**Interpretation:** the separation set out in Section 1.8 holds in practice: what leaves Colab is documentation, aggregates and figures. The fingerprints let anyone check that a file they hold is the one produced by this run. The JSON records contain the time they were made, and the zip files the time each figure was saved, so their fingerprints change with every run; the other files are rebuilt from the same pinned inputs and should be identical from one run to the next.

### **12.2 README**

The README is built in two steps: the first cell collects the numbers and tables from this run, the second writes the text, saves it as `README.md`, downloads it, and displays it below as it will appear. It contains aggregates only.

The Billboard fingerprint is taken from `EXPECTED_SHA256`: Section 3.2 stops the notebook unless the downloaded file matches it exactly, so it is the fingerprint of the file we used.

In [ ]:
from IPython.display import Markdown

EXTRACT_REPO_URL = f"https://github.com/{MB_REPO}"
GENERATED_AT = datetime.now(timezone.utc).replace(microsecond=0)

def pct(x, digits=1):
    """A number as a percentage text, e.g. 5.5 -> '5.5%'."""
    return f"{x:.{digits}f}%"

def md_table(df):
    """A DataFrame as a Markdown table (no extra library needed)."""
    header = "| " + " | ".join(map(str, df.columns)) + " |"
    rule = "|" + "---|" * len(df.columns)
    body = ["| " + " | ".join(map(str, row)) + " |" for row in df.itertuples(index=False)]
    return "\n".join([header, rule, *body])

# Numbers quoted in the README, taken from this run's results (nothing typed by hand).
n_songs, n_charts = int(q("SELECT COUNT(*) AS n FROM songs")["n"][0]), int(q("SELECT COUNT(*) AS n FROM charts")["n"][0])
entered_by = ANALYSIS_RESULTS["cumulative_share_by_chart_5y"]
compared = cmp_ledger.iloc[-1]
top_type, top_genre = by_type_within.iloc[0], by_genre_within.iloc[0]
still = key_rows.set_index("chart")["% that still charted later"]

sources = pd.DataFrame([
    ["Billboard Hot 100 weekly charts", f"[{SOURCE_REPO}]({REPO_URL}), commit `{SOURCE_COMMIT[:7]}`, file `{SOURCE_PATH}`",
     f"`{EXPECTED_SHA256}`", "rights claimed by PMC, publisher of Billboard (notebook Section 1)"],
    ["MusicBrainz full export", f"`{MB_DUMP}`, MetaBrainz Foundation", "checked against the published checksums (8.2)",
     "core data CC0; genres (tags) CC BY-NC-SA 3.0"],
    ["MusicBrainz data of charting songs (Job A)", f"[{MB_REPO}]({EXTRACT_REPO_URL}), commit `{MB_COMMIT[:7]}`, file `{MB_PATH}`",
     f"`{MB_EXPECTED_SHA256}`", "CC BY-NC-SA 3.0 (contains genres)"],
    *[[f"Comparison group (Job B): `{name}`", f"[{MB_REPO}]({EXTRACT_REPO_URL}), commit `{CMP_COMMIT[:7]}`",
       f"`{sha}`", "CC BY-NC-SA 3.0 (contains genres)"] for name, sha in CMP_FILES.items()],
], columns=["Data", "Where from (fixed version)", "SHA-256", "Rights / licence"])

files = inventory[~inventory["file"].str.startswith(("figures/", "storage_comparison/"))].drop(columns="sha256")
versions = pd.DataFrame([["Python", sys.version.split()[0]], ["pandas", pd.__version__], ["numpy", np.__version__],
                         ["matplotlib", matplotlib.__version__], ["requests", requests.__version__],
                         ["SQLite", sqlite3.sqlite_version], ["Platform", platform.platform()]],
                        columns=["Software", "Version"])

In [ ]:
README = f"""# What determines whether a song reaches the Billboard Hot 100? (Group C)

*Generated automatically by `{NOTEBOOK_NAME}` on {GENERATED_AT:%Y-%m-%d %H:%M} UTC. Do not edit by hand: change the notebook and run it again.*

## Research question
What determines whether a song reaches the Billboard Hot 100? A song counts as successful if it appears on the chart at least once.
We study the weekly charts dated {STUDY_START} to {STUDY_END} ({n_charts:,} charts, {n_songs:,} songs) and compare the songs that charted
with songs by the same lead artists, released in the same years, that never charted.

## Data sources
Every source is downloaded from a fixed version and checked with its SHA-256 fingerprint before use; the notebook stops if a file differs.

{md_table(sources)}

## Where the work is kept
- **The notebook** (`{NOTEBOOK_NAME}`) contains the whole workflow, from collection to analysis, with every decision and its justification (decision log at the top).
- **[{MB_REPO}]({EXTRACT_REPO_URL})** contains the SQL scripts that extract MusicBrainz data in a local PostgreSQL database, the extracts and their run logs.
  It is separate because this step cannot run in Colab (the MusicBrainz tables are about 25 GB), its results must be downloadable at a fixed version,
  and the extracts carry MusicBrainz's licence (notebook 8.3, decision log row 20).

## How to reproduce
**Analysis only (no setup needed):** open the notebook in Google Colab and click **Runtime → Run all** (to start from scratch,
first **Runtime → Disconnect and delete runtime**). Nothing needs to be uploaded and no login is needed; allow multiple downloads if the browser asks.
The notebook downloads every source at a fixed version, including the published results of the MusicBrainz step, so steps 2–3 below are skipped.

**The whole workflow runs in four stages, in this order:**
1. **Notebook, Sections 1–8.1 (Google Colab).** Downloads the pinned Billboard archive, checks its SHA-256, cleans it and stores it in SQLite.
   Section 8.1 exports `billboard_song_names.csv`, the list of songs to match, and downloads it to your computer.
2. **Job A (local computer, PostgreSQL; about 25 GB of disk space and a few hours).** In one project folder, put `run_all.sh` and `run_comparison.sh`
   from the extract repository and `billboard_song_names.csv` at the top, the SQL scripts in `sql/`, and the MusicBrainz export `{MB_DUMP}`
   (`mbdump.tar.bz2`, `mbdump-derived.tar.bz2`, `SHA256SUMS`) in `downloads/`; the needed tables are extracted into `mbdump/`.
   Run `run_all.sh`. Output: `musicbrainz_for_billboard_songs.csv`.
3. **Job B (same database).** Run `run_comparison.sh`. Outputs: `comparison_songs_musicbrainz.csv.gz` and `billboard_song_lead_artist.csv`.
4. **Notebook, Sections 8.3–13.** Downloads the results of Jobs A and B from the extract repository at fixed commits, checks their SHA-256 and runs
   the rest of the analysis. A rerun of Jobs A and B on the same export gives files with the same SHA-256 (8.3, 10.1).

MusicBrainz removes each export from its server after about two weeks, so `{MB_DUMP}` can no longer be downloaded there; we keep a private copy (notebook 8.2).
On a later export, Jobs A and B reproduce the method, but their results and fingerprints may differ slightly.

**Intermediate files not in our submission.** They contain Billboard chart data, which we do not redistribute (notebook 1.8), or are too large;
each one is rebuilt by the step that produces it.

| File | Rebuilt by | Used by |
|---|---|---|
| `billboard_song_names.csv` (song ID, name ID, title, performer) | Stage 1 (notebook 8.1) | Job A |
| `billboard_hot100.sqlite` and the clean tables | Stage 1 (notebook Sections 5–6), extended in stage 4 | The rest of the notebook |
| `mbdump/` (MusicBrainz tables, about 25 GB) | Extracting the export `{MB_DUMP}` (stage 2) | Jobs A and B |

## Main results
1. **Time to chart.** Of the charting songs followed for five years, {pct(entered_by[1] * 100, 0)} entered on their first possible chart
   and {pct(entered_by[4] * 100, 0)} within four charts (Section 9.2).
2. **Most songs never chart.** Of {int(compared['charting songs']) + int(compared['non-charting songs']):,} songs released
   2000–{LAST_RELEASE_YEAR} by {int(compared['artists']):,} artists who have charted, {pct(overall['% of songs that charted (all songs pooled)'])} reached
   the Hot 100; the median artist charted with {pct(overall['% charted, median artist'])} of their songs (Section 11.1).
3. **Release type.** Songs first released as a {top_type.name.lower()} charted {top_type['difference_pts']:.1f} percentage points more often
   than the same artist's average (approximate 95% range: {top_type['approx_95pct_range']}; Section 11.3).
4. **Genre.** Among songs with a genre of their own, {top_genre.name} songs charted {top_genre['difference_pts']:.1f} points more often
   than the same artist's average (approximate 95% range: {top_genre['approx_95pct_range']}; Section 11.4).
5. **A sharp early drop.** At release, {pct(still[1], 1)} of songs went on to chart; after missing their first two possible charts,
   {pct(still[3], 1)}; after a year without charting, {pct(still[52], 2)} (Section 11.5).
6. **Release month (exploratory, added after the main analysis).** Singles released in November and December charted
   {-by_month.loc['Nov', 'difference_pts']:.1f} and {-by_month.loc['Dec', 'difference_pts']:.1f} points less often than the same artist's other singles,
   and January singles {by_month.loc['Jan', 'difference_pts']:.1f} points more often (Section 11.7). This is a secondary result that still needs confirming on other data.

These are associations, not causes: singles are usually chosen and promoted because they are expected to succeed, which our data do not record;
genre tags may partly follow success; and every artist compared has already charted.

## Main limitations
- MusicBrainz documents some songs and periods more completely than others, so overall levels are less reliable than comparisons within artists.
- The comparison group contains only artists who have charted, so the effect of being a known artist cannot be tested.
- Release dates are more often exact for charting songs than for non-charting songs (10.4).
- The comparison group excludes every song whose title equals a Billboard title by any artist; this can only make chart rates too high,
  by at most about a quarter in the worst case, without changing the shape of the time-to-chart curve (11.6).
- Genre is the least certain variable: for about one charting song in eight, the top genre is tied with a genre of another family (8.7);
  the pop result holds without these songs (11.6).
- About 3% of comparison songs (0.6% of charting songs) are intros, live tracks or medleys rather than separate songs (10.5); removing them
  changes neither the chart rate (5.5% → 5.6%) nor the main results (11.6).

## Files produced by the notebook
All files are written to the `outputs/` folder in Colab, which is deleted when the runtime ends. Files with song-level chart data are kept in Colab only
(apart from `billboard_song_names.csv`, needed for the PostgreSQL step and kept private). Fingerprints of every file: `file_inventory.csv`.
Figures are in `figures_all.zip`; individual figures and the storage-test files of Section 6.7 are listed only in `file_inventory.csv`.

{md_table(files)}

## Software used in this run
{md_table(versions)}
"""

README_FILE = OUTPUT_DIR / "README.md"
README_FILE.write_text(README, encoding="utf-8")
download_file(README_FILE)
display(Markdown(README))

### **12.3 Final checks**

Before finishing, we check that:
- every JSON record can be read by any program. Python writes missing numbers as `NaN`, which is not valid JSON; 9.8 and 11.8 already prevent this, and here we check all records;
- the database is intact and all its relationships (foreign keys) hold;
- the README contains no missing values;
- no file with chart data was downloaded, apart from `billboard_song_names.csv`;
- the stored time from release to first chart (`days_release_to_chart`, 8.7) still matches the dates it was computed from, so the database contains no outdated derived value.

The notebook stops if a check fails. Finally the database connection is closed, so the database file is complete. After this, run any cell again only with **Runtime → Run all**.

In [ ]:
import re   # standard library: text patterns

def no_nan(token):
    """Called by json.loads for NaN or Infinity, which are not valid JSON."""
    raise ValueError(f"{token} is not valid JSON")

checks = {}
for path in sorted(OUTPUT_DIR.glob("*.json")):
    try:
        json.loads(path.read_text(encoding="utf-8"), parse_constant=no_nan)
        checks[f"{path.name} is valid JSON"] = "passed"
    except ValueError as error:
        checks[f"{path.name} is valid JSON"] = f"FAILED: {error}"

checks["database integrity (PRAGMA integrity_check)"] = "passed" if q("PRAGMA integrity_check").iloc[0, 0] == "ok" else "FAILED"
checks["no foreign-key violations"] = "passed" if q("PRAGMA foreign_key_check").empty else "FAILED"
checks["README has no missing values"] = "passed" if not re.search(r"\b(nan|None|NaN)\b|<NA>", README) else "FAILED"
checks["no chart-data file downloaded except billboard_song_names.csv"] = (
    "passed" if set(private_sent) <= {"billboard_song_names.csv"} else "check: " + ", ".join(private_sent))

checks["stored days_release_to_chart matches the chart and release dates"] = "passed" if q("""
    SELECT COUNT(*) AS n FROM song_mb_clean c JOIN song_summary s USING (song_id)
    WHERE c.release_date IS NOT NULL
      AND c.days_release_to_chart <> CAST(julianday(s.first_chart_date) - julianday(c.release_date) AS INTEGER)
""")["n"][0] == 0 else "FAILED"

final_checks = pd.Series(checks, name="result").to_frame()
display(final_checks)
if final_checks["result"].str.startswith("FAILED").any():
    raise ValueError("A final check failed – see the table above.")

con.commit()
con.close()
print("Database connection closed. To run any cell again, use Runtime → Run all.")

### **12.4 Summary of Section 12**

Section 12 documents the notebook's own run: an inventory of every file it produced, with fingerprints and whether each one left Colab, and a README generated from the same results, so the documentation always matches the analysis. The final checks confirm that every record is valid, that the database is intact, and that chart data stayed in Colab as set out in Section 1.8.

## **13. Conclusion**

**Research question:** what determines whether a song reaches the Billboard Hot 100?


### **13.1 Answer**

We can answer this question for songs by artists who reach the Hot 100 at least once: 178,338 songs released 2000–2023 by 2,029 such artists, of which only 5.5% charted (the median artist: 3.5% of their songs). Comparing each artist's songs with each other, three factors are clearly associated with reaching the chart:

1. **Release type.** Songs first released as singles charted 11.3 percentage points more often than the same artist's average (11.3).
2. **Genre.** Songs tagged pop charted 12.1 points more often than the same artist's average, and still about 7.5 points more often among songs of the same release type, so this is not only a single-versus-album effect; holiday songs charted 5.0 points less often (11.4, 11.6). Other genres show no robust difference.
3. **Time to chart.** Charting is decided early. Of charting songs, 38% entered on their first possible chart and 54% within four charts (9.2). Including songs that never charted, a song's chance of ever charting falls from 5.2% at release to 2.7% once it has missed its first two possible charts, and to 0.45% after a year (11.5).

The release-type and genre results hold when success requires at least two weeks on the chart, with 5 or 20 songs per artist instead of 10, and without the Glee Cast outlier; the pop result also holds without songs whose genre is uncertain. The early drop is not caused by album release dates: among songs first released as singles it is even sharper. The overall levels are probably slightly too high, because the comparison group excludes songs that share a title with any Billboard hit; in the worst case 4.2% instead of 5.5% of songs charted, with the same shape of the time-to-chart curve (11.6).

4. **Release month (exploratory).** Among singles, those released in November–December charted about 3.6 points less often than the same artist's other singles (about one fifth lower than the 16.9% average), and January singles 5.2 points more often; the pattern holds without holiday songs and with 3 or 10 singles per artist instead of 5 (11.7). Because release month was added after the main analysis, this is a secondary result: a consistent pattern, but one to be confirmed on other data, such as later years.

**Artist popularity** could only be studied among charting songs: since 2010, artists who had charted before entered faster than newcomers (9.7), and in the streaming era they also entered higher (7.5). Whether being known makes a song more likely to chart at all cannot be tested with our data, because every artist in the comparison group has charted.

### **13.2 Our hypothesis**

*Most songs that reach the Hot 100 do so soon after release; beyond a certain time, a song becomes much less likely to ever enter the chart.*

**Supported.** The threshold comes early: the chance of entering drops sharply over the first two possible charts (9.4). It is not a cut-off, however: after those two charts songs keep entering at a low, roughly constant weekly rate, so a song's chance of ever charting keeps falling slowly without reaching zero (9.4, 11.5). This early drop became stronger in the streaming era (9.3, 7.4). It is clearest for songs first released as singles, whose release date is the single's own: their chance of ever charting falls by 56% over the first two charts, against 43% for album tracks (11.6).

### **13.3 What these results do not show**


- **They are associations, not causes.** Artists and labels usually choose which songs become singles and how much to promote them, and we observe neither the choice nor the promotion. Genre tags are added by volunteers and may partly follow success; about one tag in seven was questionable in our manual check (26 / 30 sensible, 8.4d), and for about one song in eight the top genre is tied with a genre of another family (8.7).
- **The comparison covers only artists who have charted.** Songs by artists who never reached the Hot 100 are not included, so the results describe what separates an established artist's hits from their other songs.
- **The comparison group's title rule.** Songs that share a title with any Billboard hit are excluded (10.1), so chart rates are probably slightly too high, by at most about a quarter; its effect on the within-artist comparisons could not be tested without re-running the extraction (11.6).
- **Not every comparison entry is a song.** In our manual check of 30 comparison songs (10.5), three were a spoken intro, a live introduction and a live medley, and one carried a reissue date. About 3% of comparison songs, against 0.6% of charting songs, have an intro, live, medley or joined title. Such entries lower the share of songs that charted slightly, partly offsetting the title rule above, but removing them changes neither the chart rate (5.5% → 5.6%) nor the main results (11.6).
- **Data limitations.** MusicBrainz documents some songs and periods more completely than others, and exact release dates are more common for charting songs (10.4). An album track's release date is the album's, which may be earlier than the song's promotion (11.6). The Billboard data come from a public archive whose legal status is uncertain (Section 1).
- **Release month** (11.7) was chosen after the main analysis, and release dates are themselves strategic choices: strong singles may be scheduled differently from weaker ones.

### **13.4 Further work**

- Data on promotion, radio play and streaming, to separate the effect of how songs are released from how much they are promoted.
- Songs by artists who never charted, to test the effect of being a known artist.
- A model of charting on release type, genre and release month together, with each artist as its own baseline, to estimate each factor while holding the others constant. We compared factors one or two at a time (11.3–11.7), which keeps each result transparent; a model would need to be discussed with our professor first, as the project brief requires (decision log row 22).
- Re-running the comparison group with the title rule limited to the same artist, to measure its effect on the within-artist comparisons (10.1, 11.6).
- Single and radio release dates, to date album tracks by the moment they were promoted rather than by the album (11.6).
- Audio features (e.g. tempo, energy), which we had planned to take from the Kaggle Spotify dataset (1.10); this needs a source that also covers songs that never charted.
- Other characteristics planned only now, such as collaborations, song length or release weekday, tested on new data rather than chosen from these results.
- Confirming the release-month pattern on later years (2024 onwards) or on other charts, with a fair test that counts, for every single, only chart entries within the same time after release, so that late-year releases are not disadvantaged by a shorter follow-up (11.7).